# Buscador y descarga de escenas Sentinel-2 para YOLO · Nevado Santa Isabel

Notebook autónomo (kernel del entorno `gee`, Python 3.11). Permite definir área y fecha, consultar adquisiciones Sentinel-2 Level-2A, revisar la visibilidad de la cumbre, seleccionar escenas por número de fila y descargar productos locales alineados: GeoTIFF de reflectancia, índices y calidad, más dos PNG de tres canales (**RGB** y **espectral**) con idéntica malla, de modo que un mismo polígono sirve en ambas variantes. La sección 9 anota esos parches de nieve/hielo visible como etiqueta YOLO Segmentation, sin conexión.

**Uso habitual**
1. En la celda 1 cambia `fecha_objetivo` y `ventana_dias` (y `proyecto_ee` la primera vez).
2. Ejecuta todo (*Run All*). La sección 4 muestra la tabla de candidatos con filas numeradas.
3. En la sección 5 elige una fila en el desplegable para revisar la cumbre sobre el mapa.
4. En la sección 6 marca las filas aceptadas y pulsa «Descargar marcadas»; luego ejecuta la sección 7 para verificarlas.
5. Ejecuta de nuevo la celda 9d y anota la escena: dibuja los parches, guarda el borrador y aprueba la etiqueta YOLO. Para una descarga antigua basta con las celdas 9a–9d (sin Earth Engine).

Con `ids_seleccionados = []` el notebook completo se ejecuta sin descargar nada.

**Advertencias**
- El objeto observable es la cobertura **visible** de nieve/hielo en la adquisición. Identificar hielo glaciar permanente requiere evidencia adicional.
- SCL se usa solo como criterio reproducible de calidad. Puede confundir nieve, nube y sombra, y **nunca** se usa como etiqueta de entrenamiento.
- La ausencia de observación o la presencia de nubes no justifican una etiqueta negativa.
- Datos: Copernicus Sentinel-2 (ESA), procesados mediante Google Earth Engine.

## 1. Instrucciones y parámetros

Esta es la única celda que se modifica en el uso habitual.

- **Fecha**: `fecha_objetivo` es un día UTC. `ventana_dias = 0` consulta solo ese día; `ventana_dias = 15` consulta desde 15 días antes hasta 15 días después (ambos extremos incluidos).
- **AOI**: el recuadro inicial es un encuadre de trabajo ajustado al macizo del Santa Isabel, **no un lindero oficial**. Compáralo en el primer uso con el mapa multitemporal de IDEAM y conserva el mismo AOI para toda la serie (cambiarlo cambia el `id_aoi` y la malla).
- **Otra área**: sustituye `aoi_entrada` por otra caja o por un GeoJSON Polygon/MultiPolygon en longitud/latitud WGS84, y cambia `nombre_aoi`. Ejemplo mínimo:

```python
aoi_entrada = {"type": "Polygon", "coordinates": [[[-75.40, 4.77], [-75.33, 4.78], [-75.35, 4.84], [-75.40, 4.77]]]}
nombre_aoi = "mi_area"
fecha_objetivo, ventana_dias = "2023-02-10", 5
```

- **Proyecto de Earth Engine**: escribe su ID en `proyecto_ee`. Si lo dejas vacío se usa la variable de entorno `EE_PROJECT` y, en su defecto, el proyecto guardado en tus credenciales locales (`earthengine set_project`).
- **Elegir escenas**: la tabla de la sección 4 numera cada fila (columna izquierda). Para inspeccionar usa el desplegable de la sección 5; para descargar marca casillas en la sección 6 y pulsa «Descargar marcadas». Si prefieres dejarlo fijo y reproducible, escribe los números aquí: `id_vista_previa = 2`, `ids_seleccionados = [1, 3]` (también se aceptan IDs). Los números se refieren a la tabla vigente: si cambias la fecha, vuelve a buscar antes de descargar.
- Una fecha puede devolver cero adquisiciones; en ese caso cambia la fecha o amplía la ventana. Nunca se sustituye automáticamente por fechas vecinas.

In [158]:
# 1. Parámetros (en el uso habitual solo se modifica esta celda).
import os
from pathlib import Path

# --- Earth Engine ---
proyecto_ee = ""              # ID del proyecto Cloud; "" = EE_PROJECT o el proyecto de las credenciales locales
autenticar_si_falta = True    # abre la autenticación interactiva si faltan credenciales válidas

# --- Área de interés (WGS84: longitud, latitud) ---
# Caja [oeste, sur, este, norte]: encuadre de trabajo ajustado al macizo del Santa Isabel (no es un lindero oficial).
aoi_entrada = [-75.397, 4.773, -75.348, 4.827]
nombre_aoi = "santa_isabel"
crs_salida = None             # None = UTM según el centro del AOI (aquí EPSG:32618); o un CRS métrico explícito
# Marcadores opcionales en el mapa {"nombre": (lon, lat)}; vacío = sin marcadores.
puntos_referencia = {}

# --- Fecha ---
fecha_objetivo = "2024-01-15" # día UTC, AAAA-MM-DD
ventana_dias = 15             # 0 = solo ese día; N = desde N días antes hasta N días después

# --- Calidad (porcentajes sobre el área total del AOI) ---
min_cobertura_pct = 95
min_util_pct = 70

# --- Tabla de candidatos ---
filas_por_pagina = 25
pagina = 1

# --- Inspección (sección 5): número de fila de la tabla (p. ej. 2) o ID; "" = primera fila. No descarga nada.
#     En la sección 5 también puedes cambiarla con el desplegable sobre el mapa. ---
id_vista_previa = ""

# --- Descarga (sección 6): números de fila de la tabla (p. ej. [1, 3]) o IDs. Vacía = cero descargas
#     automáticas; la sección 6 muestra entonces casillas para marcar escenas y un botón «Descargar». ---
ids_seleccionados = []
directorio_salida = Path("datos_sentinel2")

# --- Avanzado (rara vez se cambia) ---
tamano_bloque = 512           # píxeles por lado de cada petición de descarga
timeout_s = 180               # tiempo máximo por petición HTTP (s)
max_intentos = 4              # intentos por bloque ante fallos transitorios

## 2. Entorno y conexión

Muestra el intérprete y las versiones, verifica los paquetes (si falta alguno indica el comando para instalarlo en el kernel activo) y conecta con Earth Engine. Si la conexión falla se distingue entre autorización, permisos, proyecto y conexión; las celdas que consultan Earth Engine se omiten y las pruebas locales (sección 8) siguen funcionando.

In [159]:
# 2a. Entorno: intérprete, datos geoespaciales y paquetes.
import sys, importlib, platform, warnings

# Usa los datos PROJ/GDAL del propio entorno (evita conflictos con otras instalaciones, p. ej. PostGIS).
for _var, _subdirs in {"PROJ_DATA": ("Library/share/proj", "share/proj"),
                       "GDAL_DATA": ("Library/share/gdal", "share/gdal")}.items():
    for _sub in _subdirs:
        _ruta = Path(sys.prefix) / _sub
        if _ruta.is_dir():
            os.environ[_var] = str(_ruta)
            if _var == "PROJ_DATA":
                os.environ["PROJ_LIB"] = str(_ruta)
            break
# En Windows, si el kernel arranca sin activar el entorno, MKL (numpy/matplotlib) no encuentra sus DLL.
if os.name == "nt":
    _dll = [str(Path(sys.prefix) / d) for d in ("Library/bin", ".") if (Path(sys.prefix) / d).is_dir()]
    os.environ["PATH"] = os.pathsep.join(_dll + [p for p in os.environ.get("PATH", "").split(os.pathsep) if p not in _dll])
warnings.filterwarnings("ignore", category=FutureWarning, module=r"google\..*")

PAQUETES = {  # módulo -> paquete instalable
    "ee": "earthengine-api", "geemap": "geemap", "ipyleaflet": "ipyleaflet", "ipywidgets": "ipywidgets",
    "pandas": "pandas",
    "numpy": "numpy", "rasterio": "rasterio", "PIL": "pillow", "requests": "requests",
    "shapely": "shapely", "pyproj": "pyproj", "matplotlib": "matplotlib",
}
versiones_paquetes, faltantes = {}, []
for _mod, _paq in PAQUETES.items():
    try:
        versiones_paquetes[_paq] = str(getattr(importlib.import_module(_mod), "__version__", "?"))
    except ImportError:
        faltantes.append(_paq)

print(f"Python  : {platform.python_version()}  ->  {sys.executable}")
for _paq, _ver in versiones_paquetes.items():
    print(f"  {_paq:<16} {_ver}")
if faltantes:
    print("\nFaltan paquetes. Instálalos en el kernel activo con una de estas líneas y reinicia el kernel:")
    print(f"  %conda install -y -c conda-forge {' '.join(faltantes)}")
    print(f"  %pip install {' '.join(faltantes)}")
    raise ImportError(f"Paquetes faltantes: {faltantes}")

import hashlib, json, math, re, shutil, tempfile, time, functools
from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone

import numpy as np
import pandas as pd
import requests
import rasterio
from rasterio.crs import CRS as CRSRasterio
from rasterio.features import geometry_mask
from rasterio.io import MemoryFile
from rasterio.transform import Affine
from rasterio.windows import Window
from PIL import Image
import shapely
from shapely.geometry import Point, box, shape
from shapely.ops import transform as transformar_geometria
from shapely.validation import explain_validity
from pyproj import CRS, Transformer
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Polygon as PoligonoMpl
import ee
import geemap
import ipyleaflet
import ipywidgets as widgets
from IPython.display import display

CRS.from_epsg(32618)  # falla aquí si PROJ no encuentra su base de datos

# Contrato fijo de datos (SDD §2–§5).
COLECCION = "COPERNICUS/S2_SR_HARMONIZED"
RESOLUCION_M = 10
RESOLUCION_DIAG_M = 20
BANDAS_REFL = ["B2", "B3", "B4", "B8", "B11"]
RES_NATIVA_M = {"B2": 10, "B3": 10, "B4": 10, "B8": 10, "B11": 20, "SCL": 20}
ESCALA_REFL = 0.0001
NODATA = -9999.0
SCL_ADMITIDAS = (4, 5, 6, 11)
SCL_EXCLUIDAS = (0, 1, 2, 3, 7, 8, 9, 10)
NOMBRES_SCL = {0: "Sin datos", 1: "Saturado o defectuoso", 2: "Área oscura", 3: "Sombra de nube",
               4: "Vegetación", 5: "Suelo desnudo", 6: "Agua", 7: "Nube prob. baja / sin clasificar",
               8: "Nube prob. media", 9: "Nube prob. alta", 10: "Cirros", 11: "Nieve / hielo",
               255: "Sin observación (fuera del gránulo)"}
RGB_MAX_REFL = 0.60
LIMITE_BYTES_PETICION = 32 * 1024 * 1024
LIMITE_DIM_PETICION = 10000
RECETA_VERSION = "1.0"
print("\nEntorno verificado.")

Python  : 3.11.15  ->  c:\Users\ing_c\miniconda3\envs\gee\python.exe
  earthengine-api  1.7.35
  geemap           0.37.1
  ipyleaflet       0.20.0
  ipywidgets       8.1.8
  pandas           3.0.3
  numpy            2.4.6
  rasterio         1.4.4
  pillow           12.3.0
  requests         2.34.2
  shapely          2.1.2
  pyproj           3.7.2
  matplotlib       3.11.0

Entorno verificado.


In [160]:
# 2b. Conexión con Earth Engine.
def diagnosticar_error_ee(exc):
    """Clasifica un fallo como autorización, permisos, proyecto o conexión y propone una acción."""
    texto = f"{type(exc).__name__}: {exc}".lower()
    reglas = [
        ("conexión", ("connectionerror", "connection aborted", "connection reset", "connection refused",
                      "timed out", "timeout", "name resolution", "getaddrinfo", "unreachable",
                      "max retries", "sslerror", "proxyerror", "unable to find the server",
                      "servernotfound", "temporarily unavailable"),
         "No hay conexión con Earth Engine. Revisa la red, el proxy/VPN o el firewall y repite la celda."),
        ("autorización", ("authorize", "authenticate", "credentials", "invalid_grant", "refresherror",
                          "unauthenticated", "token has been expired", "reauth"),
         "Faltan credenciales válidas. Ejecuta ee.Authenticate() (o deja autenticar_si_falta=True) y repite."),
        ("permisos", ("permission", "403", "not registered", "has not been used", "is disabled",
                      "serviceusage", "caller does not have", "forbidden"),
         "La cuenta no tiene permisos sobre el proyecto, o la API de Earth Engine no está habilitada/registrada en él."),
        ("proyecto", ("project", "not found", "404", "invalid_argument"),
         "El proyecto no existe o está mal escrito. Revisa proyecto_ee o EE_PROJECT."),
    ]
    for tipo, claves, consejo in reglas:
        if any(k in texto for k in claves):
            return tipo, consejo
    return "desconocido", "Error no clasificado; revisa el detalle."


def proyecto_de_credenciales():
    """Proyecto guardado por `earthengine authenticate` / `earthengine set_project`, si existe."""
    try:
        return str(json.loads(Path(ee.oauth.get_credentials_path()).read_text()).get("project") or "").strip()
    except Exception:
        return ""


def resolver_proyecto(proyecto):
    """Prioridad: celda 1 > variable EE_PROJECT > proyecto de las credenciales locales."""
    for valor, origen in ((proyecto, "celda 1"), (os.environ.get("EE_PROJECT"), "variable EE_PROJECT"),
                          (proyecto_de_credenciales(), "credenciales locales de Earth Engine")):
        if str(valor or "").strip():
            return str(valor).strip(), origen
    return "", ""


def conectar_ee(proyecto, autenticar=True):
    """Inicializa Earth Engine con el proyecto indicado y confirma permisos con una petición real."""
    proyecto, origen = resolver_proyecto(proyecto)
    if not proyecto and autenticar:
        print("Sin proyecto ni credenciales: se abre la autenticación de Earth Engine.")
        try:
            ee.Authenticate()
        except Exception as exc:
            print(f"  Autenticación no completada: {str(exc)[:300]}")
        proyecto, origen = resolver_proyecto("")
    if not proyecto:
        print("Falta el proyecto: escribe proyecto_ee = 'mi-proyecto' en la celda 1, define EE_PROJECT "
              "o fija uno con `earthengine set_project mi-proyecto`.")
        return None
    for intento in (1, 2):
        try:
            ee.Initialize(project=proyecto)
            ee.Number(1).getInfo()
            print(f"Earth Engine listo · proyecto '{proyecto}' ({origen}) · earthengine-api {ee.__version__}")
            return proyecto
        except Exception as exc:
            tipo, consejo = diagnosticar_error_ee(exc)
            if tipo == "autorización" and autenticar and intento == 1:
                print("Sin autorización válida: se inicia la autenticación interactiva de Earth Engine…")
                try:
                    ee.Authenticate()
                    continue
                except Exception as exc_auth:
                    exc = exc_auth
            print(f"Fallo de {tipo.upper()} al conectar con Earth Engine (proyecto '{proyecto}', {origen}).")
            print(f"  {consejo}\n  Detalle: {str(exc)[:400]}")
            return None
    return None


proyecto_activo = conectar_ee(proyecto_ee, autenticar_si_falta)
ee_listo = proyecto_activo is not None
if not ee_listo:
    print("Las celdas que consultan Earth Engine se omitirán; las pruebas locales (sección 8) sí se ejecutan.")

Earth Engine listo · proyecto 'mapsapi1-203022' (credenciales locales de Earth Engine) · earthengine-api 1.7.35


## 3. Área de interés, malla y fechas

Se calcula **una sola cuadrícula por AOI**: los límites del AOI se proyectan al CRS de destino y se ajustan hacia afuera a múltiplos de 10 m. Todas las fechas y productos usan el mismo origen, ancho, alto y afín. El `id_aoi` incorpora una huella de la geometría y de la malla. Las imágenes **no** se redimensionan a 640 × 640; el teselado para YOLO se decide al preparar el dataset.

In [161]:
# 3a. Funciones: AOI, CRS, malla común y fechas.
@dataclass(frozen=True)
class Malla:
    """Cuadrícula única del AOI: misma afín, filas y columnas para todas las fechas y productos."""
    crs: str
    transform: tuple  # (a, b, c, d, e, f) de rasterio: x = c + a·col, y = f + e·fila
    ancho: int
    alto: int

    @property
    def afin(self):
        return Affine(*self.transform)

    @property
    def limites(self):
        a, _, x0, _, e, y0 = self.transform
        return (x0, y0 + e * self.alto, x0 + a * self.ancho, y0)

    def como_dict(self):
        return {"crs": self.crs, "transform": list(self.transform), "ancho": self.ancho,
                "alto": self.alto, "resolucion_m": self.transform[0], "limites": list(self.limites)}


def normalizar_aoi(entrada):
    """Convierte una caja [oeste, sur, este, norte] o un GeoJSON Polygon/MultiPolygon WGS84 en geometría válida."""
    if isinstance(entrada, (str, Path)):
        texto = str(entrada).strip()
        entrada = json.loads(texto if texto.startswith("{") else Path(texto).read_text(encoding="utf-8"))
    if isinstance(entrada, (list, tuple)):
        if len(entrada) != 4:
            raise ValueError("La caja debe tener 4 valores: [oeste, sur, este, norte].")
        oeste, sur, este, norte = (float(v) for v in entrada)
        if not (oeste < este and sur < norte):
            raise ValueError(f"Caja invertida o vacía {list(entrada)}: se requiere oeste < este y sur < norte.")
        geom = box(oeste, sur, este, norte)
    elif isinstance(entrada, dict):
        g = entrada
        if g.get("type") == "FeatureCollection":
            if len(g.get("features", [])) != 1:
                raise ValueError("El FeatureCollection debe contener exactamente un polígono.")
            g = g["features"][0]
        if g.get("type") == "Feature":
            g = g.get("geometry") or {}
        if g.get("type") not in ("Polygon", "MultiPolygon"):
            raise ValueError(f"Tipo GeoJSON no admitido: {g.get('type')!r}. Usa Polygon o MultiPolygon.")
        try:
            geom = shape(g)
        except Exception as exc:
            raise ValueError(f"GeoJSON mal formado: {exc}") from None
    else:
        raise TypeError("aoi_entrada debe ser una caja [oeste, sur, este, norte] o un GeoJSON.")
    if geom.is_empty or geom.area == 0:
        raise ValueError("La geometría del AOI está vacía o no tiene área.")
    if not geom.is_valid:
        raise ValueError(f"Geometría inválida: {explain_validity(geom)}")
    minx, miny, maxx, maxy = geom.bounds
    if not (-180 <= minx and maxx <= 180 and -90 <= miny and maxy <= 90):
        raise ValueError("Coordenadas fuera de rango: usa [longitud, latitud] en grados WGS84.")
    return geom


def zona_utm(lon):
    return min(int((lon + 180) // 6) + 1, 60)


def elegir_crs(geom, crs_explicito=None):
    """CRS métrico explícito o UTM derivado del centro; exige CRS explícito en zonas polares o si cruza zonas UTM."""
    if crs_explicito:
        crs = CRS.from_user_input(crs_explicito)
        unidad = crs.axis_info[0].unit_name if crs.axis_info else ""
        if not crs.is_projected or unidad not in ("metre", "meter"):
            raise ValueError(f"crs_salida debe ser proyectado y en metros (recibido: {crs_explicito}).")
        epsg = crs.to_epsg()
        return f"EPSG:{epsg}" if epsg else crs.to_wkt()
    minx, miny, maxx, maxy = geom.bounds
    if maxy > 84 or miny < -80:
        raise ValueError("AOI en zona polar: define crs_salida explícito (p. ej. EPSG:3413 o EPSG:3031).")
    if zona_utm(minx) != zona_utm(maxx):
        raise ValueError(f"El AOI cruza las zonas UTM {zona_utm(minx)} y {zona_utm(maxx)}: "
                         "define crs_salida con un CRS métrico explícito.")
    c = geom.centroid
    return f"EPSG:{(32600 if c.y >= 0 else 32700) + zona_utm(c.x)}"


def calcular_malla(geom, crs, res=RESOLUCION_M):
    """Proyecta el AOI (bordes densificados), ajusta hacia afuera a múltiplos de res y devuelve (Malla, AOI proyectado)."""
    tr = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
    geom_proj = transformar_geometria(tr.transform, shapely.segmentize(geom, 0.0005))
    minx, miny, maxx, maxy = geom_proj.bounds
    x0, y0 = math.floor(minx / res) * res, math.floor(miny / res) * res
    x1, y1 = math.ceil(maxx / res) * res, math.ceil(maxy / res) * res
    malla = Malla(crs, (float(res), 0.0, float(x0), 0.0, -float(res), float(y1)),
                  int(round((x1 - x0) / res)), int(round((y1 - y0) / res)))
    return malla, geom_proj


def mascara_aoi(malla, geom_proj):
    """True en los píxeles cuyo centro cae dentro de la geometría real del AOI."""
    return geometry_mask([geom_proj], out_shape=(malla.alto, malla.ancho), transform=malla.afin, invert=True)


def huella(obj, n=10):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str).encode()).hexdigest()[:n]


def geojson_wgs84(geom):
    return json.loads(shapely.to_geojson(geom))


def aoi_a_ee(geom):
    """Geometría de Earth Engine con aristas planas en lon/lat (igual que la geometría local)."""
    return ee.Geometry(geojson_wgs84(geom), None, False)


def intervalo_fechas(fecha, ventana):
    """Devuelve (fecha, primer día, último día inclusivo, fin exclusivo para filterDate)."""
    f = date.fromisoformat(str(fecha))
    if isinstance(ventana, bool) or not isinstance(ventana, int) or not 0 <= ventana <= 3650:
        raise ValueError("ventana_dias debe ser un entero entre 0 y 3650.")
    return f, f - timedelta(days=ventana), f + timedelta(days=ventana), f + timedelta(days=ventana + 1)


def preparar_contexto(aoi, nombre, crs_explicito, fecha, ventana, min_cob, min_util):
    """Reúne AOI, malla, fechas y umbrales; su 'firma' detecta consultas obsoletas."""
    for nombre_umbral, valor in (("min_cobertura_pct", min_cob), ("min_util_pct", min_util)):
        if not 0 <= float(valor) <= 100:
            raise ValueError(f"{nombre_umbral} debe estar entre 0 y 100.")
    geom = normalizar_aoi(aoi)
    crs = elegir_crs(geom, crs_explicito)
    malla, geom_proj = calcular_malla(geom, crs)
    nombre = re.sub(r"[^A-Za-z0-9_-]+", "_", str(nombre)).strip("_") or "aoi"
    id_aoi = f"{nombre}_{huella({'geom': shapely.to_wkt(geom, rounding_precision=7), 'malla': malla.como_dict()})}"
    f, ini, fin, fin_excl = intervalo_fechas(fecha, ventana)
    ctx = {"geom": geom, "geom_proj": geom_proj, "crs": crs, "malla": malla, "id_aoi": id_aoi,
           "nombre_aoi": nombre, "fecha": f, "ventana": ventana, "inicio": ini, "fin": fin,
           "fin_exclusivo": fin_excl, "min_cob": float(min_cob), "min_util": float(min_util)}
    ctx["firma"] = huella({"id_aoi": id_aoi, "ini": ini, "fin": fin, "cob": ctx["min_cob"],
                           "util": ctx["min_util"], "col": COLECCION, "fecha": f})
    return ctx


def contexto_actual():
    """Contexto derivado de los valores vigentes de la celda 1."""
    return preparar_contexto(aoi_entrada, nombre_aoi, crs_salida, fecha_objetivo, ventana_dias,
                             min_cobertura_pct, min_util_pct)


def plan_bloques(malla, tam):
    """Ventanas (fila0, col0, alto, ancho) que cubren la malla sin solaparse."""
    if not 16 <= int(tam) <= LIMITE_DIM_PETICION:
        raise ValueError(f"tamano_bloque debe estar entre 16 y {LIMITE_DIM_PETICION}.")
    return [(f, c, min(tam, malla.alto - f), min(tam, malla.ancho - c))
            for f in range(0, malla.alto, tam) for c in range(0, malla.ancho, tam)]


class MapaSeguro(geemap.Map):
    """geemap.Map que ignora la vista nula que envía el visor mientras el mapa aún no tiene tamaño (VS Code)."""
    _CLAVES_VISTA = {"north", "south", "east", "west", "bottom", "top", "right", "left", "center", "zoom"}

    def set_state(self, sync_data):
        super().set_state({k: v for k, v in sync_data.items()
                           if k not in self._CLAVES_VISTA
                           or not (v is None or (isinstance(v, (list, tuple)) and None in v))})


def mapa_base(geom):
    minx, miny, maxx, maxy = geom.bounds
    mapa = MapaSeguro(center=((miny + maxy) / 2, (minx + maxx) / 2), zoom=13, ee_initialize=False)
    mapa.add_basemap("OpenTopoMap")
    mapa.add_basemap("Esri.WorldImagery")
    mapa.fit_bounds([[miny, minx], [maxy, maxx]])
    return mapa


def dibujar_aoi(mapa, ctx, puntos):
    """AOI (amarillo), rectángulo de la malla (cian discontinuo) y puntos de referencia."""
    # Capa ipyleaflet directa: add_geojson de geemap recolorea los Feature y su aviso emergente falla con geometrías.
    def capa(geom, nombre, estilo):
        mapa.add(ipyleaflet.GeoJSON(data={"type": "Feature", "properties": {"capa": nombre},
                                          "geometry": geojson_wgs84(geom)}, style=estilo, name=nombre))
    capa(ctx["geom"], "AOI", {"color": "#ffff00", "weight": 2, "fillOpacity": 0.0})
    inv = Transformer.from_crs(ctx["crs"], "EPSG:4326", always_xy=True)
    rect = transformar_geometria(inv.transform, shapely.segmentize(box(*ctx["malla"].limites), 50))
    capa(rect, "Malla de salida", {"color": "#00ffff", "weight": 1, "dashArray": "6 4", "fillOpacity": 0.0})
    for nombre, (lon, lat) in (puntos or {}).items():
        mapa.add(ipyleaflet.Marker(location=(lat, lon), title=nombre, draggable=False))

In [162]:
# 3b. Área, malla y fechas vigentes.
ctx = contexto_actual()
m = ctx["malla"]
km2 = ctx["geom_proj"].area / 1e6
print(f"AOI        : {ctx['id_aoi']} · límites WGS84 {tuple(round(v, 4) for v in ctx['geom'].bounds)} · {km2:.1f} km²")
print(f"CRS        : {ctx['crs']} · resolución {RESOLUCION_M} m")
print(f"Malla      : {m.ancho} × {m.alto} px · origen (x0, y0) = ({m.transform[2]:.0f}, {m.transform[5]:.0f}) · "
      f"afín {tuple(m.transform)}")
n_bloques = len(plan_bloques(m, tamano_bloque))
mb = m.ancho * m.alto * (5 * 4 + 2 * 4 + 1 + 1 + 3 + 3) / 1e6
print(f"Descarga   : {n_bloques} bloque(s) por producto (≤ {tamano_bloque}×{tamano_bloque} px) · "
      f"≈ {mb:.0f} MB por escena sin comprimir")
print(f"Fechas UTC : {ctx['inicio']} a {ctx['fin']}, ambos inclusive ({(ctx['fin'] - ctx['inicio']).days + 1} días) · "
      f"filterDate('{ctx['inicio']}', '{ctx['fin_exclusivo']}') con fin exclusivo")
for nombre, (lon, lat) in puntos_referencia.items():
    estado = "dentro" if ctx["geom"].contains(Point(lon, lat)) else "FUERA"
    print(f"Referencia : {nombre} ({lon}, {lat}) -> {estado} del AOI")

mapa_aoi = mapa_base(ctx["geom"])
dibujar_aoi(mapa_aoi, ctx, puntos_referencia)
mapa_aoi

AOI        : santa_isabel_54c3fecf1b · límites WGS84 (-75.397, 4.773, -75.348, 4.827) · 32.4 km²
CRS        : EPSG:32618 · resolución 10 m
Malla      : 545 × 598 px · origen (x0, y0) = (455970, 533560) · afín (10.0, 0.0, 455970.0, 0.0, -10.0, 533560.0)
Descarga   : 4 bloque(s) por producto (≤ 512×512 px) · ≈ 12 MB por escena sin comprimir
Fechas UTC : 2023-12-31 a 2024-01-30, ambos inclusive (31 días) · filterDate('2023-12-31', '2024-01-31') con fin exclusivo


MapaSeguro(center=[4.8, -75.3725], controls=(WidgetControl(options=['position', 'transparent_bg'], position='t…

## 4. Búsqueda de adquisiciones

Cada fila es **un gránulo Level-2A** de `COPERNICUS/S2_SR_HARMONIZED` (sin medianas ni mezcla de días). Si el AOI intersecta varias teselas MGRS, cada una aparece por separado aunque compartan fecha.

- **Cobertura** = área del AOI con datos en B2, B3, B4, B8, B11 y SCL ÷ área **total** del AOI.
- **Fracción útil** = área con datos y SCL admitida (4 vegetación, 5 suelo, 6 agua, 11 nieve/hielo) ÷ el mismo total. Se excluyen 0, 1, 2, 3, 7, 8, 9 y 10 (regla conservadora).
- Ambos se calculan con sumas de área de píxel sobre una malla común de 20 m. Una escena que cubre medio AOI nunca alcanza 95 % de cobertura. La nube del gránulo es solo informativa.
- Orden: cumplimiento de umbrales, distancia en días, mayor fracción útil e ID. Las descartadas también se muestran con su motivo.

In [163]:
# 4a. Funciones de búsqueda y calidad.
PROPIEDADES_S2 = {"satelite": "SPACECRAFT_NAME", "tesela": "MGRS_TILE", "orbita": "SENSING_ORBIT_NUMBER",
                  "direccion_orbita": "SENSING_ORBIT_DIRECTION", "baseline": "PROCESSING_BASELINE",
                  "nube_granulo_pct": "CLOUDY_PIXEL_PERCENTAGE"}
consulta_actual = None


def coleccion_consulta(ctx):
    return (ee.ImageCollection(COLECCION).filterBounds(aoi_a_ee(ctx["geom"]))
            .filterDate(ctx["inicio"].isoformat(), ctx["fin_exclusivo"].isoformat()))


def transformacion_diag(malla):
    """Malla de diagnóstico a 20 m anclada en el origen de la malla de salida."""
    return [RESOLUCION_DIAG_M, 0, malla.transform[2], 0, -RESOLUCION_DIAG_M, malla.transform[5]]


def sumar_area_ee(imagen, ctx):
    return imagen.reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi_a_ee(ctx["geom"]), crs=ctx["malla"].crs,
                               crsTransform=transformacion_diag(ctx["malla"]), maxPixels=1e10)


def capas_calidad_ee(img):
    """Imágenes 0/1 sin máscara: datos completos (5 bandas + SCL) y píxel útil (datos y SCL admitida)."""
    datos = img.select(BANDAS_REFL + ["SCL"]).mask().reduce(ee.Reducer.min()).gt(0).unmask(0, False)
    admitida = img.select("SCL").remap(list(SCL_ADMITIDAS), [1] * len(SCL_ADMITIDAS), 0).unmask(0, False)
    return datos, datos.And(admitida)


def area_total_aoi(ctx):
    return sumar_area_ee(ee.Image.pixelArea().rename("a"), ctx).getNumber("a")


def funcion_anotar(ctx, area_total, indices_ordenados):
    """Devuelve una función que resume cada imagen en un Feature con porcentajes y clave de orden."""
    t0 = ee.Date(ctx["fecha"].isoformat()).millis()

    def anotar(img):
        datos, util = capas_calidad_ee(img)
        area = ee.Image.pixelArea()
        sumas = sumar_area_ee(ee.Image.cat(area.multiply(datos).rename("datos"),
                                           area.multiply(util).rename("util")), ctx)
        cob = sumas.getNumber("datos").divide(area_total).multiply(100)
        utl = sumas.getNumber("util").divide(area_total).multiply(100)
        dist = ee.Number(img.get("system:time_start")).subtract(t0).divide(86400000).floor().abs()
        cumple = cob.gte(ctx["min_cob"]).And(utl.gte(ctx["min_util"]))
        rango = ee.List(indices_ordenados).indexOf(img.get("system:index"))
        # Clave numérica exacta: cumple, distancia, mayor útil, ID (orden alfabético).
        clave = (ee.Number(1).subtract(cumple).multiply(1e15).add(dist.multiply(1e11))
                 .add(ee.Number(100).subtract(utl).max(0).multiply(100).round().multiply(1e4)).add(rango))
        props = {"id": img.get("system:id"), "indice": img.get("system:index"), "t": img.get("system:time_start"),
                 "cobertura_pct": cob, "util_pct": utl, "dist_dias": dist, "cumple": cumple, "clave_orden": clave}
        props.update({k: img.get(v) for k, v in PROPIEDADES_S2.items()})
        return ee.Feature(None, props)

    return anotar


def coleccion_anotada(ctx, col, area_total):
    indices = col.aggregate_array("system:index").sort()
    return ee.FeatureCollection(col.map(funcion_anotar(ctx, area_total, indices))).sort("clave_orden")


def estado_fila(p, ctx):
    motivos = []
    if p["cobertura_pct"] < ctx["min_cob"]:
        motivos.append(f"cobertura {p['cobertura_pct']:.1f} % < {ctx['min_cob']:g} %")
    if p["util_pct"] < ctx["min_util"]:
        motivos.append(f"útil {p['util_pct']:.1f} % < {ctx['min_util']:g} %")
    return "cumple" if not motivos else "descartada: " + "; ".join(motivos)


def fecha_utc(t_ms):
    return datetime.fromtimestamp(t_ms / 1000, tz=timezone.utc)


def tabla_candidatos(filas, ctx, desplazamiento=0):
    registros = [{
        "id": p["id"], "fecha_hora_utc": fecha_utc(p["t"]).strftime("%Y-%m-%d %H:%M:%S"),
        "dist_dias": int(p["dist_dias"]), "cobertura_pct": round(p["cobertura_pct"], 1),
        "util_pct": round(p["util_pct"], 1),
        "nube_granulo_pct": None if p.get("nube_granulo_pct") is None else round(p["nube_granulo_pct"], 1),
        "estado": estado_fila(p, ctx), "satelite": p.get("satelite"), "tesela": p.get("tesela"),
        "orbita": None if p.get("orbita") is None else int(p["orbita"]), "baseline": p.get("baseline"),
    } for p in filas]
    tabla = pd.DataFrame(registros)
    tabla.index = range(desplazamiento + 1, desplazamiento + 1 + len(tabla))
    return tabla


def buscar(ctx, filas_pag, pag):
    """Consulta la colección; solo viaja al cliente la página pedida (más la lista de IDs para validar selecciones)."""
    col = coleccion_consulta(ctx)
    resumen = ee.Dictionary({"n": col.size(), "ids": col.aggregate_array("system:id")}).getInfo()
    n, ids = int(resumen["n"]), list(resumen["ids"])
    consulta = {"firma": ctx["firma"], "ids": ids, "n": n, "filas": {}, "orden_pagina": [],
                "pagina": 1, "paginas": 0, "n_cumple": 0, "primera_fila": 1}
    if n == 0:
        return consulta, pd.DataFrame()
    if n > 9999:
        raise ValueError(f"La consulta devuelve {n} gránulos: reduce ventana_dias o el AOI.")
    filas_pag = max(1, int(filas_pag))
    paginas = math.ceil(n / filas_pag)
    if not 1 <= int(pag) <= paginas:
        print(f"La página {pag} no existe (1–{paginas}); se muestra la página 1.")
        pag = 1
    area_total = area_total_aoi(ctx)
    fc = coleccion_anotada(ctx, col, area_total)
    datos = ee.Dictionary({"filas": fc.toList(filas_pag, (pag - 1) * filas_pag),
                           "n_cumple": fc.aggregate_sum("cumple")}).getInfo()
    filas = [f["properties"] for f in datos["filas"]]
    consulta.update(filas={p["id"]: p for p in filas}, orden_pagina=[p["id"] for p in filas],
                    pagina=pag, paginas=paginas, n_cumple=int(datos["n_cumple"]),
                    primera_fila=(pag - 1) * filas_pag + 1)
    return consulta, tabla_candidatos(filas, ctx, (pag - 1) * filas_pag)


def filas_por_ids(ctx, ids):
    """Recalcula en Earth Engine las propiedades de calidad de IDs concretos de la consulta."""
    col = coleccion_consulta(ctx).filter(ee.Filter.inList("system:id", list(ids)))
    fc = coleccion_anotada(ctx, col, area_total_aoi(ctx))
    return {f["properties"]["id"]: f["properties"] for f in fc.getInfo()["features"]}


def numero_fila(id_, consulta):
    """Número de fila con el que la tabla vigente muestra un ID (None si no está en la página)."""
    orden = consulta.get("orden_pagina", [])
    return consulta.get("primera_fila", 1) + orden.index(id_) if id_ in orden else None


def etiqueta_fila(id_, consulta, ctx):
    p = consulta["filas"][id_]
    return (f"{numero_fila(id_, consulta)} · {fecha_utc(p['t']):%Y-%m-%d %H:%M} UTC · {p.get('tesela')} · "
            f"cob. {p['cobertura_pct']:.0f} % · útil {p['util_pct']:.1f} % · {estado_fila(p, ctx)}")


def validar_ids(ids, consulta, ctx):
    """Acepta números de fila de la tabla vigente o IDs de la consulta; rechaza ajenos u obsoletos."""
    if consulta is None:
        raise RuntimeError("Primero ejecuta la búsqueda (sección 4).")
    if consulta["firma"] != ctx["firma"]:
        raise RuntimeError("Los parámetros de la celda 1 cambiaron desde la última búsqueda: "
                           "vuelve a ejecutar las secciones 3 y 4 antes de inspeccionar o descargar.")
    if isinstance(ids, (str, int)) or not isinstance(ids, (list, tuple)):
        raise TypeError("Usa una lista de números de fila o IDs, p. ej. [1, 3].")
    validos = set(consulta["ids"])
    por_indice = {i.split("/")[-1]: i for i in consulta["ids"]}
    orden, primera = consulta.get("orden_pagina", []), consulta.get("primera_fila", 1)
    salida, rechazados = [], []
    for i in ids:
        if isinstance(i, bool):
            rechazados.append(i)
            continue
        if isinstance(i, int):
            if not orden or not primera <= i < primera + len(orden):
                rango = f"{primera}–{primera + len(orden) - 1}" if orden else "ninguna"
                raise ValueError(f"La fila {i} no está en la tabla mostrada (filas {rango}); "
                                 "cambia `pagina` en la celda 1 o usa el ID.")
            completo = orden[i - primera]
            print(f"Fila {i} -> {completo}")
        else:
            i = str(i).strip()
            completo = i if i in validos else por_indice.get(i)
            if completo is None:
                rechazados.append(i)
                continue
            if completo != i:
                print(f"Nota: '{i}' se interpreta como '{completo}'.")
        if completo not in salida:
            salida.append(completo)
    if rechazados:
        raise ValueError(f"IDs ajenos a la consulta actual u obsoletos: {rechazados}. "
                         "Usa números de fila o IDs de la tabla vigente de la sección 4.")
    return salida

In [164]:
# 4b. Búsqueda con los parámetros vigentes.
consulta_actual, tabla = None, pd.DataFrame()
if not ee_listo:
    print("Sin conexión a Earth Engine: búsqueda omitida.")
else:
    ctx = contexto_actual()
    t_inicio = time.time()
    consulta_actual, tabla = buscar(ctx, filas_por_pagina, pagina)
    if consulta_actual["n"] == 0:
        print(f"Sin adquisiciones entre {ctx['inicio']} y {ctx['fin']} (UTC, inclusive) para el AOI {ctx['id_aoi']} "
              f"{tuple(round(v, 3) for v in ctx['geom'].bounds)}.\n"
              "Amplía ventana_dias o cambia fecha_objetivo en la celda 1 y vuelve a ejecutar. "
              "No se sustituye automáticamente por fechas vecinas y no se descargará nada.")
    else:
        c = consulta_actual
        teselas = sorted({i.split("_")[-1] for i in c["ids"]})
        print(f"{c['n']} gránulo(s) entre {ctx['inicio']} y {ctx['fin']} (UTC, inclusive) · cumplen umbrales: "
              f"{c['n_cumple']} · teselas: {', '.join(teselas)} · {time.time() - t_inicio:.0f} s")
        if len(teselas) > 1:
            print("El AOI intersecta varias teselas: cada gránulo se lista y se descarga por separado (cobertura parcial).")
        print(f"Página {c['pagina']} de {c['paginas']}" +
              (f" · hay otra página: pagina = {c['pagina'] + 1}" if c["pagina"] < c["paginas"] else ""))
        with pd.option_context("display.max_colwidth", None, "display.width", 250, "display.max_rows", 500):
            display(tabla)
        print("Elige por número de fila (columna izquierda): desplegable en la sección 5 para inspeccionar, "
              "casillas en la sección 6 para descargar.")

6 gránulo(s) entre 2023-12-31 y 2024-01-30 (UTC, inclusive) · cumplen umbrales: 3 · teselas: T18NVL · 4 s
Página 1 de 1


,id,fecha_hora_utc,dist_dias,cobertura_pct,util_pct,nube_granulo_pct,estado,satelite,tesela,orbita,baseline
1,COPERNICUS/S2_SR_HARMONIZED/20240108T152651_20240108T152927_T18NVL,2024-01-08 15:31:43,7,100.0,96.6,38.5,cumple,Sentinel-2A,18NVL,25,05.10
2,COPERNICUS/S2_SR_HARMONIZED/20240123T152649_20240123T152651_T18NVL,2024-01-23 15:31:47,8,100.0,97.7,5.9,cumple,Sentinel-2B,18NVL,25,05.10
3,COPERNICUS/S2_SR_HARMONIZED/20240103T152649_20240103T152652_T18NVL,2024-01-03 15:31:48,12,100.0,95.5,46.2,cumple,Sentinel-2B,18NVL,25,05.10
4,COPERNICUS/S2_SR_HARMONIZED/20240113T152649_20240113T152650_T18NVL,2024-01-13 15:31:46,2,100.0,2.1,74.2,descartada: útil 2.1 % < 70 %,Sentinel-2B,18NVL,25,05.10
5,COPERNICUS/S2_SR_HARMONIZED/20240118T152641_20240118T152922_T18NVL,2024-01-18 15:31:39,3,100.0,13.2,52.6,descartada: útil 13.2 % < 70 %,Sentinel-2A,18NVL,25,05.10
6,COPERNICUS/S2_SR_HARMONIZED/20240128T152651_20240128T152904_T18NVL,2024-01-28 15:31:43,13,100.0,60.1,42.1,descartada: útil 60.1 % < 70 %,Sentinel-2A,18NVL,25,05.10


Elige por número de fila (columna izquierda): desplegable en la sección 5 para inspeccionar, casillas en la sección 6 para descargar.


## 5. Inspección de una adquisición

Muestra sobre todo el AOI: RGB **sin** enmascarar nubes (B4-B3-B2, 0–0,60 de reflectancia, la misma escala del PNG), NDSI con leyenda, máscara de calidad SCL conmutable (capa apagada por defecto; actívala en el control de capas) y en gris translúcido los sectores sin observación. Cambia de escena con el **desplegable**: el mapa se actualiza en el mismo lugar. Inspeccionar **no** agrega escenas a la descarga. La decisión final se toma con esta vista y con los indicadores del AOI.

In [165]:
# 5a. Funciones de inspección.
PALETA_NDSI = ["8c510a", "d8b365", "f6e8c3", "f5f5f5", "c7eae5", "5ab4ac", "01665e"]


def ndsi_ee(img):
    b3, b11 = img.select("B3").multiply(ESCALA_REFL), img.select("B11").multiply(ESCALA_REFL)
    den = b3.add(b11)
    return b3.subtract(b11).divide(den).updateMask(den.neq(0)).rename("NDSI")


def resumen_scl(img, ctx):
    """Porcentaje del área total del AOI por clase SCL (255 = fuera del gránulo)."""
    scl = img.select("SCL").unmask(255, False)
    res = ee.Image.pixelArea().addBands(scl).reduceRegion(
        reducer=ee.Reducer.sum().group(1, "clase"), geometry=aoi_a_ee(ctx["geom"]), crs=ctx["malla"].crs,
        crsTransform=transformacion_diag(ctx["malla"]), maxPixels=1e10)
    datos = ee.Dictionary({"grupos": res.get("groups"), "total": area_total_aoi(ctx)}).getInfo()
    filas = [{"clase_scl": int(g["clase"]), "descripcion": NOMBRES_SCL.get(int(g["clase"]), "?"),
              "regla": "admitida" if int(g["clase"]) in SCL_ADMITIDAS else
                       ("sin observación" if int(g["clase"]) in (0, 255) else "excluida"),
              "area_pct": round(100 * g["sum"] / datos["total"], 2)} for g in datos["grupos"]]
    return pd.DataFrame(filas).sort_values("clase_scl").reset_index(drop=True)


VIS_RGB = {"min": 0, "max": RGB_MAX_REFL}
VIS_NDSI = {"min": -1, "max": 1, "palette": PALETA_NDSI}


def capas_inspeccion(img, ctx):
    """Capas EE de una escena: (objeto, visualización, nombre, visible, opacidad)."""
    aoi_ee = aoi_a_ee(ctx["geom"])
    rgb = img.select(["B4", "B3", "B2"]).multiply(ESCALA_REFL)
    datos, util = capas_calidad_ee(img)
    calidad = datos.add(util)
    return [
        (rgb, VIS_RGB, "RGB contexto (sin recorte)", False, 1.0),
        (rgb.clip(aoi_ee), VIS_RGB, "RGB B4-B3-B2 (sin máscara de nubes)", True, 1.0),
        (ndsi_ee(img).clip(aoi_ee), VIS_NDSI, "NDSI", False, 1.0),
        (calidad.updateMask(calidad.gt(0)).clip(aoi_ee), {"min": 1, "max": 2, "palette": ["e41a1c", "4daf4a"]},
         "Calidad SCL", False, 0.5),
        (datos.Not().selfMask().clip(aoi_ee), {"palette": ["9e9e9e"]}, "Sin observación", True, 0.6),
        (ee.FeatureCollection([ee.Feature(aoi_ee)]).style(color="ffff00", fillColor="00000000", width=2),
         {}, "AOI", True, 1.0),
    ]


def pintar_escena(mapa, img, ctx):
    """Agrega o sustituye las capas de la escena; conserva la visibilidad y opacidad que elegiste en el mapa."""
    for obj, vis, nombre, visible, opacidad in capas_inspeccion(img, ctx):
        previa = mapa.ee_layers.get(nombre, {}).get("ee_layer")
        if previa is not None:
            visible, opacidad = previa.visible, previa.opacity
        mapa.addLayer(obj, vis, nombre, visible, opacidad)
    mapa.add_colorbar(VIS_NDSI, label="NDSI", layer_name="NDSI")  # sustituir la capa NDSI retira su barra


def mapa_inspeccion(img, ctx):
    mapa = mapa_base(ctx["geom"])
    pintar_escena(mapa, img, ctx)
    mapa.add_legend(title="Calidad SCL", legend_dict={"Admitida (4, 5, 6, 11)": "4daf4a",
                                                      "Excluida por SCL": "e41a1c",
                                                      "Sin observación (gris translúcido)": "9e9e9e"})
    return mapa


def resumen_inspeccion(id_insp, consulta, ctx):
    """Imprime el resumen y la tabla SCL de un ID de la consulta vigente; devuelve la ee.Image."""
    fila = consulta["filas"].get(id_insp) or filas_por_ids(ctx, [id_insp])[id_insp]
    img = ee.Image(id_insp)
    n = numero_fila(id_insp, consulta)
    print(f"Inspección: {'fila ' + str(n) + ' · ' if n else ''}{id_insp}")
    print(f"  Adquisición {fecha_utc(fila['t']):%Y-%m-%d %H:%M:%S} UTC · {fila.get('satelite')} · tesela {fila.get('tesela')} · "
          f"{int(fila['dist_dias'])} día(s) de la fecha objetivo")
    print(f"  Cobertura {fila['cobertura_pct']:.1f} % · útil {fila['util_pct']:.1f} % · nube del gránulo "
          f"{fila.get('nube_granulo_pct')} % · {estado_fila(fila, ctx)}")
    print("  (Esta vista no agrega la escena a la descarga.)")
    display(resumen_scl(img, ctx))
    return img

In [166]:
# 5b. Inspección: elige la escena en el desplegable (o fija id_vista_previa en la celda 1).
if not ee_listo:
    print("Sin conexión a Earth Engine: inspección omitida.")
elif consulta_actual is None or consulta_actual["n"] == 0:
    print("No hay candidatos en la consulta vigente: ajusta fecha o ventana en la celda 1 y repite la sección 4.")
else:
    ctx = contexto_actual()
    _inicial = id_vista_previa if isinstance(id_vista_previa, int) else (str(id_vista_previa).strip()
                                                                          or consulta_actual["primera_fila"])
    [id_insp] = validar_ids([_inicial], consulta_actual, ctx)
    _opciones = [(etiqueta_fila(i, consulta_actual, ctx), i) for i in consulta_actual["orden_pagina"]]
    if id_insp not in consulta_actual["orden_pagina"]:
        _opciones.insert(0, (id_insp, id_insp))
    selector_vista = widgets.Dropdown(options=_opciones, value=id_insp, description="Escena:",
                                      layout=widgets.Layout(width="95%"), style={"description_width": "60px"})
    # El resumen va en un Output; el mapa se muestra una sola vez fuera de él y solo se le cambian las capas.
    salida_vista = widgets.Output()
    with salida_vista:
        img_insp = resumen_inspeccion(id_insp, consulta_actual, ctx)
    mapa_insp = mapa_inspeccion(img_insp, ctx)

    def _al_cambiar_vista(cambio):
        global id_insp, img_insp
        id_insp = cambio["new"]
        salida_vista.clear_output(wait=True)
        with salida_vista:
            try:
                c = contexto_actual()
                img_insp = resumen_inspeccion(id_insp, consulta_actual, c)
                pintar_escena(mapa_insp, img_insp, c)
            except Exception as exc:
                print(f"No se pudo inspeccionar: {exc}")

    selector_vista.observe(_al_cambiar_vista, names="value")
    display(selector_vista, salida_vista, mapa_insp)

Fila 1 -> COPERNICUS/S2_SR_HARMONIZED/20240108T152651_20240108T152927_T18NVL


Dropdown(description='Escena:', layout=Layout(width='95%'), options=(('1 · 2024-01-08 15:31 UTC · 18NVL · cob.…

Output()

MapaSeguro(center=[4.8, -75.3725], controls=(WidgetControl(options=['position', 'transparent_bg'], position='t…

## 6. Selección y descarga

Dos formas equivalentes de elegir, ambas explícitas:

- **Casillas** (si `ids_seleccionados = []`): marca las filas y pulsa «Descargar marcadas». Al terminar se imprime la línea `ids_seleccionados = [...]` para pegarla en la celda 1 si quieres que la selección quede fija al reejecutar.
- **Celda 1**: `ids_seleccionados = [1, 3]` (números de fila de la tabla vigente) o IDs. Se descarga al ejecutar la celda 6b.

Ejecutar todo el notebook con la lista vacía no descarga nada. Se rechazan filas o IDs ajenos a la consulta vigente. Una escena por debajo de los umbrales puede descargarse si se elige explícitamente; queda anotado en `metadata.json`.

**Productos por escena** en `datos_sentinel2/<id_aoi>/<clave>/` (misma malla, CRS, afín, filas y columnas):

| Archivo | Contenido |
|---|---|
| `reflectancia.tif` | float32 B2, B3, B4, B8, B11 (reflectancia = DN × 0,0001), NoData −9999, incluye píxeles de mala calidad |
| `indices.tif` | float32 NDSI = (B3−B11)/(B3+B11), NDVI = (B8−B4)/(B8+B4); denominador cero → NoData −9999 |
| `calidad.tif` | uint8: 0 exterior o sin observación completa, 1 observado y excluido por SCL, 2 observado y admitido |
| `soporte_png.tif` | uint8: 1 píxel representado en ambos PNG, 0 relleno (0,0,0) |
| `rgb/<clave>.png` | R=B4, G=B3, B=B2; round(255 × clip(refl/0,60, 0, 1)) |
| `espectral/<clave>.png` | R=NDSI, G=NDVI (round(255 × (clip(i, −1, 1)+1)/2)), B=B4 con la escala RGB |
| `metadata.json` | procedencia, malla, receta, porcentajes, versiones, huellas SHA-256 |

Bandas continuas: remuestreo bilineal; SCL: vecino más próximo; un píxel solo es válido si toda su vecindad bilineal tiene datos (no se interpolan rellenos). B11 y SCL conservan detalle nativo de 20 m aunque se escriban a 10 m. Los PNG se generan localmente desde los GeoTIFF con orden RGB explícito (si se leen con OpenCV, convertir de BGR). El redondeo es al entero más próximo con mitades hacia arriba.

Descarga por bloques (≤ 512 × 512, < 32 MB y ≤ 10 000 px por dimensión) con `crs`, `crs_transform` y `dimensions` explícitos, reintentos con espera y renovación de la URL temporal. Cada escena se escribe en `_tmp/` y se publica solo tras verificar todos los archivos; una reejecución reutiliza las escenas completas con la misma receta y una receta distinta crea otra revisión (`_r2`, …). `catalogo.csv` se reconstruye desde los `metadata.json` y solo lista escenas completas.

**Para YOLO**: ambos directorios de PNG usan el mismo nombre base y la misma malla, así que un polígono normalizado sobre el RGB vale sin cambios en la variante espectral. `grupo_temporal` agrupa todas las teselas y representaciones de un mismo AOI y día UTC: reserva fechas completas para prueba antes de recortar o aumentar imágenes.

In [167]:
# 6a. Funciones de descarga, productos locales y catálogo.
PRODUCTOS_EE = {"reflectancia": {"bandas": len(BANDAS_REFL), "bytes": 4}, "scl": {"bandas": 1, "bytes": 1}}
ARCHIVOS_TIF = {"reflectancia.tif": (5, "float32", NODATA), "indices.tif": (2, "float32", NODATA),
                "calidad.tif": (1, "uint8", None), "soporte_png.tif": (1, "uint8", None)}


class ErrorTransitorio(Exception):
    pass


def sin_url(texto):
    """Oculta enlaces temporales (contienen tokens) en mensajes y registros."""
    return re.sub(r"https?://\S+", "<url temporal>", str(texto))


def receta_productos(ctx):
    receta = {
        "version": RECETA_VERSION, "coleccion": COLECCION, "id_aoi": ctx["id_aoi"], "malla": ctx["malla"].como_dict(),
        "bandas_reflectancia": BANDAS_REFL, "escala_reflectancia": ESCALA_REFL,
        "remuestreo": {**{b: "bilinear" for b in BANDAS_REFL}, "SCL": "nearest"},
        "mascara_remuestreo": "píxel válido solo si la máscara remuestreada es 1 (no se interpolan rellenos)",
        "nodata": {"reflectancia": NODATA, "indices": NODATA, "calidad": 0, "soporte_png": 0},
        "indices": {"NDSI": "(B3-B11)/(B3+B11)", "NDVI": "(B8-B4)/(B8+B4)", "denominador_cero": "NoData"},
        "calidad": {"scl_admitidas": list(SCL_ADMITIDAS), "scl_excluidas": list(SCL_EXCLUIDAS),
                    "codigos": {"0": "exterior o sin observación completa", "1": "observado y excluido por SCL",
                                "2": "observado y admitido"}},
        "png": {"rgb": "R=B4,G=B3,B=B2; round(255*clip(refl/0.60,0,1))",
                "espectral": "R=NDSI,G=NDVI: round(255*(clip(i,-1,1)+1)/2); B=B4 escala RGB",
                "redondeo": "floor(x+0.5)", "fuera_de_soporte": [0, 0, 0]},
    }
    receta["huella"] = huella(receta, 12)
    return receta


def volumen_bloque(producto, alto, ancho):
    p = PRODUCTOS_EE[producto]
    return alto * ancho * p["bandas"] * p["bytes"]


def transformacion_ventana(malla, fila0, col0):
    a, _, x0, _, e, y0 = malla.transform
    return [a, 0, x0 + col0 * a, 0, e, y0 + fila0 * e]


def imagen_producto_ee(img, producto):
    if producto == "reflectancia":
        r = img.select(BANDAS_REFL).multiply(ESCALA_REFL).resample("bilinear")
        return r.updateMask(r.mask().gte(1)).unmask(NODATA, False).toFloat()
    return img.select("SCL").unmask(0, False).toUint8()  # vecino más próximo (sin resample)


def obtener_bloque_ee(id_asset, producto, malla, ventana, timeout=180, intentos=4):
    """Descarga una ventana con crs, crs_transform y dimensions explícitos; reintenta y renueva la URL."""
    fila0, col0, alto, ancho = ventana
    volumen = volumen_bloque(producto, alto, ancho)
    if volumen > LIMITE_BYTES_PETICION or max(alto, ancho) > LIMITE_DIM_PETICION:
        raise ValueError(f"Bloque {ventana} excede los límites ({volumen / 2**20:.1f} MiB): reduce tamano_bloque.")
    imagen = imagen_producto_ee(ee.Image(id_asset), producto)
    t = transformacion_ventana(malla, fila0, col0)
    params = {"crs": malla.crs, "crs_transform": t, "dimensions": [ancho, alto], "format": "GEO_TIFF"}
    url = None
    for intento in range(1, intentos + 1):
        try:
            if url is None:
                url = imagen.getDownloadURL(params)
            r = requests.get(url, timeout=timeout)
            if r.status_code in (400, 401, 403, 404, 410):
                url = None  # URL caducada o inválida: se renueva
                raise ErrorTransitorio(f"HTTP {r.status_code}: {r.text[:200]}")
            if r.status_code == 429 or r.status_code >= 500:
                raise ErrorTransitorio(f"HTTP {r.status_code}")
            r.raise_for_status()
            with MemoryFile(r.content) as mem, mem.open() as ds:
                if ((ds.width, ds.height) != (ancho, alto) or ds.count != PRODUCTOS_EE[producto]["bandas"]
                        or not ds.transform.almost_equals(Affine(*t), precision=1e-6)
                        or ds.crs != CRSRasterio.from_user_input(malla.crs)):
                    raise ValueError(f"El bloque {ventana} recibido no coincide con la malla solicitada.")
                return ds.read()
        except (ErrorTransitorio, requests.RequestException, ee.EEException) as exc:
            if intento == intentos:
                raise RuntimeError(f"{producto} {ventana}: falló tras {intentos} intentos: {sin_url(exc)}") from None
            espera = min(60, 2 ** intento)
            print(f"      reintento {intento}/{intentos - 1} en {espera} s · {sin_url(exc)[:150]}")
            time.sleep(espera)


def a_uint8_reflectancia(r):
    return np.floor(np.clip(np.asarray(r, dtype=np.float64) / RGB_MAX_REFL, 0, 1) * 255 + 0.5).astype(np.uint8)


def a_uint8_indice(i):
    return np.floor((np.clip(np.asarray(i, dtype=np.float64), -1, 1) + 1) / 2 * 255 + 0.5).astype(np.uint8)


def derivar_productos(refl, scl, dentro):
    """Índices, calidad, soporte y PNG desde las bandas alineadas en float32 (antes de cuantizar)."""
    refl = np.asarray(refl, dtype=np.float32)
    valida = dentro & np.all(np.isfinite(refl) & (refl != NODATA), axis=0)
    b2, b3, b4, b8, b11 = refl.astype(np.float64)
    den_ndsi, den_ndvi = b3 + b11, b8 + b4
    ok_ndsi, ok_ndvi = valida & (den_ndsi != 0), valida & (den_ndvi != 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        ndsi = np.where(ok_ndsi, (b3 - b11) / den_ndsi, NODATA).astype(np.float32)
        ndvi = np.where(ok_ndvi, (b8 - b4) / den_ndvi, NODATA).astype(np.float32)
    observado = valida & (scl != 0)
    calidad = np.where(observado, np.where(np.isin(scl, SCL_ADMITIDAS), 2, 1), 0).astype(np.uint8)
    soporte = ok_ndsi & ok_ndvi
    rf = refl.astype(np.float64)
    rgb = np.stack([a_uint8_reflectancia(np.where(soporte, rf[k], 0)) for k in (2, 1, 0)], axis=-1)
    esp = np.stack([a_uint8_indice(np.where(soporte, ndsi, 0)), a_uint8_indice(np.where(soporte, ndvi, 0)),
                    a_uint8_reflectancia(np.where(soporte, rf[2], 0))], axis=-1)
    rgb[~soporte] = 0
    esp[~soporte] = 0
    util = soporte & (calidad == 2)
    sat = [util & (rf[k] >= RGB_MAX_REFL) for k in (2, 1, 0)]
    conteos = {"dentro": int(dentro.sum()), "observado": int(observado.sum()), "util": int((calidad == 2).sum()),
               "soporte": int(soporte.sum()), "util_soporte": int(util.sum()),
               "sat_alguno": int((sat[0] | sat[1] | sat[2]).sum()),
               "sat_r": int(sat[0].sum()), "sat_g": int(sat[1].sum()), "sat_b": int(sat[2].sum())}
    return {"indices": np.stack([ndsi, ndvi]), "calidad": calidad, "soporte": soporte.astype(np.uint8),
            "rgb": rgb, "espectral": esp, "conteos": conteos}


def porcentajes_locales(c):
    pct = lambda a, b: round(100.0 * a / b, 3) if b else None
    return {"cobertura_pct": pct(c["observado"], c["dentro"]), "util_pct": pct(c["util"], c["dentro"]),
            "soporte_png_pct": pct(c["soporte"], c["dentro"]),
            "saturacion_rgb_pct": pct(c["sat_alguno"], c["util_soporte"]),
            "saturacion_por_canal_pct": {"R": pct(c["sat_r"], c["util_soporte"]),
                                         "G": pct(c["sat_g"], c["util_soporte"]),
                                         "B": pct(c["sat_b"], c["util_soporte"])}}


def perfil_tif(malla, count, dtype, nodata):
    return dict(driver="GTiff", width=malla.ancho, height=malla.alto, count=count, dtype=dtype, crs=malla.crs,
                transform=malla.afin, nodata=nodata, compress="deflate", tiled=True, blockxsize=256, blockysize=256)


def sha256_archivo(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for trozo in iter(lambda: f.read(1 << 20), b""):
            h.update(trozo)
    return h.hexdigest()


def leer_metadata(carpeta):
    try:
        return json.loads((Path(carpeta) / "metadata.json").read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return None


def clave_escena(info):
    """Fecha + tesela + identificador estable del asset (system:index)."""
    tesela = info.get("tesela") or "NA"
    indice = re.sub(rf"_T{tesela}$", "", info["indice"])
    return f"{fecha_utc(info['t']):%Y%m%d}_T{tesela}_{indice}"


def mover_con_reintentos(origen, destino, intentos=6):
    """os.replace con reintentos (carpetas sincronizadas como OneDrive pueden bloquear un instante)."""
    Path(destino).parent.mkdir(parents=True, exist_ok=True)
    for k in range(intentos):
        try:
            os.replace(origen, destino)
            return
        except PermissionError:
            if k == intentos - 1:
                raise
            time.sleep(1 + k)


def verificar_escena(carpeta, malla, n_muestras=6):
    """Comprueba archivos, huellas, malla, tamaños de PNG y valores de PNG recalculados desde los rásteres."""
    carpeta = Path(carpeta)
    meta = leer_metadata(carpeta)
    if meta is None or meta.get("estado") != "completa":
        return False, ["metadata.json ausente o incompleto"], {}
    problemas, detalle = [], {}
    for rel, inf in meta["archivos"].items():
        ruta = carpeta / rel
        if not ruta.is_file():
            problemas.append(f"falta {rel}")
        elif sha256_archivo(ruta) != inf["sha256"]:
            problemas.append(f"huella distinta en {rel}")
    if problemas:
        return False, problemas, detalle
    for nombre, (count, dtype, nodata) in ARCHIVOS_TIF.items():
        with rasterio.open(carpeta / nombre) as ds:
            detalle[nombre] = f"{ds.width}×{ds.height}×{ds.count} {ds.dtypes[0]}"
            if ds.crs != CRSRasterio.from_user_input(malla.crs):
                problemas.append(f"{nombre}: CRS distinto")
            if not ds.transform.almost_equals(malla.afin, precision=1e-6):
                problemas.append(f"{nombre}: afín distinta")
            if (ds.width, ds.height, ds.count, ds.dtypes[0]) != (malla.ancho, malla.alto, count, dtype):
                problemas.append(f"{nombre}: dimensiones o tipo distintos")
            if ds.nodata != nodata:
                problemas.append(f"{nombre}: NoData {ds.nodata} ≠ {nodata}")
    pngs = {}
    for var in ("rgb", "espectral"):
        with Image.open(carpeta / var / f"{meta['nombre_escena']}.png") as im:
            detalle[f"{var}.png"] = f"{im.width}×{im.height} {im.mode}"
            if im.size != (malla.ancho, malla.alto) or im.mode != "RGB":
                problemas.append(f"{var}.png: tamaño o modo distintos")
            pngs[var] = np.asarray(im.convert("RGB"))
    if problemas:
        return False, problemas, detalle
    with rasterio.open(carpeta / "soporte_png.tif") as ds:
        soporte = ds.read(1).astype(bool)
    for var, arr in pngs.items():
        if arr[~soporte].any():
            problemas.append(f"{var}.png tiene valores ≠ 0 fuera del soporte")
    filas, cols = np.nonzero(soporte)
    if len(filas):
        rng = np.random.default_rng(0)
        elegidos = sorted({0, len(filas) // 2, len(filas) - 1, *rng.integers(0, len(filas), n_muestras).tolist()})
        with rasterio.open(carpeta / "reflectancia.tif") as dr, rasterio.open(carpeta / "indices.tif") as di:
            for k in elegidos:
                f, c = int(filas[k]), int(cols[k])
                v = dr.read(window=Window(c, f, 1, 1))[:, 0, 0]
                ndsi, ndvi = di.read(window=Window(c, f, 1, 1))[:, 0, 0]
                esp_rgb = a_uint8_reflectancia([v[2], v[1], v[0]])
                esp_esp = np.array([a_uint8_indice(ndsi), a_uint8_indice(ndvi), a_uint8_reflectancia(v[2])])
                if not (np.array_equal(pngs["rgb"][f, c], esp_rgb) and np.array_equal(pngs["espectral"][f, c], esp_esp)):
                    problemas.append(f"píxel ({f}, {c}): PNG no coincide con los rásteres")
        detalle["pixeles_comprobados"] = len(elegidos)
    return not problemas, problemas, detalle


def reconstruir_catalogo(raiz):
    """catalogo.csv desde los metadata.json de escenas completas (se excluyen _tmp y _descartadas)."""
    raiz = Path(raiz)
    filas = []
    for ruta in sorted(raiz.glob("*/metadata.json")):
        if ruta.parent.name.startswith("_"):
            continue
        m = json.loads(ruta.read_text(encoding="utf-8"))
        if m.get("estado") != "completa":
            continue
        rel = ruta.parent.name
        filas.append({
            "nombre_escena": m["nombre_escena"], "id_asset": m["id_asset"], "adquisicion_utc": m["adquisicion_utc"],
            "fecha_solicitada": m["consulta"]["fecha_solicitada"], "ventana_dias": m["consulta"]["ventana_dias"],
            "grupo_temporal": m["grupo_temporal"], "satelite": m["satelite"], "tesela": m["tesela"],
            "cobertura_pct": m["porcentajes"]["diagnostico_ee_20m"]["cobertura_pct"],
            "util_pct": m["porcentajes"]["diagnostico_ee_20m"]["util_pct"],
            "cobertura_local_pct": m["porcentajes"]["local_10m"]["cobertura_pct"],
            "util_local_pct": m["porcentajes"]["local_10m"]["util_pct"],
            "saturacion_rgb_pct": m["porcentajes"]["local_10m"]["saturacion_rgb_pct"],
            "cumple_umbrales": m["seleccion"]["cumple_umbrales"],
            "bajo_umbral_seleccion_manual": m["seleccion"]["bajo_umbral_seleccion_manual"],
            "receta_version": m["receta"]["version"], "receta_huella": m["receta"]["huella"],
            "ruta_rgb": f"{rel}/{m['rutas']['rgb']}", "ruta_espectral": f"{rel}/{m['rutas']['espectral']}",
            "ruta_metadata": f"{rel}/metadata.json"})
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla = tabla.sort_values(["adquisicion_utc", "nombre_escena"]).reset_index(drop=True)
    raiz.mkdir(parents=True, exist_ok=True)
    tmp = raiz / "catalogo.csv.tmp"
    tabla.to_csv(tmp, index=False, encoding="utf-8")
    os.replace(tmp, raiz / "catalogo.csv")
    return tabla


def construir_metadata(info, ctx, receta, nombre, conteos, seleccion):
    adq = fecha_utc(info["t"])
    return {
        "estado": "completa", "nombre_escena": nombre, "clave": clave_escena(info),
        "id_asset": info["id"], "system_index": info["indice"], "coleccion": COLECCION,
        "satelite": info.get("satelite"), "tesela": info.get("tesela"),
        "orbita_relativa": None if info.get("orbita") is None else int(info["orbita"]),
        "direccion_orbita": info.get("direccion_orbita"), "baseline": info.get("baseline"),
        "nube_granulo_pct": info.get("nube_granulo_pct"),
        "adquisicion_utc": adq.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "grupo_temporal": f"{ctx['id_aoi']}_{adq:%Y%m%d}",
        "consulta": {"fecha_solicitada": ctx["fecha"].isoformat(), "ventana_dias": ctx["ventana"],
                     "intervalo_utc_inclusivo": [ctx["inicio"].isoformat(), ctx["fin"].isoformat()],
                     "distancia_dias": int(info["dist_dias"])},
        "aoi": {"id_aoi": ctx["id_aoi"], "nombre": ctx["nombre_aoi"], "geojson_wgs84": geojson_wgs84(ctx["geom"]),
                "nota": "Encuadre de trabajo; no representa un lindero oficial."},
        "malla": ctx["malla"].como_dict(),
        "bandas": {"reflectancia.tif": BANDAS_REFL, "indices.tif": ["NDSI", "NDVI"], "calidad.tif": ["calidad"],
                   "soporte_png.tif": ["soporte"]},
        "orden_canales": {"rgb": ["R=B4", "G=B3", "B=B2"], "espectral": ["R=NDSI", "G=NDVI", "B=B4"]},
        "resolucion_nativa_m": RES_NATIVA_M, "remuestreo": receta["remuestreo"],
        "escalas": {"reflectancia": ESCALA_REFL, "rgb_max_reflectancia": RGB_MAX_REFL,
                    "indices_png": "round(255*(clip(i,-1,1)+1)/2)"},
        "nodata": receta["nodata"],
        "regla_calidad": {**receta["calidad"], "uso": "solo calidad; SCL nunca es etiqueta de entrenamiento",
                          "advertencia": "SCL puede confundir nieve, nube y sombra"},
        "porcentajes": {"diagnostico_ee_20m": {"cobertura_pct": info["cobertura_pct"], "util_pct": info["util_pct"]},
                        "local_10m": porcentajes_locales(conteos)},
        "seleccion": seleccion,
        "receta": {"version": RECETA_VERSION, "huella": receta["huella"], "detalle": receta},
        "versiones_paquetes": {**versiones_paquetes, "python": platform.python_version()},
        "rutas": {"reflectancia": "reflectancia.tif", "indices": "indices.tif", "calidad": "calidad.tif",
                  "soporte_png": "soporte_png.tif", "rgb": f"rgb/{nombre}.png",
                  "espectral": f"espectral/{nombre}.png"},
        "creado_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "atribucion": f"Contiene datos modificados de Copernicus Sentinel-2 ({adq.year}) procesados mediante "
                      f"Google Earth Engine ({COLECCION}).",
    }


def procesar_escena(info, ctx, receta, raiz, obtener_bloque, tam_bloque, seleccion, avance=print):
    """Descarga y publica una escena; reutiliza una descarga completa si coinciden receta y verificación."""
    malla, raiz = ctx["malla"], Path(raiz)
    clave = clave_escena(info)
    raiz.mkdir(parents=True, exist_ok=True)
    for d in sorted(p for p in raiz.iterdir() if p.is_dir() and re.fullmatch(rf"{re.escape(clave)}(_r\d+)?", p.name)):
        meta = leer_metadata(d)
        if meta and meta.get("id_asset") == info["id"] and meta["receta"]["huella"] == receta["huella"]:
            ok, problemas, _ = verificar_escena(d, malla)
            if ok:
                return d, "reutilizada"
            apartada = raiz / "_descartadas" / f"{d.name}_{datetime.now():%Y%m%d%H%M%S}"
            mover_con_reintentos(d, apartada)
            avance(f"  {d.name} no pasa la verificación ({problemas[0]}); se apartó en {apartada.name} y se descarga de nuevo.")
    usados, nombre, r = {p.name for p in raiz.iterdir()}, clave, 1
    while nombre in usados:
        r += 1
        nombre = f"{clave}_r{r}"
    tmp = raiz / "_tmp" / nombre
    if tmp.exists():
        avance("  Descarga interrumpida previa: se reinicia esta escena.")
        shutil.rmtree(tmp)
    (tmp / "rgb").mkdir(parents=True)
    (tmp / "espectral").mkdir()
    dentro = mascara_aoi(malla, ctx["geom_proj"])
    bloques = plan_bloques(malla, tam_bloque)
    ruta_scl = tmp / "_scl_temporal.tif"
    # 1) Descarga por producto homogéneo y bloques sobre la malla común.
    with rasterio.open(tmp / "reflectancia.tif", "w", **perfil_tif(malla, 5, "float32", NODATA)) as dst_r, \
         rasterio.open(ruta_scl, "w", **perfil_tif(malla, 1, "uint8", None)) as dst_s:
        dst_r.descriptions = tuple(BANDAS_REFL)
        for producto, dst in (("reflectancia", dst_r), ("scl", dst_s)):
            for k, (f0, c0, h, w) in enumerate(bloques, 1):
                avance(f"    {producto}: bloque {k}/{len(bloques)} · filas {f0}–{f0 + h - 1} · columnas {c0}–{c0 + w - 1}"
                       f" · {volumen_bloque(producto, h, w) / 1e6:.1f} MB")
                arr = np.asarray(obtener_bloque(info["id"], producto, malla, (f0, c0, h, w)))
                if arr.shape != (PRODUCTOS_EE[producto]["bandas"], h, w):
                    raise ValueError(f"Forma inesperada {arr.shape} en {producto} {(f0, c0, h, w)}")
                fuera = ~dentro[f0:f0 + h, c0:c0 + w]
                if producto == "reflectancia":
                    arr = arr.astype(np.float32)
                    arr[~np.isfinite(arr)] = NODATA
                    arr[:, fuera] = NODATA
                else:
                    arr = arr.astype(np.uint8)
                    arr[:, fuera] = 0
                dst.write(arr, window=Window(c0, f0, w, h))
    # 2) Productos locales desde los GeoTIFF.
    rgb = np.zeros((malla.alto, malla.ancho, 3), np.uint8)
    esp = np.zeros_like(rgb)
    total = dict.fromkeys(["dentro", "observado", "util", "soporte", "util_soporte", "sat_alguno",
                           "sat_r", "sat_g", "sat_b"], 0)
    with rasterio.open(tmp / "reflectancia.tif") as src_r, rasterio.open(ruta_scl) as src_s, \
         rasterio.open(tmp / "indices.tif", "w", **perfil_tif(malla, 2, "float32", NODATA)) as d_i, \
         rasterio.open(tmp / "calidad.tif", "w", **perfil_tif(malla, 1, "uint8", None)) as d_c, \
         rasterio.open(tmp / "soporte_png.tif", "w", **perfil_tif(malla, 1, "uint8", None)) as d_p:
        d_i.descriptions = ("NDSI", "NDVI")
        for f0, c0, h, w in bloques:
            win = Window(c0, f0, w, h)
            p = derivar_productos(src_r.read(window=win), src_s.read(1, window=win), dentro[f0:f0 + h, c0:c0 + w])
            d_i.write(p["indices"], window=win)
            d_c.write(p["calidad"], 1, window=win)
            d_p.write(p["soporte"], 1, window=win)
            rgb[f0:f0 + h, c0:c0 + w] = p["rgb"]
            esp[f0:f0 + h, c0:c0 + w] = p["espectral"]
            for k_, v_ in p["conteos"].items():
                total[k_] += v_
    ruta_scl.unlink()
    for var, arr in (("rgb", rgb), ("espectral", esp)):
        imagen = Image.fromarray(arr)
        assert imagen.mode == "RGB"
        imagen.save(tmp / var / f"{nombre}.png", format="PNG")
    # 3) Metadatos con huellas, verificación y publicación atómica.
    meta = construir_metadata(info, ctx, receta, nombre, total, seleccion)
    meta["archivos"] = {}
    for rel in ["reflectancia.tif", "indices.tif", "calidad.tif", "soporte_png.tif",
                f"rgb/{nombre}.png", f"espectral/{nombre}.png"]:
        meta["archivos"][rel] = {"sha256": sha256_archivo(tmp / rel), "bytes": (tmp / rel).stat().st_size}
    (tmp / "metadata.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    ok, problemas, _ = verificar_escena(tmp, malla)
    if not ok:
        raise RuntimeError(f"Verificación fallida antes de publicar {nombre}: {'; '.join(problemas)}")
    mover_con_reintentos(tmp, raiz / nombre)
    try:
        (raiz / "_tmp").rmdir()
    except OSError:
        pass
    reconstruir_catalogo(raiz)
    return raiz / nombre, "descargada"


def descargar_seleccion(ids, consulta, ctx, directorio, obtener_bloque, tam_bloque, avance=print):
    """Valida la selección explícita y procesa cada escena en orden cronológico."""
    ids = validar_ids(ids, consulta, ctx)
    infos = filas_por_ids(ctx, ids)
    receta = receta_productos(ctx)
    raiz = Path(directorio) / ctx["id_aoi"]
    avance(f"Destino: {raiz.resolve()} · receta {RECETA_VERSION} ({receta['huella']})")
    resultados = []
    orden = sorted(ids, key=lambda i: infos[i]["t"])
    for n, id_ in enumerate(orden, 1):
        info = infos[id_]
        cumple = bool(info["cumple"])
        seleccion = {"cumple_umbrales": cumple, "bajo_umbral_seleccion_manual": not cumple,
                     "umbrales": {"min_cobertura_pct": ctx["min_cob"], "min_util_pct": ctx["min_util"]}}
        avance(f"Escena {n}/{len(orden)} · {id_} · {fecha_utc(info['t']):%Y-%m-%d %H:%M} UTC")
        if not cumple:
            avance(f"  Atención: {estado_fila(info, ctx)}. Selección manual explícita: se anota en metadata.json.")
        t0 = time.time()
        try:
            carpeta, estado = procesar_escena(info, ctx, receta, raiz, obtener_bloque, tam_bloque, seleccion, avance)
        except Exception as exc:
            carpeta, estado = None, f"error: {sin_url(exc)}"
        avance(f"  -> {estado} ({time.time() - t0:.0f} s)")
        resultados.append({"id": id_, "adquisicion_utc": f"{fecha_utc(info['t']):%Y-%m-%d %H:%M:%S}",
                           "estado": estado, "carpeta": None if carpeta is None else str(carpeta)})
    return resultados

In [168]:
# 6b. Descarga: casillas + botón si ids_seleccionados está vacía; si no, la lista de la celda 1.
resultados_descarga = []
_descargar = functools.partial(obtener_bloque_ee, timeout=timeout_s, intentos=max_intentos)


def _mostrar_resultados(res):
    with pd.option_context("display.max_colwidth", None, "display.width", 250):
        display(pd.DataFrame(res))


if not ee_listo:
    print("Sin conexión a Earth Engine: descarga omitida.")
elif ids_seleccionados:
    resultados_descarga = descargar_seleccion(ids_seleccionados, consulta_actual, contexto_actual(),
                                              directorio_salida, _descargar, tamano_bloque)
    _mostrar_resultados(resultados_descarga)
elif consulta_actual is None or consulta_actual["n"] == 0:
    print("No hay candidatos en la consulta vigente: no se descarga nada.")
else:
    _ctx6 = contexto_actual()
    casillas = {i: widgets.Checkbox(value=False, description=etiqueta_fila(i, consulta_actual, _ctx6),
                                    indent=False, layout=widgets.Layout(width="95%"))
                for i in consulta_actual["orden_pagina"]}
    boton_descarga = widgets.Button(description="Descargar marcadas", button_style="primary", icon="download",
                                    layout=widgets.Layout(width="220px"))
    salida_descarga = widgets.Output()

    def _al_descargar(_):
        global resultados_descarga
        marcados = [numero_fila(i, consulta_actual) for i, c in casillas.items() if c.value]
        salida_descarga.clear_output()
        with salida_descarga:
            if not marcados:
                print("No hay casillas marcadas: no se descarga nada.")
                return
            boton_descarga.disabled = True
            try:
                res = descargar_seleccion(marcados, consulta_actual, contexto_actual(), directorio_salida,
                                          _descargar, tamano_bloque)
                resultados_descarga = res
                _mostrar_resultados(res)
                print(f"\nPara repetir esta selección al reejecutar, pega en la celda 1:\n"
                      f"  ids_seleccionados = {marcados}\n"
                      f"o, independiente de la tabla:\n"
                      f"  ids_seleccionados = {[r['id'].split('/')[-1] for r in res]}\n"
                      "Ejecuta la sección 7 para verificar las escenas descargadas.")
            except Exception as exc:
                print(f"Descarga no realizada: {sin_url(exc)}")
            finally:
                boton_descarga.disabled = False

    boton_descarga.on_click(_al_descargar)
    print("ids_seleccionados = []: nada se descarga automáticamente. Marca escenas y pulsa «Descargar marcadas».")
    display(widgets.VBox(list(casillas.values()) + [boton_descarga, salida_descarga]))

ids_seleccionados = []: nada se descarga automáticamente. Marca escenas y pulsa «Descargar marcadas».


## 7. Verificación

Para cada escena de esta ejecución: archivos, dimensiones y estado de la verificación (huellas, CRS, afín, tamaños y valores de PNG recalculados desde los rásteres), y las vistas RGB, espectral, NDSI continuo y calidad. El contorno amarillo es el **mismo polígono normalizado** dibujado en todas las vistas: debe caer sobre el mismo terreno.

In [169]:
# 7a. Funciones de verificación visual.
CONTORNO_PRUEBA = [(0.35, 0.30), (0.65, 0.33), (0.62, 0.68), (0.38, 0.65)]  # coordenadas normalizadas (x, y)


def mostrar_escena(carpeta, contorno=CONTORNO_PRUEBA):
    carpeta = Path(carpeta)
    meta = leer_metadata(carpeta)
    nombre = meta["nombre_escena"]
    rgb = np.asarray(Image.open(carpeta / "rgb" / f"{nombre}.png").convert("RGB"))
    esp = np.asarray(Image.open(carpeta / "espectral" / f"{nombre}.png").convert("RGB"))
    with rasterio.open(carpeta / "indices.tif") as ds:
        ndsi = ds.read(1, masked=True)
    with rasterio.open(carpeta / "calidad.tif") as ds:
        cal = ds.read(1)
    alto, ancho = cal.shape
    # Normalizado (bordes de imagen, como YOLO) -> coordenadas de imshow (centros de píxel).
    vertices = np.array([(x * ancho - 0.5, y * alto - 0.5) for x, y in contorno])
    fig, ejes = plt.subplots(1, 4, figsize=(22, 6))
    ejes[0].imshow(rgb)
    ejes[0].set_title("RGB (R=B4, G=B3, B=B2)")
    ejes[1].imshow(esp)
    ejes[1].set_title("Espectral (R=NDSI, G=NDVI, B=B4)")
    cmap = plt.get_cmap("RdBu").copy()
    cmap.set_bad("#bdbdbd")
    im = ejes[2].imshow(ndsi, cmap=cmap, vmin=-1, vmax=1, interpolation="nearest")
    fig.colorbar(im, ax=ejes[2], fraction=0.046, pad=0.02, label="NDSI")
    ejes[2].set_title("NDSI continuo (gris = NoData)")
    colores = ["#000000", "#e41a1c", "#4daf4a"]
    ejes[3].imshow(cal, cmap=ListedColormap(colores), vmin=-0.5, vmax=2.5, interpolation="nearest")
    ejes[3].legend(handles=[Patch(color=c, label=l) for c, l in zip(colores, ["0 sin observación / exterior",
                                                                              "1 excluido por SCL", "2 admitido"])],
                   loc="lower left", fontsize=8)
    ejes[3].set_title("Calidad (SCL)")
    for ax in ejes:
        ax.add_patch(PoligonoMpl(vertices, closed=True, fill=False, edgecolor="yellow", linewidth=1.5))
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(f"{nombre} · {meta['adquisicion_utc']} · {meta['id_asset']}", fontsize=10)
    plt.tight_layout()
    plt.show()

In [170]:
# 7b. Verificación de las escenas de esta ejecución.
escenas = [r for r in resultados_descarga if r["carpeta"]]
if not escenas:
    print("No hay escenas descargadas o reutilizadas en esta ejecución (marca casillas en la sección 6 o "
          "rellena ids_seleccionados, y vuelve a ejecutar esta celda).")
    _cat = Path(directorio_salida) / contexto_actual()["id_aoi"] / "catalogo.csv"
    if _cat.exists():
        print(f"Catálogo existente: {_cat}")
        display(pd.read_csv(_cat))
else:
    malla_v = contexto_actual()["malla"]
    registro = []
    for r in escenas:
        ok, problemas, detalle = verificar_escena(r["carpeta"], malla_v)
        meta = leer_metadata(r["carpeta"])
        registro.append({"id": r["id"], "adquisicion_utc": meta["adquisicion_utc"], "estado": r["estado"],
                         "verificacion": "correcta" if ok else "; ".join(problemas),
                         "util_local_pct": meta["porcentajes"]["local_10m"]["util_pct"],
                         "saturacion_rgb_pct": meta["porcentajes"]["local_10m"]["saturacion_rgb_pct"],
                         **detalle})
        mostrar_escena(r["carpeta"])
    with pd.option_context("display.max_colwidth", None, "display.width", 250):
        display(pd.DataFrame(registro))
    mallas = {json.dumps(leer_metadata(r["carpeta"])["malla"], sort_keys=True) for r in escenas}
    ids_unicos = {leer_metadata(r["carpeta"])["id_asset"] for r in escenas}
    print(f"Malla común entre escenas: {'sí' if len(mallas) == 1 else 'NO'} · IDs de origen distintos: "
          f"{len(ids_unicos)} de {len(escenas)}")

No hay escenas descargadas o reutilizadas en esta ejecución (marca casillas en la sección 6 o rellena ids_seleccionados, y vuelve a ejecutar esta celda).


## 8. Pruebas de aceptación (SDD §7)

**8a. Pruebas locales**: no requieren Earth Engine. Usan datos sintéticos deterministas en una carpeta temporal y ejercitan las mismas funciones del notebook (AOI, fechas, calidad, fórmulas, malla, bloques, reanudación, catálogo).

**8b. Pruebas con Earth Engine**: requieren una sesión autenticada. Están desactivadas por defecto porque consultan y descargan (a una carpeta temporal que se borra al final); activa `ejecutar_pruebas_ee = True` para correrlas.

In [171]:
# 8a. Pruebas locales (sin Earth Engine).
resultados_pruebas = []


def prueba(nombre):
    """Ejecuta la función decorada y registra el resultado."""
    def decorador(funcion):
        t0 = time.time()
        try:
            funcion()
            resultados_pruebas.append({"prueba": nombre, "resultado": "aprobada", "detalle": f"{time.time() - t0:.1f} s"})
        except Exception as exc:
            resultados_pruebas.append({"prueba": nombre, "resultado": "FALLÓ", "detalle": f"{type(exc).__name__}: {exc}"})
        return funcion
    return decorador


SI_CAJA = [-75.397, 4.773, -75.348, 4.827]   # mismo encuadre que el aoi_entrada por defecto
ctx_si = preparar_contexto(SI_CAJA, "santa_isabel", None, "2024-01-15", 15, 95, 70)


def info_sintetica(indice, dia, cob=97.0, util=80.0):
    t = int(datetime(2024, 1, dia, 15, 30, 12, tzinfo=timezone.utc).timestamp() * 1000)
    return {"id": f"{COLECCION}/{indice}", "indice": indice, "t": t, "tesela": indice.split("_T")[-1],
            "satelite": "Sentinel-2A", "orbita": 25, "direccion_orbita": "DESCENDING", "baseline": "05.10",
            "nube_granulo_pct": 12.0, "cobertura_pct": cob, "util_pct": util, "dist_dias": abs(dia - 15),
            "cumple": int(cob >= 95 and util >= 70)}


def reflectancia_sintetica(filas, cols, semilla=0):
    """Campo determinista de la posición global: permite comprobar la continuidad entre bloques."""
    base = 0.05 + 0.5 * (((filas * 7 + cols * 13 + semilla) % 101) / 100.0)
    r = np.stack([base * k for k in (0.8, 1.0, 0.9, 1.1, 0.6)]).astype(np.float32)
    r[:, cols < 40] = NODATA                         # franja oeste sin datos
    r[4, (filas >= 100) & (filas < 110)] = NODATA    # falta solo B11
    cero = (filas >= 200) & (filas < 205)
    r[1, cero] = 0.0                                 # B3 + B11 = 0 -> NDSI indefinido
    r[4, cero] = 0.0
    return r


def scl_sintetica(filas, cols):
    s = np.full(filas.shape, 4, np.uint8)
    s[filas < 300] = 11
    s[(filas >= 300) & (filas < 400)] = 9
    s[cols < 40] = 0
    return s[None]


def descargador_sintetico(id_asset, producto, malla, ventana, registro=None, fallar_en=None):
    if registro is not None:
        registro.append((producto, ventana))
        if fallar_en is not None and len(registro) >= fallar_en:
            raise ConnectionError("interrupción simulada")
    f0, c0, h, w = ventana
    filas, cols = np.mgrid[f0:f0 + h, c0:c0 + w]
    semilla = int(hashlib.md5(id_asset.encode()).hexdigest(), 16) % 50
    return reflectancia_sintetica(filas, cols, semilla) if producto == "reflectancia" else scl_sintetica(filas, cols)


SELECCION_OK = {"cumple_umbrales": True, "bajo_umbral_seleccion_manual": False, "umbrales": {}}
tmp_pruebas = Path(tempfile.mkdtemp(prefix="s2_pruebas_"))


@prueba("Santa Isabel por defecto: AOI con referencias IDEAM, EPSG:32618 y malla en múltiplos de 10 m")
def _():
    assert ctx_si["crs"] == "EPSG:32618", ctx_si["crs"]
    for lon, lat in [(-75.3500, 4.8167), (-75.3667, 4.8000)]:
        assert ctx_si["geom"].contains(Point(lon, lat))
    m = ctx_si["malla"]
    assert m.transform[2] % 10 == 0 and m.transform[5] % 10 == 0
    assert box(*m.limites).contains(ctx_si["geom_proj"])
    assert 400 < m.ancho < 1100 and 400 < m.alto < 1100, (m.ancho, m.alto)
    assert ctx_si["id_aoi"] == preparar_contexto(SI_CAJA, "santa_isabel", None, "2023-05-01", 3, 95, 70)["id_aoi"]


@prueba("Rechazo de AOI vacío, inválido, invertido, fuera de rango o que cruza zonas UTM; CRS explícito métrico")
def _():
    malos = [[-75.32, 4.76, -75.41, 4.85], [-75.41, 4.85, -75.32, 4.76], [1, 2, 3],
             {"type": "Polygon", "coordinates": []},
             {"type": "Polygon", "coordinates": [[[0, 0], [1, 1], [1, 0], [0, 1], [0, 0]]]},
             {"type": "Point", "coordinates": [-75.3, 4.8]}, [-75.41, 4.76, -75.32, 95.0]]
    for aoi in malos:
        try:
            elegir_crs(normalizar_aoi(aoi))
        except (ValueError, TypeError):
            continue
        raise AssertionError(f"Se aceptó un AOI inválido: {aoi}")
    for aoi in ([-72.5, 4.0, -71.5, 4.5], [10, 85, 11, 86]):
        try:
            elegir_crs(normalizar_aoi(aoi))
            raise AssertionError(f"Debía exigir CRS explícito: {aoi}")
        except ValueError:
            pass
    assert elegir_crs(normalizar_aoi([-72.5, 4.0, -71.5, 4.5]), "EPSG:3116") == "EPSG:3116"
    try:
        elegir_crs(normalizar_aoi(SI_CAJA), "EPSG:4326")
        raise AssertionError("Se aceptó un CRS no métrico")
    except ValueError:
        pass


@prueba("Fechas: ventana 0 = solo ese día; ventana 1 incluye ambos vecinos; fin exclusivo +1 día")
def _():
    f, ini, fin, excl = intervalo_fechas("2024-01-15", 0)
    assert ini == fin == f == date(2024, 1, 15) and excl == date(2024, 1, 16)
    f, ini, fin, excl = intervalo_fechas("2024-01-15", 1)
    assert (ini, fin, excl) == (date(2024, 1, 14), date(2024, 1, 16), date(2024, 1, 17))
    for fecha, ventana in (("2024-02-30", 0), ("2024-01-15", -1), ("2024-01-15", 1.5)):
        try:
            intervalo_fechas(fecha, ventana)
            raise AssertionError(f"Se aceptó {fecha}, {ventana}")
        except ValueError:
            pass
    c0 = preparar_contexto(SI_CAJA, "santa_isabel", None, "2024-01-15", 0, 95, 70)
    c1 = preparar_contexto(SI_CAJA, "santa_isabel", None, "2024-01-16", 0, 95, 70)
    assert c0["firma"] != c1["firma"] and c0["id_aoi"] == c1["id_aoi"]


@prueba("Otro AOI: GeoJSON no rectangular con CRS apropiado y máscara exterior en todos los productos")
def _():
    triangulo = {"type": "Polygon", "coordinates": [[[-70.85, -13.95], [-70.78, -13.94], [-70.82, -13.88], [-70.85, -13.95]]]}
    assert preparar_contexto(triangulo, "quelccaya", None, "2024-01-15", 0, 95, 70)["crs"] == "EPSG:32719"
    tri_si = {"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [[[-75.40, 4.77], [-75.33, 4.78],
                                                                                    [-75.36, 4.84], [-75.40, 4.77]]]}}
    c = preparar_contexto(tri_si, "triangulo", None, "2024-01-15", 0, 95, 70)
    assert c["crs"] == "EPSG:32618"
    carpeta, estado = procesar_escena(info_sintetica("20240113T152649_20240113T152650_T18NVL", 13), c,
                                      receta_productos(c), tmp_pruebas / "otro", descargador_sintetico, 256,
                                      SELECCION_OK, avance=lambda *_: None)
    dentro = mascara_aoi(c["malla"], c["geom_proj"])
    assert 0.2 < dentro.mean() < 0.8 and not dentro[0, 0] and not dentro[0, -1]
    with rasterio.open(carpeta / "calidad.tif") as a, rasterio.open(carpeta / "reflectancia.tif") as b, \
         rasterio.open(carpeta / "indices.tif") as i_, rasterio.open(carpeta / "soporte_png.tif") as s:
        assert (a.read(1)[~dentro] == 0).all() and (b.read()[:, ~dentro] == NODATA).all()
        assert (i_.read()[:, ~dentro] == NODATA).all() and (s.read(1)[~dentro] == 0).all()
    meta = leer_metadata(carpeta)
    rgb = np.asarray(Image.open(carpeta / meta["rutas"]["rgb"]))
    assert (rgb[~dentro] == 0).all() and rgb[dentro].any()


@prueba("Calidad controlada: SCL 11 útil, SCL 9 observable excluida, sin datos = 0; 0 ≤ útil ≤ cobertura ≤ 100")
def _():
    refl = np.full((5, 1, 6), 0.3, np.float32)
    refl[:, 0, 2] = NODATA                      # sin datos
    scl = np.array([[11, 9, 0, 4, 11, 3]], np.uint8)
    dentro = np.array([[True, True, True, True, True, False]])
    p = derivar_productos(refl, scl, dentro)
    assert p["calidad"].tolist() == [[2, 1, 0, 2, 2, 0]], p["calidad"]
    pc = porcentajes_locales(p["conteos"])
    assert pc["cobertura_pct"] == 80.0 and pc["util_pct"] == 60.0
    assert 0 <= pc["util_pct"] <= pc["cobertura_pct"] <= 100
    mitad = np.full((5, 4, 4), 0.3, np.float32)
    mitad[:, :, :2] = NODATA                    # escena que cubre medio AOI
    pm = porcentajes_locales(derivar_productos(mitad, np.full((4, 4), 4, np.uint8), np.ones((4, 4), bool))["conteos"])
    assert pm["cobertura_pct"] == 50.0 < 95


@prueba("Fórmulas y orden de canales: NDSI = NDVI = 0,50; PNG espectral (191, 191, 42–43); denominador cero")
def _():
    refl = np.array([0.05, 0.60, 0.10, 0.30, 0.20], np.float32).reshape(5, 1, 1)
    p = derivar_productos(refl, np.array([[11]], np.uint8), np.ones((1, 1), bool))
    assert abs(p["indices"][0, 0, 0] - 0.5) < 1e-6 and abs(p["indices"][1, 0, 0] - 0.5) < 1e-6
    r, g, b = p["espectral"][0, 0]
    assert (r, g) == (191, 191) and b in (42, 43), p["espectral"][0, 0]
    assert p["rgb"][0, 0].tolist() == [int(a_uint8_reflectancia(refl[k, 0, 0])) for k in (2, 1, 0)]
    assert p["rgb"][0, 0, 1] == 255
    cero = np.array([0.05, 0.0, 0.10, 0.30, 0.0], np.float32).reshape(5, 1, 1)
    q = derivar_productos(cero, np.array([[4]], np.uint8), np.ones((1, 1), bool))
    assert q["indices"][0, 0, 0] == NODATA and q["indices"][1, 0, 0] != NODATA
    assert q["soporte"][0, 0] == 0 and not q["rgb"].any() and not q["espectral"].any()
    assert q["calidad"][0, 0] == 2  # la calidad SCL se consulta por separado del soporte


@prueba("Plan de bloques: cubre la malla sin solapes y respeta 32 MB / 10 000 px")
def _():
    m = ctx_si["malla"]
    cubierto = np.zeros((m.alto, m.ancho), np.int32)
    bloques = plan_bloques(m, 512)
    for f0, c0, h, w in bloques:
        cubierto[f0:f0 + h, c0:c0 + w] += 1
        assert volumen_bloque("reflectancia", h, w) <= LIMITE_BYTES_PETICION
    assert (cubierto == 1).all() and len(bloques) == 4
    try:
        obtener_bloque_ee("x", "reflectancia", m, (0, 0, 2000, 2000))
        raise AssertionError("Debía rechazar un bloque > 32 MB")
    except ValueError:
        pass


escena_a = info_sintetica("20240113T152649_20240113T152650_T18NVL", 13)
escena_b = info_sintetica("20240118T152641_20240118T152922_T18NVL", 18, cob=60.0, util=20.0)
raiz_si = tmp_pruebas / "serie"
receta_si = receta_productos(ctx_si)
procesar = functools.partial(procesar_escena, ctx=ctx_si, receta=receta_si, raiz=raiz_si, tam_bloque=256,
                             seleccion=SELECCION_OK, avance=lambda *_: None)


def round_mitad(x):
    return int(math.floor(x + 0.5))


@prueba("Coincidencia espacial: CRS, afín y tamaños iguales; PNG = valores esperados en ≥ 3 píxeles")
def _():
    carpeta, estado = procesar(escena_a, obtener_bloque=descargador_sintetico)
    assert estado == "descargada"
    ok, problemas, detalle = verificar_escena(carpeta, ctx_si["malla"])
    assert ok and detalle["pixeles_comprobados"] >= 3, problemas
    perfiles = []
    for nombre in ARCHIVOS_TIF:
        with rasterio.open(carpeta / nombre) as ds:
            perfiles.append((ds.crs.to_string(), tuple(ds.transform), ds.width, ds.height))
    assert len(set(perfiles)) == 1
    meta = leer_metadata(carpeta)
    rgb = np.asarray(Image.open(carpeta / meta["rutas"]["rgb"]).convert("RGB"))
    esp = np.asarray(Image.open(carpeta / meta["rutas"]["espectral"]).convert("RGB"))
    assert rgb.shape == esp.shape == (ctx_si["malla"].alto, ctx_si["malla"].ancho, 3)
    with rasterio.open(carpeta / "reflectancia.tif") as dr, rasterio.open(carpeta / "indices.tif") as di:
        refl, ind = dr.read(), di.read()
    for f, c in [(150, 150), (500, 400), (580, 520), (350, 420)]:
        b2, b3, b4, b8, b11 = refl[:, f, c].astype(np.float64)
        assert rgb[f, c].tolist() == [round_mitad(min(max(v / 0.6, 0), 1) * 255) for v in (b4, b3, b2)]
        ndsi = (b3 - b11) / (b3 + b11)
        assert abs(ind[0, f, c] - ndsi) < 1e-6
        assert esp[f, c].tolist() == [round_mitad((min(max(float(ind[0, f, c]), -1), 1) + 1) / 2 * 255),
                                      round_mitad((min(max(float(ind[1, f, c]), -1), 1) + 1) / 2 * 255),
                                      round_mitad(255 * min(max(b4 / 0.6, 0), 1))]
    assert not rgb[:, :40].any() and not esp[:, :40].any()  # soporte común: franja sin datos


@prueba("Bloques: escena de varios bloques sin costuras (igual al campo continuo) y reejecución reutilizada")
def _():
    carpeta = raiz_si / clave_escena(escena_a)
    m = ctx_si["malla"]
    filas, cols = np.mgrid[0:m.alto, 0:m.ancho]
    esperado = reflectancia_sintetica(filas, cols, int(hashlib.md5(escena_a["id"].encode()).hexdigest(), 16) % 50)
    esperado[:, ~mascara_aoi(m, ctx_si["geom_proj"])] = NODATA
    with rasterio.open(carpeta / "reflectancia.tif") as ds:
        assert np.array_equal(ds.read(), esperado)
    huellas = {p: p.stat().st_mtime_ns for p in carpeta.rglob("*") if p.is_file()}
    registro = []
    carpeta2, estado = procesar(escena_a, obtener_bloque=functools.partial(descargador_sintetico, registro=registro))
    assert estado == "reutilizada" and carpeta2 == carpeta and not registro
    assert huellas == {p: p.stat().st_mtime_ns for p in carpeta.rglob("*") if p.is_file()}


@prueba("Interrupción: la escena incompleta no entra al catálogo, las completas no cambian y se reanuda")
def _():
    carpeta_a = raiz_si / clave_escena(escena_a)
    antes = {p.name: sha256_archivo(p) for p in carpeta_a.rglob("*") if p.is_file()}
    try:
        procesar(escena_b, obtener_bloque=functools.partial(descargador_sintetico, registro=[], fallar_en=10))
        raise AssertionError("La interrupción simulada no ocurrió")
    except ConnectionError:
        pass
    cat = reconstruir_catalogo(raiz_si)
    assert list(cat["id_asset"]) == [escena_a["id"]]
    assert (raiz_si / "_tmp" / clave_escena(escena_b)).exists() and not (raiz_si / clave_escena(escena_b)).exists()
    assert antes == {p.name: sha256_archivo(p) for p in carpeta_a.rglob("*") if p.is_file()}
    carpeta_b, estado = procesar(escena_b, obtener_bloque=descargador_sintetico,
                                 seleccion={**SELECCION_OK, "cumple_umbrales": False, "bajo_umbral_seleccion_manual": True})
    assert estado == "descargada" and not (raiz_si / "_tmp").exists()
    cat = pd.read_csv(raiz_si / "catalogo.csv")
    assert sorted(cat["id_asset"]) == sorted([escena_a["id"], escena_b["id"]]) and cat["nombre_escena"].is_unique
    assert bool(cat.set_index("id_asset").loc[escena_b["id"], "bajo_umbral_seleccion_manual"])


@prueba("Serie multitemporal: misma malla, IDs distintos, fechas reales y grupo_temporal por día UTC")
def _():
    ma, mb = (leer_metadata(raiz_si / clave_escena(e)) for e in (escena_a, escena_b))
    assert ma["malla"] == mb["malla"] and ma["id_asset"] != mb["id_asset"]
    assert ma["adquisicion_utc"] == "2024-01-13T15:30:12Z" and mb["adquisicion_utc"] == "2024-01-18T15:30:12Z"
    assert ma["grupo_temporal"] == f"{ctx_si['id_aoi']}_20240113"
    assert ma["consulta"]["fecha_solicitada"] == "2024-01-15" and "url" not in json.dumps(ma).lower()
    assert Path(ma["rutas"]["rgb"]).name == Path(ma["rutas"]["espectral"]).name


@prueba("Receta distinta: se escribe otra revisión sin tocar la anterior")
def _():
    receta2 = {**receta_si, "version": "1.1-prueba"}
    receta2["huella"] = huella({k: v for k, v in receta2.items() if k != "huella"}, 12)
    carpeta, estado = procesar_escena(escena_a, ctx_si, receta2, raiz_si, descargador_sintetico, 256,
                                      SELECCION_OK, avance=lambda *_: None)
    assert estado == "descargada" and carpeta.name == clave_escena(escena_a) + "_r2"
    assert (raiz_si / clave_escena(escena_a) / "metadata.json").exists()
    assert len(pd.read_csv(raiz_si / "catalogo.csv")) == 3


@prueba("Selección: filas e IDs de la tabla vigente; ajenos, obsoletos o fuera de página se rechazan; vacía = sin descargas")
def _():
    consulta = {"firma": ctx_si["firma"], "ids": [escena_a["id"], escena_b["id"]],
                "orden_pagina": [escena_b["id"], escena_a["id"]], "primera_fila": 26}
    assert validar_ids([escena_a["indice"]], consulta, ctx_si) == [escena_a["id"]]
    assert validar_ids([27, 26, escena_b["id"]], consulta, ctx_si) == [escena_a["id"], escena_b["id"]]
    assert numero_fila(escena_a["id"], consulta) == 27
    assert validar_ids([], consulta, ctx_si) == []
    for ids, ctx_ in (([f"{COLECCION}/ajeno"], ctx_si), ([1], ctx_si), ([28], ctx_si), ([True], ctx_si),
                      ([escena_a["id"]], preparar_contexto(SI_CAJA, "santa_isabel", None, "2024-03-01", 15, 95, 70)),
                      ([26], preparar_contexto(SI_CAJA, "santa_isabel", None, "2024-03-01", 15, 95, 70))):
        try:
            validar_ids(ids, consulta, ctx_)
            raise AssertionError(f"Se aceptó una selección ajena u obsoleta: {ids}")
        except (ValueError, RuntimeError):
            pass


@prueba("Diagnóstico de errores de Earth Engine y enlaces temporales ocultos")
def _():
    casos = {"Project 'projects/xyz' not found or deleted.": "proyecto",
             "Caller does not have required permission to use project xyz": "permisos",
             "Please authorize access to your Earth Engine account by running earthengine authenticate": "autorización",
             "HTTPSConnectionPool(host='earthengine.googleapis.com'): Max retries exceeded": "conexión"}
    for texto, tipo in casos.items():
        assert diagnosticar_error_ee(ee.EEException(texto))[0] == tipo, texto
    assert "token" not in sin_url("fallo en https://earthengine.googleapis.com/v1/x/thumbnails/abc:getPixels?token=1")


shutil.rmtree(tmp_pruebas, ignore_errors=True)
tabla_pruebas = pd.DataFrame(resultados_pruebas)
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(tabla_pruebas)
n_ok = (tabla_pruebas["resultado"] == "aprobada").sum()
print(f"Pruebas locales: {n_ok}/{len(tabla_pruebas)} aprobadas")
assert n_ok == len(tabla_pruebas), "Hay pruebas locales fallidas"

Nota: '20240113T152649_20240113T152650_T18NVL' se interpreta como 'COPERNICUS/S2_SR_HARMONIZED/20240113T152649_20240113T152650_T18NVL'.
Fila 27 -> COPERNICUS/S2_SR_HARMONIZED/20240113T152649_20240113T152650_T18NVL
Fila 26 -> COPERNICUS/S2_SR_HARMONIZED/20240118T152641_20240118T152922_T18NVL


,prueba,resultado,detalle
0,"Santa Isabel por defecto: AOI con referencias IDEAM, EPSG:32618 y malla en múltiplos de 10 m",aprobada,0.0 s
1,"Rechazo de AOI vacío, inválido, invertido, fuera de rango o que cruza zonas UTM; CRS explícito métrico",aprobada,0.0 s
2,Fechas: ventana 0 = solo ese día; ventana 1 incluye ambos vecinos; fin exclusivo +1 día,aprobada,0.0 s
3,Otro AOI: GeoJSON no rectangular con CRS apropiado y máscara exterior en todos los productos,aprobada,0.5 s
4,"Calidad controlada: SCL 11 útil, SCL 9 observable excluida, sin datos = 0; 0 ≤ útil ≤ cobertura ≤ 100",aprobada,0.0 s
5,"Fórmulas y orden de canales: NDSI = NDVI = 0,50; PNG espectral (191, 191, 42–43); denominador cero",aprobada,0.0 s
6,Plan de bloques: cubre la malla sin solapes y respeta 32 MB / 10 000 px,aprobada,0.0 s
7,"Coincidencia espacial: CRS, afín y tamaños iguales; PNG = valores esperados en ≥ 3 píxeles",aprobada,0.3 s
8,Bloques: escena de varios bloques sin costuras (igual al campo continuo) y reejecución reutilizada,aprobada,0.1 s
9,"Interrupción: la escena incompleta no entra al catálogo, las completas no cambian y se reanuda",aprobada,0.4 s


Pruebas locales: 14/14 aprobadas


In [172]:
# 8b. Pruebas con Earth Engine (requieren sesión autenticada; consultan y descargan a una carpeta temporal).
ejecutar_pruebas_ee = False

resultados_ee = []


def prueba_ee(nombre):
    def decorador(funcion):
        t0 = time.time()
        try:
            detalle = funcion()
            resultados_ee.append({"prueba": nombre, "resultado": "aprobada",
                                  "detalle": f"{detalle or ''} ({time.time() - t0:.0f} s)"})
        except Exception as exc:
            resultados_ee.append({"prueba": nombre, "resultado": "FALLÓ", "detalle": sin_url(f"{type(exc).__name__}: {exc}")})
        return funcion
    return decorador


if not ejecutar_pruebas_ee:
    print("Pruebas con Earth Engine desactivadas (ejecutar_pruebas_ee = False).")
elif not ee_listo:
    print("Pruebas con Earth Engine omitidas: no hay sesión autenticada.")
else:
    tmp_ee = Path(tempfile.mkdtemp(prefix="s2_pruebas_ee_"))
    silencio = lambda *_: None
    bloque_ee = functools.partial(obtener_bloque_ee, timeout=timeout_s, intentos=max_intentos)
    ctx_base = preparar_contexto(SI_CAJA, "santa_isabel", None, "2024-01-15", 15, 95, 70)
    base, _ = buscar(ctx_base, 100, 1)
    dias_con_datos = sorted({fecha_utc(p["t"]).date() for p in base["filas"].values()})

    @prueba_ee("Calidad controlada en EE: SCL 11 útil, SCL 9 excluida, sin datos fuera del numerador")
    def _():
        oeste, sur, este, norte = SI_CAJA
        xm, ym = (oeste + este) / 2, (sur + norte) / 2
        lon = ee.Image.pixelLonLat()
        scl = ee.Image(11).where(lon.select("longitude").gte(xm), 9)
        sin_datos = lon.select("longitude").gte(xm).And(lon.select("latitude").lt(ym))
        img = ee.Image.constant([300, 300, 300, 300, 300]).rename(BANDAS_REFL).addBands(scl.rename("SCL")) \
            .updateMask(sin_datos.Not()).set({"system:time_start": ee.Date("2024-01-15").millis(), "system:index": "x"})
        p = funcion_anotar(ctx_base, area_total_aoi(ctx_base), ee.List(["x"]))(img).getInfo()["properties"]
        assert abs(p["cobertura_pct"] - 75) < 1.5 and abs(p["util_pct"] - 50) < 1.5, p
        assert 0 <= p["util_pct"] <= p["cobertura_pct"] <= 100 + 1e-9
        return f"cobertura {p['cobertura_pct']:.2f} %, útil {p['util_pct']:.2f} %"

    @prueba_ee("Fecha exacta (ventana 0), vecinos (ventana 1) y consulta vacía sin descarga")
    def _():
        d = dias_con_datos[0]
        c0 = preparar_contexto(SI_CAJA, "santa_isabel", None, d.isoformat(), 0, 95, 70)
        q0, t0 = buscar(c0, 50, 1)
        assert q0["n"] >= 1 and all(fecha_utc(p["t"]).date() == d for p in q0["filas"].values())
        for vecino in (d - timedelta(days=1), d + timedelta(days=1)):
            q1, _ = buscar(preparar_contexto(SI_CAJA, "santa_isabel", None, vecino.isoformat(), 1, 95, 70), 50, 1)
            assert any(fecha_utc(p["t"]).date() == d for p in q1["filas"].values()), vecino
        vacio = next(d + timedelta(days=k) for k in range(1, 10) if d + timedelta(days=k) not in dias_con_datos)
        cv = preparar_contexto(SI_CAJA, "santa_isabel", None, vacio.isoformat(), 0, 95, 70)
        qv, tv = buscar(cv, 50, 1)
        assert qv["n"] == 0 and tv.empty
        assert descargar_seleccion([], qv, cv, tmp_ee, bloque_ee, 512, silencio) == []
        try:
            descargar_seleccion(q0["ids"][:1], qv, cv, tmp_ee, bloque_ee, 512, silencio)
            raise AssertionError("Se aceptó un ID de otra consulta")
        except ValueError:
            pass
        return f"{d}: {q0['n']} gránulo(s); {vacio}: vacío"

    @prueba_ee("Otro AOI (GeoJSON no rectangular, Quelccaya): EPSG:32719, teselas 19L y máscara exterior")
    def _():
        tri = {"type": "Polygon", "coordinates": [[[-70.85, -13.95], [-70.80, -13.94], [-70.83, -13.90], [-70.85, -13.95]]]}
        c = preparar_contexto(tri, "quelccaya_prueba", None, "2024-07-15", 15, 0, 0)
        assert c["crs"] == "EPSG:32719"
        q, t = buscar(c, 50, 1)
        assert q["n"] > 0 and all(p["tesela"].startswith("19L") for p in q["filas"].values())
        res = descargar_seleccion(q["orden_pagina"][:1], q, c, tmp_ee, bloque_ee, 512, silencio)
        assert res[0]["estado"] == "descargada", res
        dentro = mascara_aoi(c["malla"], c["geom_proj"])
        with rasterio.open(Path(res[0]["carpeta"]) / "calidad.tif") as a, \
             rasterio.open(Path(res[0]["carpeta"]) / "reflectancia.tif") as b:
            assert (a.read(1)[~dentro] == 0).all() and (b.read()[:, ~dentro] == NODATA).all()
            assert (b.read()[:, dentro] != NODATA).any()
        return f"{q['n']} gránulos; {res[0]['id']}"

    @prueba_ee("Serie Santa Isabel: dos fechas, bloques sin costuras, reejecución e interrupción")
    def _():
        ids = sorted(base["ids"])[:3]
        res = descargar_seleccion(ids[:2], base, ctx_base, tmp_ee, bloque_ee, 512, silencio)
        assert all(r["estado"] == "descargada" for r in res), res
        metas = [leer_metadata(r["carpeta"]) for r in res]
        assert metas[0]["malla"] == metas[1]["malla"] and metas[0]["id_asset"] != metas[1]["id_asset"]
        for r, m_ in zip(res, metas):
            ok, problemas, det = verificar_escena(r["carpeta"], ctx_base["malla"])
            assert ok, problemas
            assert m_["adquisicion_utc"].startswith(r["adquisicion_utc"][:10])
        # Continuidad: ventana de 16×16 sobre la esquina de cuatro bloques, pedida en una sola petición.
        directo = bloque_ee(ids[0], "reflectancia", ctx_base["malla"], (504, 504, 16, 16))
        carpeta0 = next(r["carpeta"] for r in res if r["id"] == ids[0])
        with rasterio.open(Path(carpeta0) / "reflectancia.tif") as ds:
            cosido = ds.read(window=Window(504, 504, 16, 16))
        assert np.allclose(directo, cosido, atol=1e-7), np.abs(directo - cosido).max()
        assert all(r["estado"] == "reutilizada" for r in descargar_seleccion(ids[:2], base, ctx_base, tmp_ee, bloque_ee, 512, silencio))
        llamadas = []
        def interrumpido(*a, **k):
            llamadas.append(1)
            if len(llamadas) > 3:
                raise ConnectionError("interrupción simulada")
            return bloque_ee(*a, **k)
        r3 = descargar_seleccion(ids[2:3], base, ctx_base, tmp_ee, interrumpido, 512, silencio)
        assert r3[0]["estado"].startswith("error")
        raiz = tmp_ee / ctx_base["id_aoi"]
        assert ids[2] not in set(pd.read_csv(raiz / "catalogo.csv")["id_asset"])
        r3 = descargar_seleccion(ids[2:3], base, ctx_base, tmp_ee, bloque_ee, 512, silencio)
        cat = pd.read_csv(raiz / "catalogo.csv")
        assert r3[0]["estado"] == "descargada" and len(cat) == 3 and cat["id_asset"].is_unique
        return "; ".join(f"{r['id'].split('/')[-1]} {r['adquisicion_utc']}" for r in res + r3)

    shutil.rmtree(tmp_ee, ignore_errors=True)
    with pd.option_context("display.max_colwidth", None, "display.width", 250):
        display(pd.DataFrame(resultados_ee))
    print(f"Pruebas con Earth Engine: {sum(r['resultado'] == 'aprobada' for r in resultados_ee)}/{len(resultados_ee)} aprobadas")

Pruebas con Earth Engine desactivadas (ejecutar_pruebas_ee = False).


## 9. Anotar escena

Editor de etiquetas **YOLO Segmentation** de una sola clase (`0 · nieve_hielo_visible`) sobre las escenas descargadas. Solo lee los archivos locales de cada escena: funciona sin conexión y sin Earth Engine.

- **Tras descargar en esta sesión**: ejecuta 9d. Si descargaste una sola escena se abre directamente; con varias aparece un selector con fecha, clave y estado de anotación. Si descargaste con el botón de la sección 6, vuelve a ejecutar 9d.
- **Reabrir una descarga antigua** (kernel recién iniciado): escribe su carpeta en `ruta_escena` (celda 9a) y ejecuta solo **9a → 9b → 9c → 9d**. `ruta_escena` acepta la carpeta de una escena, la raíz de un AOI (`datos_sentinel2/<id_aoi>`) o `datos_sentinel2`. Si la dejas vacía se listan todas las escenas de `datos_sentinel2`.
- 9e ejecuta las pruebas de aceptación con datos sintéticos en un directorio temporal; no toca tus anotaciones.

**Polígonos NDSI automáticos.** Al abrir una escena sin borrador ni revisión, el editor dibuja como polígonos (origen «asistido», violeta; los manuales son verdes) los parches con NDSI ≥ umbral en soporte válido de toda la imagen. Tu trabajo es revisarlos: corrige los vértices, elimina los que sobren y dibuja los que falten. Al cambiar el umbral se redibujan los automáticos que no editaste; los editados, los manuales y los eliminados se respetan. «Deshacer» revierte el último dibujo automático. Un parche con huecos interiores queda como candidato pendiente (cian), porque YOLO usa un anillo simple por instancia. «Dibujar con NDSI» repite el proceso en una escena que ya tenía trabajo. Para anotar sin ayuda (fechas de prueba), pon `poligonos_ndsi_al_abrir = False` en 9a.

**Controles**

| Control | Uso |
|---|---|
| Vista | RGB, Espectral (NDSI, NDVI, B4) o NDSI con escala fija −1…1. Al cambiar de vista se conservan el zoom, el trazo y los contornos. |
| Calidad dudosa / Sin soporte | Superposiciones naranja (calidad 1, observación dudosa) y magenta (soporte 0, sin datos compartidos). |
| **Navegar** | La rueda del ratón amplía alrededor del cursor. La lupa y la cruz de la barra de herramientas hacen zoom y desplazamiento. Los clics no editan. |
| **Dibujar parche** | Cada clic añade un vértice. Para un vértice en el borde de la imagen, haz clic justo fuera de ella: se ajusta al borde. Clic derecho, doble clic o «Cerrar polígono» cierra el parche; «Cancelar trazo» lo borra. |
| **Editar** | Un clic sobre un contorno lo selecciona (amarillo). Arrastra un vértice para moverlo; un clic sobre un lado añade un vértice. «Eliminar vértice» quita el vértice activo (rojo) y «Eliminar polígono» quita el contorno seleccionado. |
| **Región NDSI** | Dibuja una región como si fuera un parche. Al cerrarla se proponen candidatos con NDSI ≥ umbral (cian discontinuo; ⚠ = toca el borde de la región y puede estar cortado). Selecciona un candidato en modo Editar y pulsa «Aceptar» o «Descartar». Al cambiar el umbral se regeneran solo los candidatos pendientes. |
| Umbral NDSI · Dibujar con NDSI | El umbral redibuja los polígonos automáticos no editados y los candidatos pendientes. «Dibujar con NDSI» dibuja los parches de toda la imagen con el umbral actual (útil en escenas con borrador o revisión previa). |
| Deshacer · Ver todo | Revierte la última edición · restablece el encuadre completo. |
| Guardar borrador | Escribe `anotaciones/borrador.json` con polígonos, candidatos, trazo en curso y notas. |
| Aprobar y exportar | Guarda el borrador, valida, escribe la etiqueta provisional, la relee y la dibuja sobre RGB y espectral. Si coincide, pulsa «Coincide: publicar». |

Mientras la lupa o la cruz de la barra de herramientas estén activas, los clics no editan; desactívalas para volver a dibujar. En VS Code, si se pide permiso para cargar los scripts de widgets desde la CDN, acéptalo. Si el lienzo no responde, instala `ipympl` en el entorno gee (`conda install -n gee -c conda-forge ipympl`) y reinicia el kernel.

**Criterio de anotación**
- Dibuja un polígono por cada parche continuo de nieve o hielo visible en esa adquisición. Revisa las cumbres y el resto del encuadre a resolución nativa. La permanencia glaciar requiere evidencia temporal adicional.
- Si nube, sombra o falta de datos impiden decidir en una zona relevante, marca «Pendiente» y escribe una nota. Una escena pendiente no se puede aprobar.
- **Negativa explícita**: sin polígonos, marca «Revisé toda la superficie» y «Negativa». Un borrador vacío nunca se aprueba como negativa.
- Para comparar después con NDSI de forma independiente, anota las fechas de prueba con `poligonos_ndsi_al_abrir = False`, solo con la vista RGB y sin sugerencias. El JSON registra qué vistas de índices y qué sugerencias se usaron (`anotacion_manual_sin_indices`).

**Coordenadas**: bordes de píxel con origen arriba a la izquierda; los límites son [0, W] × [0, H] y el centro del píxel (columna c, fila f) es (c + 0,5; f + 0,5). Bajo el lienzo se muestran la columna, la fila y el NDSI del cursor.

**Salida** (dentro de la carpeta de la escena; el TXT tiene el mismo nombre base que los dos PNG y sirve para ambos):

```
anotaciones/borrador.json
anotaciones/revisiones/0001/anotacion.json
anotaciones/revisiones/0001/labels/<nombre_escena>.txt
```

Usa siempre la revisión de número mayor. Una corrección posterior parte de la última revisión y publica la siguiente (0002, …) sin modificar las anteriores.

In [173]:
# 9a. Parámetros de anotación y entorno (esta celda basta para empezar desde un kernel recién iniciado).
import os, sys
from pathlib import Path

ruta_escena = ""                # "" = escenas descargadas en esta sesión o todas las de datos_sentinel2
poligonos_ndsi_al_abrir = True  # escena nueva: dibuja los parches NDSI ≥ umbral de toda la imagen (False = anotación manual)
umbral_ndsi_inicial = 0.40      # punto de partida experimental del NDSI (editable en el editor)
area_minima_candidato_px = 4    # los componentes NDSI más pequeños no se dibujan ni se proponen
area_minima_hueco_px = 4        # huecos menores se cierran (queda registrado en el candidato)
simplificar_candidato_px = 0.0  # tolerancia de simplificación de candidatos (0 = solo quita vértices colineales)

# Mismo ajuste de entorno que 2a: datos PROJ/GDAL y DLL del propio entorno gee.
for _var, _subdirs in {"PROJ_DATA": ("Library/share/proj", "share/proj"),
                       "GDAL_DATA": ("Library/share/gdal", "share/gdal")}.items():
    for _sub in _subdirs:
        _ruta = Path(sys.prefix) / _sub
        if _ruta.is_dir():
            os.environ[_var] = str(_ruta)
            if _var == "PROJ_DATA":
                os.environ["PROJ_LIB"] = str(_ruta)
            break
if os.name == "nt":
    _dll = [str(Path(sys.prefix) / d) for d in ("Library/bin", ".") if (Path(sys.prefix) / d).is_dir()]
    os.environ["PATH"] = os.pathsep.join(_dll + [p for p in os.environ.get("PATH", "").split(os.pathsep) if p not in _dll])

import copy, hashlib, io, json, math, re, shutil, tempfile, time, uuid
from datetime import datetime, timezone
from itertools import combinations

import numpy as np
import rasterio
from rasterio import Affine
from rasterio.crs import CRS as CRSRasterio
from rasterio.features import geometry_mask, shapes as formas_raster
from PIL import Image
from shapely.geometry import LinearRing, Point, Polygon as PoligonoShp, shape as forma_shp
from shapely.ops import unary_union
import matplotlib
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.backend_bases import MouseEvent
from matplotlib.patches import Polygon as PoligonoMpl
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as ImagenIPy

def _registrar_backends_nuevos():
    """Kernel abierto antes de instalar ipympl: matplotlib guardó su lista de backends sin él; se añaden los nuevos."""
    from matplotlib.backends.registry import backend_registry as registro
    conocidos = set(registro._backend_to_gui_framework) | set(registro._BUILTIN_BACKEND_TO_GUI_FRAMEWORK)
    registro._validate_and_store_entry_points(
        [(n, m) for n, m in registro._read_entry_points() if n.lower() not in conocidos])


backend_previo = matplotlib.get_backend()
try:
    try:
        import ipympl
    except ValueError:
        _registrar_backends_nuevos()
        import ipympl
    from ipympl.backend_nbagg import Canvas as LienzoIpympl, FigureManager as GestorIpympl
    lienzo_interactivo_ok = True
    print(f"ipympl {ipympl.__version__}: lienzo interactivo disponible.")
except ImportError:
    lienzo_interactivo_ok = False
    print("Falta ipympl: el editor se mostrará como imagen estática. Instálalo en el entorno gee con\n"
          "    conda install -n gee -c conda-forge ipympl\ny reinicia el kernel.")
except Exception as e:
    lienzo_interactivo_ok = False
    print(f"ipympl no pudo activarse en este kernel ({type(e).__name__}): el editor será estático.\n"
          "Reinicia el kernel (Restart) y ejecuta de nuevo 9a.")
# El editor crea su propio lienzo ipympl; las figuras de pyplot (sección 7) conservan el backend que tenían.
if matplotlib.get_backend() != backend_previo:
    matplotlib.use(backend_previo)

# Reutiliza las funciones de la sección 6 si ya están definidas; si no (kernel nuevo), las define aquí.
if "sha256_archivo" not in globals():
    def sha256_archivo(ruta):
        h = hashlib.sha256()
        with open(ruta, "rb") as f:
            for trozo in iter(lambda: f.read(1 << 20), b""):
                h.update(trozo)
        return h.hexdigest()

if "leer_metadata" not in globals():
    def leer_metadata(carpeta):
        try:
            return json.loads((Path(carpeta) / "metadata.json").read_text(encoding="utf-8"))
        except (OSError, ValueError):
            return None

ipympl 0.10.0: lienzo interactivo disponible.


In [174]:
# 9b. Lógica de anotación sin Earth Engine: lectura de la escena, geometría, NDSI, formato YOLO y revisiones.
CLASE_ID, CLASE_NOMBRE = 0, "nieve_hielo_visible"
REGION_IMAGEN = "r_imagen"          # región de los polígonos NDSI automáticos (toda la imagen)
FORMATO_ANOTACION = "1.0"
TOLERANCIA_IDA_VUELTA_PX = 0.01


class ErrorEscena(Exception):
    """Problema en los archivos de una escena: se detiene la edición de esa escena."""


def ahora_utc():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def nuevo_id(prefijo):
    return f"{prefijo}_{uuid.uuid4().hex[:10]}"


# --- Lectura y verificación de los artefactos de la descarga -----------------------------
def cargar_escena(carpeta):
    """Lee metadata, PNG e índices; verifica nombres, huellas, tamaños, transformación y CRS."""
    carpeta = Path(carpeta)
    meta = leer_metadata(carpeta)
    if meta is None:
        raise ErrorEscena(f"{carpeta / 'metadata.json'}: falta o no es un JSON válido.")
    problemas = []
    nombre = str(meta.get("nombre_escena", ""))
    if meta.get("estado") != "completa":
        problemas.append(f"metadata.json: estado {meta.get('estado')!r}; se esperaba 'completa'.")
    if carpeta.name != nombre:
        problemas.append(f"La carpeta '{carpeta.name}' no coincide con nombre_escena '{nombre}' de metadata.json.")
    esperadas = {"indices": "indices.tif", "calidad": "calidad.tif", "soporte_png": "soporte_png.tif",
                 "rgb": f"rgb/{nombre}.png", "espectral": f"espectral/{nombre}.png"}
    rutas = meta.get("rutas", {})
    for k, rel in esperadas.items():
        if rutas.get(k) != rel:
            problemas.append(f"metadata.json rutas['{k}'] = {rutas.get(k)!r}; se esperaba {rel!r} (clave distinta).")
    try:
        W, H = int(meta["malla"]["ancho"]), int(meta["malla"]["alto"])
        T = Affine(*meta["malla"]["transform"][:6])
        crs = CRSRasterio.from_user_input(meta["malla"]["crs"])
    except (KeyError, TypeError, ValueError) as e:
        raise ErrorEscena(f"metadata.json: malla incompleta o inválida ({e}).")
    archivos, huellas = meta.get("archivos", {}), {}
    for k, rel in esperadas.items():
        ruta = carpeta / rel
        if not ruta.is_file():
            problemas.append(f"Falta {rel}.")
            continue
        huellas[k] = sha256_archivo(ruta)
        if rel in archivos and huellas[k] != archivos[rel].get("sha256"):
            problemas.append(f"{rel}: SHA-256 distinto del registrado en metadata.json (archivo modificado).")
    if problemas:
        raise ErrorEscena("\n".join(problemas))

    capas = {}
    for k in ("indices", "calidad", "soporte_png"):
        with rasterio.open(carpeta / esperadas[k]) as ds:
            dif = []
            if (ds.width, ds.height) != (W, H):
                dif.append(f"tamaño {ds.width}×{ds.height} ≠ malla {W}×{H}")
            if not ds.transform.almost_equals(T, precision=1e-6):
                dif.append(f"transformación {tuple(ds.transform)[:6]} ≠ malla {tuple(T)[:6]} (capa desplazada)")
            if ds.crs is None or CRSRasterio.from_user_input(ds.crs) != crs:
                dif.append(f"CRS {ds.crs} ≠ malla {crs}")
            if dif:
                problemas.append(f"{esperadas[k]}: " + "; ".join(dif) + ".")
            else:
                capas[k] = (ds.read(), ds.nodata)
    imagenes = {}
    for k in ("rgb", "espectral"):
        with Image.open(carpeta / esperadas[k]) as im:
            if im.size != (W, H) or im.mode != "RGB":
                problemas.append(f"{esperadas[k]}: {im.size[0]}×{im.size[1]} {im.mode}; se esperaba {W}×{H} RGB.")
            else:
                imagenes[k] = np.asarray(im)
    if problemas:
        raise ErrorEscena("\n".join(problemas))

    datos, nodata = capas["indices"]
    orden = meta.get("bandas", {}).get("indices.tif", ["NDSI", "NDVI"])
    invalido = ~np.isfinite(datos) | ((datos == nodata) if nodata is not None else False)
    datos = np.where(invalido, np.nan, datos).astype("float32")
    identidad = {"id_asset": meta.get("id_asset"), "nombre_escena": nombre, "clave": meta.get("clave"),
                 "grupo_temporal": meta.get("grupo_temporal"), "ancho": W, "alto": H,
                 "receta_huella": meta.get("receta", {}).get("huella"),
                 "sha256_rgb": huellas["rgb"], "sha256_espectral": huellas["espectral"]}
    return {"carpeta": carpeta, "meta": meta, "nombre": nombre, "clave": meta.get("clave", nombre),
            "adquisicion_utc": meta.get("adquisicion_utc", "?"), "ancho": W, "alto": H,
            "rgb": imagenes["rgb"], "espectral": imagenes["espectral"],
            "ndsi": datos[orden.index("NDSI")], "ndvi": datos[orden.index("NDVI")] if "NDVI" in orden else None,
            "calidad": capas["calidad"][0][0], "soporte": capas["soporte_png"][0][0] == 1, "identidad": identidad}


# --- Rutas de anotación y listado de escenas ----------------------------------------------
def rutas_anotacion(carpeta):
    base = Path(carpeta) / "anotaciones"
    return base, base / "borrador.json", base / "revisiones"


def revisiones_publicadas(carpeta):
    rev = rutas_anotacion(carpeta)[2]
    if not rev.is_dir():
        return []
    return sorted((p for p in rev.iterdir() if p.is_dir() and re.fullmatch(r"\d{4}", p.name)), key=lambda p: int(p.name))


def ultima_revision(carpeta):
    """(carpeta, anotacion.json) de la revisión aprobada más reciente, o (None, None)."""
    for p in reversed(revisiones_publicadas(carpeta)):
        try:
            return p, json.loads((p / "anotacion.json").read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
    return None, None


def estado_anotacion(carpeta):
    _, borrador, _ = rutas_anotacion(carpeta)
    p, rev = ultima_revision(carpeta)
    texto = ""
    if rev:
        texto = (f"negativa r{p.name}" if rev.get("tipo") == "negativa"
                 else f"aprobada r{p.name} ({rev.get('instancias', '?')} parches)")
    if borrador.is_file():
        try:
            b = json.loads(borrador.read_text(encoding="utf-8")).get("estado", "borrador")
        except (OSError, ValueError):
            b = "borrador ilegible"
        return f"{b} · {texto}" if texto else b
    return texto or "sin anotar"


def escenas_en(ruta):
    """Escenas (carpetas con metadata.json) en la ruta: una escena, la raíz de un AOI o el directorio de salida."""
    ruta = Path(ruta)
    if (ruta / "metadata.json").is_file():
        return [ruta]
    if not ruta.is_dir():
        return []
    encontradas = set()
    for patron in ("*/metadata.json", "*/*/metadata.json"):
        for m in ruta.glob(patron):
            partes = m.parent.relative_to(ruta).parts
            if not any(p.startswith("_") or p == "anotaciones" for p in partes):
                encontradas.add(m.parent)
    return sorted(encontradas, key=lambda c: ((leer_metadata(c) or {}).get("adquisicion_utc", ""), c.name))


def etiqueta_escena(carpeta):
    meta = leer_metadata(carpeta) or {}
    fecha = str(meta.get("adquisicion_utc", "?"))[:16].replace("T", " ")
    return f"{fecha} UTC · {meta.get('clave', Path(carpeta).name)} · {estado_anotacion(carpeta)}"


# --- Geometría en coordenadas de bordes de píxel ------------------------------------------
def validar_poligono(vertices, W, H):
    """Lista de problemas del anillo (vacía si es válido): finito, dentro de [0,W]×[0,H], simple y con área."""
    v = np.asarray(vertices, dtype=float)
    if v.ndim != 2 or v.shape[1] != 2:
        return ["formato de vértices inválido"]
    if not np.isfinite(v).all():
        return ["coordenadas no finitas"]
    problemas = []
    if (v[:, 0] < 0).any() or (v[:, 0] > W).any() or (v[:, 1] < 0).any() or (v[:, 1] > H).any():
        problemas.append(f"vértices fuera del lienzo [0,{W}]×[0,{H}]")
    if len({(float(x), float(y)) for x, y in v}) < 3:
        return problemas + ["menos de 3 vértices distintos"]
    if PoligonoShp(v).area <= 1e-9:
        problemas.append("área nula")
    elif not LinearRing(v).is_simple:
        problemas.append("el contorno se autointersecta")
    return problemas


def mascara_poligono(vertices, W, H):
    """Píxeles cuyo centro (c+0,5; f+0,5) cae dentro del polígono."""
    anillo = [[float(x), float(y)] for x, y in vertices]
    return geometry_mask([{"type": "Polygon", "coordinates": [anillo + [anillo[0]]]}], out_shape=(H, W),
                         transform=Affine.identity(), invert=True)


def lectura_cursor(escena, x, y):
    if x is None or y is None:
        return ""
    c, f = int(math.floor(x)), int(math.floor(y))
    if not (0 <= c < escena["ancho"] and 0 <= f < escena["alto"]):
        return f"x={x:.1f} y={y:.1f} (fuera de la imagen)"
    v = escena["ndsi"][f, c]
    texto = f"columna {c} · fila {f} · NDSI {'sin dato' if np.isnan(v) else f'{v:+.3f}'}"
    if not escena["soporte"][f, c]:
        texto += " · sin soporte"
    elif escena["calidad"][f, c] == 1:
        texto += " · calidad dudosa"
    return texto


def sugerir_candidatos(ndsi, soporte, region, umbral, excluir=(), descartes=(), area_min=4, hueco_min=4,
                       simplificar=0.0):
    """Componentes 4-conexas de NDSI ≥ umbral en soporte válido dentro de la región, vectorizadas en la malla."""
    H, W = ndsi.shape
    dentro = mascara_poligono(region, W, H)
    with np.errstate(invalid="ignore"):
        binaria = dentro & soporte & (ndsi >= umbral)
    for v in excluir:                       # no se proponen zonas ya cubiertas por polígonos
        binaria &= ~mascara_poligono(v, W, H)
    omitidos = {"pequenos": 0, "descartados": 0}
    if not binaria.any():
        return [], omitidos
    borde_region = unary_union([forma_shp(g) for g, _ in formas_raster(
        dentro.astype("uint8"), mask=dentro, connectivity=4, transform=Affine.identity())]).boundary
    geoms_descarte = [PoligonoShp(d).buffer(0) for d in descartes]
    candidatos = []
    for g, _ in formas_raster(binaria.astype("uint8"), mask=binaria, connectivity=4, transform=Affine.identity()):
        p = forma_shp(g)
        if p.area < area_min:
            omitidos["pequenos"] += 1
            continue
        if any(p.intersection(d).area > 0 for d in geoms_descarte):
            omitidos["descartados"] += 1
            continue
        huecos = [h for h in p.interiors if PoligonoShp(h).area >= hueco_min]
        cerrados = len(p.interiors) - len(huecos)
        p = PoligonoShp(p.exterior, huecos).simplify(simplificar, preserve_topology=True)
        candidatos.append({
            "vertices": [[float(x), float(y)] for x, y in list(p.exterior.coords)[:-1]],
            "huecos": [[[float(x), float(y)] for x, y in list(h.coords)[:-1]] for h in p.interiors],
            "huecos_menores_cerrados": cerrados, "area_px": float(p.area),
            "toca_limite": bool(p.exterior.intersection(borde_region).length > 0)})
    return candidatos, omitidos


# --- Formato YOLO Segmentation ------------------------------------------------------------
def serializar_yolo(poligonos, W, H):
    """Una línea por parche: clase y pares x/W y/H con 8 decimales, sin repetir el primer vértice."""
    lineas = []
    for v in poligonos:
        pares = " ".join(f"{x / W + 0.0:.8f} {y / H + 0.0:.8f}" for x, y in v)
        lineas.append(f"{CLASE_ID} {pares}\n")
    return "".join(lineas)


def leer_txt_yolo(ruta):
    """Lector independiente del serializador: [(clase, [(x, y), ...])] con valores normalizados."""
    instancias = []
    texto = Path(ruta).read_bytes().decode("utf-8")
    for n, linea in enumerate(texto.splitlines(), 1):
        campos = linea.split()
        if not campos:
            raise ValueError(f"línea {n} vacía")
        if any(not re.fullmatch(r"-?\d+\.\d{8}", c) for c in campos[1:]):
            raise ValueError(f"línea {n}: hay valores sin 8 decimales")
        valores = [float(c) for c in campos[1:]]
        if len(valores) % 2:
            raise ValueError(f"línea {n}: número impar de coordenadas")
        instancias.append((int(campos[0]), list(zip(valores[0::2], valores[1::2]))))
    return instancias


def verificar_ida_vuelta(ruta, poligonos, W, H, tolerancia_px=TOLERANCIA_IDA_VUELTA_PX):
    """Relee el TXT y compara clase, recuento, orden, vértices y rango; devuelve contornos en píxeles."""
    instancias = leer_txt_yolo(ruta)
    if len(instancias) != len(poligonos):
        raise ValueError(f"el TXT tiene {len(instancias)} instancias y la anotación {len(poligonos)}")
    recuperados, error_max = [], 0.0
    for i, ((clase, pares), v) in enumerate(zip(instancias, poligonos), 1):
        if clase != CLASE_ID:
            raise ValueError(f"instancia {i}: clase {clase} ≠ {CLASE_ID}")
        if len(pares) != len(v):
            raise ValueError(f"instancia {i}: {len(pares)} vértices en el TXT y {len(v)} en la anotación")
        norm = np.asarray(pares, dtype=float)
        if (norm < 0).any() or (norm > 1).any():
            raise ValueError(f"instancia {i}: coordenadas fuera de [0, 1]")
        px = norm * [W, H]
        error = float(np.abs(px - np.asarray(v, dtype=float)).max())
        if error > tolerancia_px:
            raise ValueError(f"instancia {i}: error {error:.4f} px > {tolerancia_px} px")
        error_max = max(error_max, error)
        recuperados.append(px.tolist())
    return recuperados, error_max


# --- Estado editable --------------------------------------------------------------------
class EstadoAnotacion:
    """Polígonos, candidatos NDSI, trazo y decisiones de una escena (coordenadas de bordes de píxel)."""
    CAMPOS_GEOMETRIA = ("poligonos", "candidatos", "regiones", "descartes", "trazo", "tipo_trazo",
                        "seleccion", "vertice_activo", "umbral")

    def __init__(self, W, H, umbral=0.40, area_min=4, hueco_min=4, simplificar=0.0):
        self.W, self.H = W, H
        self.poligonos, self.candidatos, self.regiones, self.descartes = [], [], [], []
        self.trazo, self.tipo_trazo = [], "parche"
        self.seleccion, self.vertice_activo = None, None     # ("poligono"|"candidato", índice)
        self.notas, self.pendiente, self.revision_completa, self.negativa = "", False, False, False
        self.revision_base, self.creado_utc = None, ahora_utc()
        self.umbral = float(umbral)
        self.parametros = {"area_min": area_min, "hueco_min": hueco_min, "simplificar": simplificar}
        self.ayudas = {"vista_espectral": False, "vista_ndsi": False, "sugerencias_generadas": 0,
                       "candidatos_aceptados": 0, "poligonos_automaticos": 0}
        self._historial = []
        self.version = 0                                     # cambia con cada modificación

    # Historial y cambios
    def _registrar(self):
        self._historial.append(copy.deepcopy({k: getattr(self, k) for k in self.CAMPOS_GEOMETRIA}))
        del self._historial[:-200]

    def _cambio(self):
        self.version += 1

    def deshacer(self):
        if not self._historial:
            raise ValueError("No hay ediciones que deshacer.")
        for k, v in self._historial.pop().items():
            setattr(self, k, v)
        self._cambio()

    def _limitar(self, x, y):
        # Recorta al lienzo y redondea a 1e-6 px (elimina ruido numérico de las transformaciones de pantalla).
        return [round(min(max(float(x), 0.0), float(self.W)), 6) + 0.0,
                round(min(max(float(y), 0.0), float(self.H)), 6) + 0.0]

    def elemento_seleccionado(self):
        if self.seleccion is None:
            return None
        tipo, i = self.seleccion
        lista = self.poligonos if tipo == "poligono" else self.candidatos
        return lista[i] if i < len(lista) else None

    # Trazo
    def anadir_vertice(self, x, y, tipo="parche"):
        if self.trazo and self.tipo_trazo != tipo:
            raise ValueError(f"Hay un trazo de {self.tipo_trazo} en curso: ciérralo o cancélalo antes.")
        self._registrar()
        self.tipo_trazo = tipo
        self.trazo.append(self._limitar(x, y))
        self._cambio()

    def cancelar_trazo(self):
        if not self.trazo:
            raise ValueError("No hay trazo en curso.")
        self._registrar()
        self.trazo = []
        self._cambio()

    def cerrar_trazo(self):
        v = [list(p) for p in self.trazo]
        v = [p for k, p in enumerate(v) if k == 0 or p != v[k - 1]]
        if len(v) > 1 and v[0] == v[-1]:
            v = v[:-1]
        if len(v) < 3:
            raise ValueError("Se necesitan al menos 3 vértices distintos para cerrar.")
        problemas = validar_poligono(v, self.W, self.H)
        if problemas:
            raise ValueError("No se puede cerrar: " + "; ".join(problemas) + ". Usa Deshacer o Cancelar trazo.")
        self._registrar()
        self.trazo = []
        if self.tipo_trazo == "region":
            region = {"id": nuevo_id("r"), "vertices": v}
            self.regiones.append(region)
            self._cambio()
            return "region", region
        self.poligonos.append({"id": nuevo_id("p"), "origen": "manual", "vertices": v, "creado_utc": ahora_utc()})
        self.seleccion, self.vertice_activo = ("poligono", len(self.poligonos) - 1), None
        self._cambio()
        return "poligono", self.poligonos[-1]

    # Selección y edición de vértices
    def seleccionar_en(self, x, y, tol):
        pt, opciones = Point(x, y), []
        for tipo, lista in (("candidato", self.candidatos), ("poligono", self.poligonos)):
            for i, e in enumerate(lista):
                g = PoligonoShp(e["vertices"])
                if g.contains(pt) or g.exterior.distance(pt) <= tol:
                    opciones.append((g.area, tipo, i))
        self.seleccion = min(opciones)[1:] if opciones else None
        self.vertice_activo = None
        return self.seleccion

    def vertice_cercano(self, x, y, tol):
        e = self.elemento_seleccionado()
        if e is None:
            return None
        d = np.hypot(*(np.asarray(e["vertices"], dtype=float) - [x, y]).T)
        i = int(d.argmin())
        return i if d[i] <= tol else None

    def _marcar_editado(self, e):
        if e.get("origen") == "asistido" or "region_id" in e:
            e["editado"] = True

    def insertar_en_lado(self, x, y, tol):
        e = self.elemento_seleccionado()
        if e is None:
            return False
        v, p, mejor = np.asarray(e["vertices"], dtype=float), np.array([x, y], dtype=float), None
        for i in range(len(v)):
            a, b = v[i], v[(i + 1) % len(v)]
            ab = b - a
            t = float(np.clip(np.dot(p - a, ab) / max(np.dot(ab, ab), 1e-12), 0, 1))
            q = a + t * ab
            d = float(np.hypot(*(p - q)))
            if d <= tol and (mejor is None or d < mejor[0]):
                mejor = (d, i, q)
        if mejor is None:
            return False
        self._registrar()
        e["vertices"].insert(mejor[1] + 1, self._limitar(*mejor[2]))
        self.vertice_activo = mejor[1] + 1
        self._marcar_editado(e)
        self._cambio()
        return True

    def mover_vertice(self, i, x, y):
        e = self.elemento_seleccionado()
        e["vertices"][i] = self._limitar(x, y)
        self._marcar_editado(e)
        self._cambio()

    def eliminar_vertice(self):
        e = self.elemento_seleccionado()
        if e is None or self.vertice_activo is None:
            raise ValueError("Selecciona un vértice: en modo Editar, haz clic sobre él.")
        if len(e["vertices"]) <= 3:
            raise ValueError("Un polígono necesita al menos 3 vértices; usa «Eliminar polígono».")
        self._registrar()
        e["vertices"].pop(self.vertice_activo)
        self.vertice_activo = None
        self._marcar_editado(e)
        self._cambio()

    def eliminar_seleccion(self):
        if self.seleccion is None:
            raise ValueError("Selecciona un polígono: en modo Editar, haz clic sobre su contorno.")
        if self.seleccion[0] == "candidato":
            return self.descartar_candidato()
        self._registrar()
        p = self.poligonos.pop(self.seleccion[1])
        if p.get("automatico"):   # un parche automático eliminado no vuelve a dibujarse al cambiar el umbral
            self.descartes.append({"vertices": p["vertices"], "umbral": p["umbral"], "region_id": p["region_id"]})
        self.seleccion, self.vertice_activo = None, None
        self._cambio()

    # Candidatos NDSI
    def _candidato_seleccionado(self):
        if self.seleccion is None or self.seleccion[0] != "candidato":
            raise ValueError("Selecciona un candidato (contorno cian) en modo Editar.")
        return self.candidatos[self.seleccion[1]]

    def aceptar_candidato(self):
        c = self._candidato_seleccionado()
        if c["huecos"]:
            raise ValueError(f"El candidato tiene {len(c['huecos'])} hueco(s) interior(es) relevante(s) y YOLO usa un "
                             "anillo simple por instancia. Dibújalo manualmente (p. ej. en dos parches) o descártalo; "
                             "mientras quede pendiente la escena seguirá en borrador.")
        problemas = validar_poligono(c["vertices"], self.W, self.H)
        if problemas:
            raise ValueError("Candidato inválido: " + "; ".join(problemas))
        self._registrar()
        self.candidatos.pop(self.seleccion[1])
        self.poligonos.append(self._poligono_asistido(c))
        self.ayudas["candidatos_aceptados"] += 1
        self.seleccion, self.vertice_activo = ("poligono", len(self.poligonos) - 1), None
        self._cambio()

    def _poligono_asistido(self, c, automatico=False):
        region = next((r["vertices"] for r in self.regiones if r["id"] == c["region_id"]), None)
        return {"id": c["id"].replace("c_", "p_", 1), "origen": "asistido", "vertices": c["vertices"],
                "umbral": c["umbral"], "region": region, "region_id": c["region_id"], "automatico": automatico,
                "toca_limite": c["toca_limite"], "huecos_menores_cerrados": c["huecos_menores_cerrados"],
                "editado": c.get("editado", False), "creado_utc": ahora_utc()}

    def descartar_candidato(self):
        c = self._candidato_seleccionado()
        self._registrar()
        self.candidatos.pop(self.seleccion[1])
        self.descartes.append({"vertices": c["vertices"], "umbral": c["umbral"], "region_id": c["region_id"]})
        self.seleccion, self.vertice_activo = None, None
        self._cambio()

    def _sugerir_en(self, escena, region, umbral):
        candidatos, omitidos = sugerir_candidatos(
            escena["ndsi"], escena["soporte"], region["vertices"], umbral,
            excluir=[p["vertices"] for p in self.poligonos], descartes=[d["vertices"] for d in self.descartes],
            area_min=self.parametros["area_min"], hueco_min=self.parametros["hueco_min"],
            simplificar=self.parametros["simplificar"])
        for c in candidatos:
            c.update(id=nuevo_id("c"), umbral=float(umbral), region_id=region["id"], editado=False)
        self.candidatos.extend(candidatos)
        self.ayudas["sugerencias_generadas"] += 1
        return candidatos, omitidos

    def _rehacer_automaticos(self, escena, umbral):
        """Sustituye los polígonos automáticos no editados por los parches NDSI ≥ umbral de toda la imagen."""
        region = next((r for r in self.regiones if r["id"] == REGION_IMAGEN), None)
        if region is None:
            region = {"id": REGION_IMAGEN, "vertices": [[0.0, 0.0], [float(self.W), 0.0],
                                                        [float(self.W), float(self.H)], [0.0, float(self.H)]]}
            self.regiones.insert(0, region)
        self.poligonos = [p for p in self.poligonos if not (p.get("automatico") and not p.get("editado"))]
        self.candidatos = [c for c in self.candidatos if c["region_id"] != REGION_IMAGEN]
        self.seleccion, self.vertice_activo = None, None
        candidatos, omitidos = self._sugerir_en(escena, region, umbral)
        dibujados = []
        for c in candidatos:
            if c["huecos"] or validar_poligono(c["vertices"], self.W, self.H):
                continue                          # queda como candidato pendiente
            self.candidatos.remove(c)
            self.poligonos.append(self._poligono_asistido(c, automatico=True))
            dibujados.append(self.poligonos[-1])
        self.ayudas["poligonos_automaticos"] = self.ayudas.get("poligonos_automaticos", 0) + len(dibujados)
        pendientes = [c for c in self.candidatos if c["region_id"] == REGION_IMAGEN]
        return dibujados, pendientes, omitidos

    def dibujar_ndsi(self, escena, umbral):
        """Dibuja como polígonos asistidos los parches NDSI ≥ umbral de toda la imagen (los de huecos quedan pendientes)."""
        self._registrar()
        resultado = self._rehacer_automaticos(escena, umbral)
        self.umbral = float(umbral)
        self._cambio()
        return resultado

    def regenerar(self, escena, umbral):
        """Cambia el umbral: redibuja los automáticos no editados y recalcula los candidatos pendientes de las regiones."""
        manuales = [r for r in self.regiones if r["id"] != REGION_IMAGEN]
        ids = {c["region_id"] for c in self.candidatos if c["region_id"] != REGION_IMAGEN}
        if manuales:
            ids.add(manuales[-1]["id"])
        automatico = len(manuales) < len(self.regiones)
        if not ids and not automatico:
            self.umbral = float(umbral)
            return [], {}
        self._registrar()
        nuevos, omitidos = [], {"pequenos": 0, "descartados": 0}
        if automatico:
            dibujados, pendientes, o = self._rehacer_automaticos(escena, umbral)
            nuevos += pendientes
            omitidos = {k: omitidos[k] + o.get(k, 0) for k in omitidos}
            omitidos["dibujados"] = len(dibujados)
        self.candidatos = [c for c in self.candidatos if c["region_id"] == REGION_IMAGEN]
        if self.seleccion and self.seleccion[0] == "candidato":
            self.seleccion, self.vertice_activo = None, None
        for r in manuales:
            if r["id"] in ids:
                c, o = self._sugerir_en(escena, r, umbral)
                nuevos += c
                omitidos = {k: omitidos.get(k, 0) + o.get(k, 0) for k in set(omitidos) | set(o)}
        self.umbral = float(umbral)
        self._cambio()
        return nuevos, omitidos

    # Serialización del borrador
    def a_dict(self, identidad):
        manual = not (self.ayudas["vista_espectral"] or self.ayudas["vista_ndsi"] or self.ayudas["sugerencias_generadas"])
        return copy.deepcopy({
            "formato": FORMATO_ANOTACION, "estado": "pendiente" if self.pendiente else "borrador",
            "escena": identidad, "clase": {"id": CLASE_ID, "nombre": CLASE_NOMBRE},
            "coordenadas": "bordes de píxel; origen arriba a la izquierda; límites [0,W]x[0,H]",
            "poligonos": self.poligonos, "candidatos_pendientes": self.candidatos, "regiones": self.regiones,
            "descartes": self.descartes, "trazo_en_curso": {"tipo": self.tipo_trazo, "vertices": self.trazo},
            "notas": self.notas, "pendiente": self.pendiente, "revision_completa": self.revision_completa,
            "negativa": self.negativa, "revision_base": self.revision_base, "umbral_ndsi": self.umbral,
            "ayudas": self.ayudas, "anotacion_manual_sin_indices": manual,
            "creado_utc": self.creado_utc, "actualizado_utc": ahora_utc()})

    def desde_dict(self, d, desde_revision=False):
        d = copy.deepcopy(d)
        self.poligonos, self.regiones = d.get("poligonos", []), d.get("regiones", [])
        self.candidatos, self.descartes = d.get("candidatos_pendientes", []), d.get("descartes", [])
        trazo = d.get("trazo_en_curso") or {}
        self.trazo, self.tipo_trazo = trazo.get("vertices", []), trazo.get("tipo", "parche")
        self.notas, self.pendiente = d.get("notas", ""), bool(d.get("pendiente", False))
        self.umbral = float(d.get("umbral_ndsi", self.umbral))
        self.ayudas.update(d.get("ayudas", {}))
        self.creado_utc = d.get("creado_utc", self.creado_utc)
        if desde_revision:   # una corrección parte de la revisión y exige confirmar de nuevo
            self.revision_base = d.get("revision")
            self.revision_completa = self.negativa = False
        else:
            self.revision_base = d.get("revision_base")
            self.revision_completa, self.negativa = bool(d.get("revision_completa")), bool(d.get("negativa"))
        self.seleccion, self.vertice_activo, self._historial = None, None, []


# --- Persistencia --------------------------------------------------------------------------
def escribir_json_atomico(ruta, datos, intentos=6):
    """Archivo temporal + fsync + reemplazo atómico (con reintentos si OneDrive bloquea un instante)."""
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    tmp = ruta.with_name(f".{ruta.name}.{uuid.uuid4().hex[:8]}.tmp")
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(datos, f, ensure_ascii=False, indent=1)
            f.flush()
            os.fsync(f.fileno())
        for k in range(intentos):
            try:
                os.replace(tmp, ruta)
                return ruta
            except PermissionError:
                if k == intentos - 1:
                    raise
                time.sleep(0.5 * (k + 1))
    finally:
        if tmp.exists():
            tmp.unlink()


def guardar_borrador(escena, estado):
    return escribir_json_atomico(rutas_anotacion(escena["carpeta"])[1], estado.a_dict(escena["identidad"]))


def cargar_trabajo(escena, umbral=0.40, **parametros):
    """Abre el borrador; si no existe, la última revisión aprobada como base; si no, un estado vacío."""
    _, borrador, _ = rutas_anotacion(escena["carpeta"])
    estado = EstadoAnotacion(escena["ancho"], escena["alto"], umbral=umbral, **parametros)
    if borrador.is_file():
        try:
            datos, origen = json.loads(borrador.read_text(encoding="utf-8")), "borrador"
        except (OSError, ValueError) as e:
            raise ErrorEscena(f"{borrador}: no se puede leer ({e}).")
    else:
        p, datos = ultima_revision(escena["carpeta"])
        origen = f"revisión {p.name}" if datos else "nueva"
    if datos:
        guardada = datos.get("escena", {})
        dif = [f"{k}: guardado {guardada.get(k)!r} ≠ actual {v!r}" for k, v in escena["identidad"].items()
               if guardada.get(k) != v]
        if dif:
            raise ErrorEscena(f"La anotación ({origen}) corresponde a otros insumos; no se reanuda:\n" + "\n".join(dif))
        estado.desde_dict(datos, desde_revision=(origen != "borrador"))
    return estado, origen


# --- Aprobación, revisión provisional y publicación ----------------------------------------
def comprobar_aprobacion(estado, escena):
    """(errores, avisos). Con errores no se escribe ninguna revisión."""
    errores, avisos = [], []
    W, H = escena["ancho"], escena["alto"]
    if estado.pendiente:
        errores.append("La escena está marcada como pendiente" + ("" if estado.notas.strip() else " (y falta la nota)")
                       + ": no se puede aprobar.")
    if estado.trazo:
        errores.append("Hay un trazo en curso: ciérralo o cancélalo.")
    if estado.candidatos:
        errores.append(f"Quedan {len(estado.candidatos)} candidato(s) NDSI pendientes: acéptalos o descártalos.")
    if not estado.revision_completa:
        errores.append("Falta confirmar «Revisé toda la superficie observable».")
    if estado.negativa and estado.poligonos:
        errores.append("Está marcada como negativa pero tiene polígonos.")
    if not estado.poligonos and not estado.negativa:
        errores.append("No hay polígonos y un borrador vacío no es una negativa: marca «Negativa» si confirmas "
                       "que no hay nieve/hielo visible.")
    geoms = []
    for i, p in enumerate(estado.poligonos, 1):
        problemas = validar_poligono(p["vertices"], W, H)
        if problemas:
            errores.append(f"Polígono {i}: " + "; ".join(problemas) + ".")
            geoms.append(None)
            continue
        m = mascara_poligono(p["vertices"], W, H)
        sin_soporte, dudosos = int((m & ~escena["soporte"]).sum()), int((m & (escena["calidad"] == 1)).sum())
        if sin_soporte:
            errores.append(f"Polígono {i}: incluye {sin_soporte} píxel(es) sin soporte (sin datos compartidos).")
        if dudosos:
            avisos.append(f"Polígono {i}: incluye {dudosos} píxel(es) de calidad dudosa; revísalos.")
        if not m.any():
            avisos.append(f"Polígono {i}: no contiene ningún centro de píxel.")
        if p.get("origen") == "asistido" and p.get("toca_limite") and not p.get("editado"):
            avisos.append(f"Polígono {i}: sugerencia que toca el borde de su región; puede estar cortada.")
        geoms.append(PoligonoShp(p["vertices"]))
    for (i, a), (j, b) in combinations(enumerate(geoms, 1), 2):
        if a is not None and b is not None and a.intersects(b):
            area = a.intersection(b).area
            if area > 1e-6:
                errores.append(f"Polígonos {i} y {j} se solapan ({area:.2f} px²); el contacto de bordes sí se admite.")
    return errores, avisos


def siguiente_revision(carpeta):
    rev = rutas_anotacion(carpeta)[2]
    nums = [int(p.name) for p in rev.iterdir() if p.is_dir() and re.fullmatch(r"\d{4}", p.name)] if rev.is_dir() else []
    return max(nums, default=0) + 1


def preparar_revision(escena, estado, avisos=()):
    """Escribe TXT y anotacion.json en un directorio temporal y verifica la ida y vuelta del TXT."""
    W, H, nombre = escena["ancho"], escena["alto"], escena["nombre"]
    numero = siguiente_revision(escena["carpeta"])
    revisiones = rutas_anotacion(escena["carpeta"])[2]
    tmp = revisiones / f"_tmp_{numero:04d}_{uuid.uuid4().hex[:8]}"
    poligonos = [p["vertices"] for p in estado.poligonos]
    try:
        (tmp / "labels").mkdir(parents=True)
        ruta_txt = tmp / "labels" / f"{nombre}.txt"
        with open(ruta_txt, "w", encoding="utf-8", newline="\n") as f:
            f.write(serializar_yolo(poligonos, W, H))
            f.flush()
            os.fsync(f.fileno())
        recuperados, error_max = verificar_ida_vuelta(ruta_txt, poligonos, W, H)
        datos = estado.a_dict(escena["identidad"])
        for k in ("candidatos_pendientes", "trazo_en_curso", "pendiente", "revision_base"):
            datos.pop(k, None)
        datos.update(estado="aprobada", tipo="negativa" if estado.negativa else "positiva",
                     revision=f"{numero:04d}", revision_base=estado.revision_base, instancias=len(poligonos),
                     txt=f"labels/{nombre}.txt", sha256_txt=sha256_archivo(ruta_txt), aprobado_utc=ahora_utc(),
                     avisos=list(avisos),
                     verificacion={"lector": "leer_txt_yolo", "error_max_px": error_max,
                                   "tolerancia_px": TOLERANCIA_IDA_VUELTA_PX})
        escribir_json_atomico(tmp / "anotacion.json", datos)
    except Exception:
        shutil.rmtree(tmp, ignore_errors=True)
        raise
    return {"tmp": tmp, "numero": numero, "txt": ruta_txt, "recuperados": recuperados, "error_max": error_max,
            "instancias": len(poligonos), "version": estado.version}


def descartar_revision(provisional):
    if provisional:
        shutil.rmtree(provisional["tmp"], ignore_errors=True)


def publicar_revision(provisional, carpeta, renombrar=None, intentos=6):
    """Renombra el directorio temporal a revisiones/NNNN (mismo sistema de archivos). Si falla, no toca nada más."""
    renombrar = renombrar or os.rename
    _, borrador, revisiones = rutas_anotacion(carpeta)
    destino = revisiones / f"{provisional['numero']:04d}"
    try:
        if destino.exists():
            raise FileExistsError(f"La revisión {destino.name} ya existe; no se sobrescribe.")
        for k in range(intentos):
            try:
                renombrar(provisional["tmp"], destino)
                break
            except PermissionError:
                if k == intentos - 1:
                    raise
                time.sleep(0.5 * (k + 1))
    except Exception:
        descartar_revision(provisional)
        raise
    if borrador.exists():
        borrador.unlink()
    return destino / "labels" / provisional["txt"].name


def figura_verificacion(escena, contornos, titulo=""):
    """PNG con los contornos releídos del TXT sobre RGB y espectral, con idéntico encuadre."""
    W, H = escena["ancho"], escena["alto"]
    if contornos:
        todos = np.vstack(contornos)
        margen = max(20.0, 0.15 * float(np.ptp(todos, axis=0).max()))
        x0, y0 = np.maximum(todos.min(axis=0) - margen, 0)
        x1, y1 = np.minimum(todos.max(axis=0) + margen, [W, H])
    else:
        x0, y0, x1, y1 = 0, 0, W, H
    fig = Figure(figsize=(13, 6.6), dpi=90)
    FigureCanvasAgg(fig)
    for ax, clave, nombre in zip(fig.subplots(1, 2), ("rgb", "espectral"), ("RGB", "Espectral (NDSI, NDVI, B4)")):
        ax.imshow(escena[clave], extent=(0, W, H, 0), interpolation="nearest")
        for i, v in enumerate(contornos, 1):
            ax.add_patch(PoligonoMpl(v, closed=True, fill=False, ec="yellow", lw=1.5))
            p = PoligonoShp(v).representative_point()
            ax.text(p.x, p.y, str(i), color="yellow", fontsize=10, fontweight="bold", ha="center", va="center")
        ax.set_xlim(x0, x1)
        ax.set_ylim(y1, y0)
        ax.set_title(f"{nombre} · x {x0:.0f}–{x1:.0f}, y {y0:.0f}–{y1:.0f}", fontsize=10)
    fig.suptitle(titulo, fontsize=11)
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    return buf.getvalue()

In [175]:
# 9c. Editor interactivo (Matplotlib + ipympl + ipywidgets) y panel de selección de escenas.
_editores_activos = globals().get("_editores_activos", [])   # sobrevive a reejecuciones de esta celda
_numero_figura = globals().get("_numero_figura", 9000)
COLORES = {"manual": "#00e676", "asistido": "#e040fb", "seleccion": "#ffd400", "candidato": "#00ffff",
           "region": "white", "trazo": "#ff1744"}
CMAP_NDSI = matplotlib.colormaps["RdBu"].with_extremes(bad="0.5")


class EditorAnotacion:
    """Un lienzo en coordenadas de imagen con controles; al crear uno nuevo se cierran los anteriores."""

    def __init__(self, escena, interactivo=True, registro=None, al_cambiar_estado=None, ndsi_al_abrir=None):
        global _numero_figura
        # Primero el trabajo previo: si la identidad no coincide se lanza ErrorEscena sin tocar otros editores.
        self.estado, self.origen = cargar_trabajo(
            escena, umbral=globals().get("umbral_ndsi_inicial", 0.40),
            area_min=globals().get("area_minima_candidato_px", 4), hueco_min=globals().get("area_minima_hueco_px", 4),
            simplificar=globals().get("simplificar_candidato_px", 0.0))
        self.registro = _editores_activos if registro is None else registro
        for previo in list(self.registro):      # desconecta callbacks de ejecuciones anteriores
            previo.cerrar()
        self.registro.clear()
        self.registro.append(self)
        self.escena, self.activo, self.al_cambiar_estado = escena, True, al_cambiar_estado
        self.sin_guardar, self.provisional, self.ultimo_txt = False, None, None
        self._arrastre, self._sincronizando, self._artistas, self._ultima_celda = None, False, [], None
        W, H = escena["ancho"], escena["alto"]

        # Lienzo: ipympl si está disponible; si no, Agg (imagen estática, también usado en las pruebas).
        self.figura = Figure(figsize=(8.6, 8.2), dpi=90)
        self.interactivo = bool(interactivo and globals().get("lienzo_interactivo_ok"))
        if self.interactivo:
            _numero_figura += 1
            self.canvas = LienzoIpympl(self.figura)
            GestorIpympl(self.canvas, _numero_figura)
            self.canvas.header_visible = False
            self.canvas.toolbar_visible = "visible"
            self.canvas.capture_scroll = True
        else:
            self.canvas = FigureCanvasAgg(self.figura)
        self.imagen_estatica = None
        self.ax = self.figura.add_axes([0.01, 0.02, 0.86, 0.96])
        self.cax = self.figura.add_axes([0.90, 0.15, 0.025, 0.7])
        extension = (0, W, H, 0)
        self.capas = {
            "rgb": self.ax.imshow(escena["rgb"], extent=extension, interpolation="nearest", zorder=0),
            "espectral": self.ax.imshow(escena["espectral"], extent=extension, interpolation="nearest", zorder=0),
            "ndsi": self.ax.imshow(np.ma.masked_invalid(escena["ndsi"]), extent=extension, interpolation="nearest",
                                   cmap=CMAP_NDSI, vmin=-1, vmax=1, zorder=0)}
        self.figura.colorbar(self.capas["ndsi"], cax=self.cax, label="NDSI")
        rgba_calidad = np.zeros((H, W, 4), "float32")
        rgba_calidad[(escena["calidad"] == 1) & escena["soporte"]] = (1.0, 0.55, 0.0, 0.45)
        rgba_soporte = np.zeros((H, W, 4), "float32")
        rgba_soporte[~escena["soporte"]] = (1.0, 0.0, 1.0, 0.6)
        self.capa_calidad = self.ax.imshow(rgba_calidad, extent=extension, interpolation="nearest", zorder=1)
        self.capa_soporte = self.ax.imshow(rgba_soporte, extent=extension, interpolation="nearest", zorder=1)
        self.ax.set_xlim(0, W)
        self.ax.set_ylim(H, 0)
        self.ax.set_autoscale_on(False)
        self.ax.format_coord = lambda x, y: lectura_cursor(self.escena, x, y)
        self.ax.tick_params(labelsize=7)

        # Controles
        fecha = escena["adquisicion_utc"].replace("T", " ").replace("Z", " UTC")
        self.encabezado = widgets.HTML(
            f"<b>{escena['clave']}</b> · adquisición {fecha} · {W} × {H} px · clase activa: "
            f"<b>{CLASE_ID} · {CLASE_NOMBRE}</b> · trabajo: {self.origen}")
        self.vista = widgets.ToggleButtons(options=[("RGB", "rgb"), ("Espectral", "espectral"), ("NDSI", "ndsi")],
                                           value="rgb", description="Vista")
        self.ver_calidad = widgets.Checkbox(value=False, description="Calidad dudosa (naranja)", indent=False)
        self.ver_soporte = widgets.Checkbox(value=False, description="Sin soporte (magenta)", indent=False)
        self.modo = widgets.ToggleButtons(
            options=[("Navegar", "navegar"), ("Dibujar parche", "dibujar"), ("Editar", "editar"),
                     ("Región NDSI", "region")], value="navegar", description="Modo")
        boton = lambda texto, icono, f, estilo="": self._boton(texto, icono, f, estilo)
        self.botones_trazo = [boton("Cerrar polígono", "check", self._cerrar), boton("Cancelar trazo", "times",
                              lambda: self.estado.cancelar_trazo()),
                              boton("Eliminar vértice", "eraser", lambda: self.estado.eliminar_vertice()),
                              boton("Eliminar polígono", "trash", lambda: self.estado.eliminar_seleccion(), "warning"),
                              boton("Deshacer", "undo", self._deshacer),
                              boton("Ver todo", "expand", self._ver_todo)]
        self.umbral = widgets.BoundedFloatText(value=self.estado.umbral, min=-1, max=1, step=0.01,
                                               description="Umbral NDSI", layout=widgets.Layout(width="190px"))
        self.boton_aceptar = boton("Aceptar candidato", "check-circle", lambda: self._aceptar(), "success")
        self.boton_descartar = boton("Descartar candidato", "ban", lambda: self.estado.descartar_candidato(), "danger")
        self.boton_ndsi = boton("Dibujar con NDSI", "magic", self._dibujar_ndsi)
        self.lectura = widgets.Label("Mueve el cursor sobre la imagen: columna · fila · NDSI")
        self.info = widgets.HTML()
        self.mensaje = widgets.HTML()
        self.notas = widgets.Textarea(value=self.estado.notas, placeholder="Dudas de interpretación, nubes, sombras…",
                                      description="Notas", layout=widgets.Layout(width="98%", height="60px"))
        self.pendiente = widgets.Checkbox(value=self.estado.pendiente, description="Pendiente (requiere nota)",
                                          indent=False)
        self.completa = widgets.Checkbox(value=self.estado.revision_completa, indent=False,
                                         description="Revisé toda la superficie observable")
        self.negativa = widgets.Checkbox(value=self.estado.negativa, indent=False,
                                         description="Negativa: sin nieve/hielo visible")
        self.boton_guardar = boton("Guardar borrador", "save", self.guardar, "info")
        self.boton_aprobar = boton("Aprobar y exportar", "upload", self._aprobar, "primary")
        self.estado_guardado = widgets.HTML()
        self.verificacion = widgets.Output()
        self.boton_coincide = boton("Coincide: publicar", "check", self._publicar, "success")
        self.boton_no_coincide = boton("No coincide", "times", self._no_coincide, "danger")
        self.caja_verificacion = widgets.HBox([self.boton_coincide, self.boton_no_coincide],
                                              layout=widgets.Layout(display="none"))

        self.vista.observe(lambda c: self._accion(self._cambiar_vista), names="value")
        self.ver_calidad.observe(lambda c: self._accion(lambda: None), names="value")
        self.ver_soporte.observe(lambda c: self._accion(lambda: None), names="value")
        self.modo.observe(lambda c: self._accion(self._cambiar_modo), names="value")
        self.umbral.observe(lambda c: self._accion(self._cambiar_umbral), names="value")
        for control, campo in ((self.notas, "notas"), (self.pendiente, "pendiente"),
                               (self.completa, "revision_completa"), (self.negativa, "negativa")):
            control.observe(lambda c, campo=campo: self._cambiar_campo(campo, c["new"]), names="value")
        self._cids = [self.canvas.mpl_connect(evento, funcion) for evento, funcion in (
            ("button_press_event", self._al_presionar), ("motion_notify_event", self._al_mover),
            ("button_release_event", self._al_soltar), ("scroll_event", self._al_rueda))]
        self._cambiar_vista()
        if ndsi_al_abrir is None:
            ndsi_al_abrir = globals().get("poligonos_ndsi_al_abrir", True)
        if self.origen == "nueva" and ndsi_al_abrir:
            # Escena sin trabajo previo: se parte de los parches NDSI (reproducible, así que no cuenta como cambio)
            self.mensaje.value = self._resumen_automatico(*self.estado.dibujar_ndsi(escena, self.umbral.value))
        self._refrescar()
        if self.origen != "nueva":
            self.mensaje.value = f"Se abrió el trabajo previo: {self.origen}."

    # --- Utilidades -----------------------------------------------------------------------
    def _boton(self, texto, icono, funcion, estilo=""):
        b = widgets.Button(description=texto, icon=icono, button_style=estilo,
                           layout=widgets.Layout(width="auto"))
        b.on_click(lambda _: self._accion(funcion))
        return b

    def _accion(self, funcion):
        """Ejecuta una acción del usuario, muestra errores y descarta la etiqueta provisional si algo cambió."""
        if not self.activo:
            self.mensaje.value = "Este editor fue reemplazado; usa el más reciente (celda 9d)."
            return None
        version, resultado = self.estado.version, None
        try:
            resultado = funcion()
        except (ValueError, ErrorEscena, OSError) as e:
            self.mensaje.value = f"<span style='color:#c62828'>⚠ {e}</span>"
        except Exception as e:
            self.mensaje.value = f"<span style='color:#c62828'>⚠ Error inesperado: {type(e).__name__}: {e}</span>"
        if self.estado.version != version:
            self.sin_guardar = True
            if self.provisional and self.provisional["version"] != self.estado.version:
                self._descartar_provisional("La anotación cambió: la etiqueta provisional se descartó.")
        self._refrescar()
        return resultado

    def _cambiar_campo(self, campo, valor):
        if self._sincronizando or not self.activo:
            return
        self._accion(lambda: (setattr(self.estado, campo, valor), self.estado._cambio()))

    def _sincronizar_controles(self):
        self._sincronizando = True
        try:
            self.notas.value, self.pendiente.value = self.estado.notas, self.estado.pendiente
            self.completa.value, self.negativa.value = self.estado.revision_completa, self.estado.negativa
        finally:
            self._sincronizando = False

    def _tolerancia(self, px=9):
        inv = self.ax.transData.inverted()
        return float(abs(inv.transform((px, 0))[0] - inv.transform((0, 0))[0]))

    def _herramienta_activa(self):
        barra = getattr(self.canvas, "toolbar", None)
        return barra is not None and str(getattr(barra, "mode", "")) != ""

    # --- Acciones -------------------------------------------------------------------------
    def _cerrar(self):
        tipo, elemento = self.estado.cerrar_trazo()
        if tipo == "region":
            candidatos, omitidos = self.estado._sugerir_en(self.escena, elemento, self.umbral.value)
            self.mensaje.value = self._resumen_candidatos(candidatos, omitidos)
            if candidatos and self.modo.value == "region":
                self.modo.value = "editar"
        else:
            self.mensaje.value = f"Parche {len(self.estado.poligonos)} cerrado ({len(elemento['vertices'])} vértices)."

    def _resumen_candidatos(self, candidatos, omitidos):
        cortados = sum(c["toca_limite"] for c in candidatos)
        huecos = sum(bool(c["huecos"]) for c in candidatos)
        texto = f"{len(candidatos)} candidato(s) con NDSI ≥ {self.umbral.value:.2f}."
        if cortados:
            texto += f" ⚠ {cortados} toca(n) el borde de la región y puede(n) estar cortado(s)."
        if huecos:
            texto += f" {huecos} con huecos interiores: requieren dibujo manual o quedan pendientes."
        if omitidos.get("descartados"):
            texto += f" {omitidos['descartados']} coincide(n) con descartes previos y no se proponen."
        return texto + " Selecciónalos en modo Editar para aceptarlos o descartarlos."

    def _aceptar(self):
        self.estado.aceptar_candidato()
        p = self.estado.poligonos[-1]
        self.mensaje.value = ("Candidato aceptado como parche asistido." +
                              (" ⚠ Toca el borde de su región: revisa si está cortado." if p["toca_limite"] else ""))

    def _resumen_automatico(self, dibujados, pendientes, omitidos):
        u = f"{self.umbral.value:.2f}".replace(".", ",")
        if not dibujados and not pendientes:
            return (f"No hay parches con NDSI ≥ {u} en toda la imagen: baja el umbral, dibuja a mano o, si no hay "
                    "nieve/hielo visible, marca la negativa.")
        texto = (f"Se dibujaron {len(dibujados)} parche(s) con NDSI ≥ {u} en toda la imagen (violeta, «asistido»). "
                 "Revísalos: corrige vértices en modo Editar, elimina los que sobren y dibuja los que falten.")
        cortados = sum(p["toca_limite"] for p in dibujados)
        if cortados:
            texto += f" {cortados} toca(n) el borde de la imagen."
        if pendientes:
            texto += (f" {len(pendientes)} con huecos interiores queda(n) como candidato(s) pendiente(s) (cian): "
                      "dibújalos a mano o descártalos.")
        if omitidos.get("descartados"):
            texto += f" {omitidos['descartados']} eliminado(s) antes no se vuelve(n) a dibujar."
        return texto + " Cambia el umbral para redibujar los que no editaste."

    def _dibujar_ndsi(self):
        self.mensaje.value = self._resumen_automatico(*self.estado.dibujar_ndsi(self.escena, self.umbral.value))

    def _deshacer(self):
        self.estado.deshacer()
        self.umbral.value = self.estado.umbral     # el umbral también se revierte (no regenera: ya coincide)

    def _cambiar_umbral(self):
        if abs(self.umbral.value - self.estado.umbral) < 1e-12:
            return
        nuevos, omitidos = self.estado.regenerar(self.escena, self.umbral.value)
        if "dibujados" in omitidos:
            u = f"{self.umbral.value:.2f}".replace(".", ",")
            self.mensaje.value = (f"Umbral {u}: {omitidos['dibujados']} parche(s) automático(s) redibujado(s); "
                                  "los editados y manuales no cambian." +
                                  (f" {len(nuevos)} candidato(s) pendiente(s)." if nuevos else ""))
        elif omitidos:
            self.mensaje.value = "Umbral cambiado; se regeneraron los candidatos pendientes. " + \
                                 self._resumen_candidatos(nuevos, omitidos)

    def _cambiar_vista(self):
        for k, capa in self.capas.items():
            capa.set_visible(k == self.vista.value)
        self.cax.set_visible(self.vista.value == "ndsi")
        if self.vista.value in ("espectral", "ndsi") and not self.estado.ayudas[f"vista_{self.vista.value}"]:
            self.estado.ayudas[f"vista_{self.vista.value}"] = True
            self.sin_guardar = True

    def _cambiar_modo(self):
        self.estado.vertice_activo = None
        if self.modo.value != "navegar" and self._herramienta_activa():
            self.mensaje.value = "Desactiva la lupa o la cruz de la barra de herramientas para que los clics editen."

    def _ver_todo(self):
        self.ax.set_xlim(0, self.escena["ancho"])
        self.ax.set_ylim(self.escena["alto"], 0)

    def guardar(self):
        ruta = guardar_borrador(self.escena, self.estado)
        self.sin_guardar = False
        aviso = " ⚠ Marcada como pendiente sin nota." if self.estado.pendiente and not self.estado.notas.strip() else ""
        self.mensaje.value = f"✓ Borrador guardado {datetime.now():%H:%M:%S} en {ruta}.{aviso}"
        if self.al_cambiar_estado:
            self.al_cambiar_estado()
        return ruta

    def _aprobar(self):
        self._descartar_provisional()
        errores, avisos = comprobar_aprobacion(self.estado, self.escena)
        if errores:
            self.mensaje.value = ("<b>No se puede aprobar:</b><ul>" + "".join(f"<li>{e}</li>" for e in errores) +
                                  "</ul>")
            return None
        self.guardar()                       # si la publicación falla, el trabajo queda en el borrador
        prov = preparar_revision(self.escena, self.estado, avisos)
        self.provisional = prov
        png = figura_verificacion(self.escena, prov["recuperados"],
                                  f"Etiqueta provisional releída del TXT · {prov['instancias']} instancia(s)")
        with self.verificacion:
            clear_output(wait=True)
            display(ImagenIPy(data=png))
        self.caja_verificacion.layout.display = "flex"
        self.mensaje.value = (
            f"Etiqueta provisional escrita y releída: <code>{prov['txt']}</code> · {prov['instancias']} instancia(s) · "
            f"error máximo {prov['error_max']:.2e} px. Compárala abajo sobre RGB y espectral y pulsa "
            "«Coincide: publicar»." + ("<ul>" + "".join(f"<li>⚠ {a}</li>" for a in avisos) + "</ul>" if avisos else ""))
        return prov

    def _publicar(self):
        prov = self.provisional
        if prov is None:
            raise ValueError("No hay etiqueta provisional: pulsa «Aprobar y exportar».")
        self.provisional = None
        self.caja_verificacion.layout.display = "none"
        try:
            ruta = publicar_revision(prov, self.escena["carpeta"])
        except OSError as e:
            raise OSError(f"No se publicó ({e}). Se conservan el borrador y las revisiones anteriores.") from e
        self.ultimo_txt = ruta
        self.estado.revision_base = f"{prov['numero']:04d}"
        self.estado.revision_completa = False
        self._sincronizar_controles()
        self.sin_guardar = False
        tipo = "negativa (TXT vacío)" if self.estado.negativa else f"{prov['instancias']} instancia(s)"
        self.mensaje.value = (f"✓ Revisión {prov['numero']:04d} publicada · {tipo} · TXT: <code>{ruta}</code>. "
                              "Una corrección posterior generará la revisión siguiente.")
        if self.al_cambiar_estado:
            self.al_cambiar_estado()
        return ruta

    def _no_coincide(self):
        self._descartar_provisional("Etiqueta provisional descartada: corrige los contornos y vuelve a aprobar.")

    def _descartar_provisional(self, texto=None):
        if self.provisional:
            descartar_revision(self.provisional)
            self.provisional = None
        self.caja_verificacion.layout.display = "none"
        self.verificacion.clear_output()
        if texto:
            self.mensaje.value = texto

    # --- Eventos del ratón ----------------------------------------------------------------
    def _punto(self, ev, margen_px=8):
        """Coordenadas de imagen del evento; un clic sobre el borde de la imagen o hasta 8 px fuera se ajusta a ese borde."""
        if ev.inaxes is self.ax and ev.xdata is not None:
            return ev.xdata, ev.ydata
        caja = self.ax.bbox
        if ev.x is None or ev.y is None or not (caja.x0 - margen_px <= ev.x <= caja.x1 + margen_px
                                                and caja.y0 - margen_px <= ev.y <= caja.y1 + margen_px):
            return None
        x, y = self.ax.transData.inverted().transform((ev.x, ev.y))
        (x0, x1), (y1, y0) = self.ax.get_xlim(), self.ax.get_ylim()
        # Holgura de 1,5 px de pantalla: los eventos llegan en píxeles enteros
        hx, hy = 1.5 * (x1 - x0) / max(caja.width, 1), 1.5 * (y1 - y0) / max(caja.height, 1)
        resultado = []
        for v, lo, hi, maximo, h in ((x, x0, x1, self.escena["ancho"], hx), (y, y0, y1, self.escena["alto"], hy)):
            if v <= lo + h:
                v = lo if lo <= 0 else None  # solo se ajusta si ese lado del encuadre es el borde de la imagen
            elif v >= hi - h:
                v = hi if hi >= maximo else None
            if v is None:
                return None
            resultado.append(float(v))
        return tuple(resultado)

    def _al_presionar(self, ev):
        punto = self._punto(ev)
        if punto is None or self._herramienta_activa():
            return
        modo, (x, y) = self.modo.value, punto
        if modo in ("dibujar", "region"):
            if ev.button == 3 or getattr(ev, "dblclick", False):
                self._accion(self._cerrar)
            elif ev.button == 1:
                self._accion(lambda: self.estado.anadir_vertice(x, y, "parche" if modo == "dibujar" else "region"))
        elif modo == "editar" and ev.button == 1:
            def editar():
                tol, est = self._tolerancia(), self.estado
                i = est.vertice_cercano(x, y, tol) if est.seleccion else None
                if i is not None:
                    est.vertice_activo, self._arrastre = i, {"registrado": False}
                elif est.seleccion and est.insertar_en_lado(x, y, tol):
                    self._arrastre = {"registrado": True}
                else:
                    est.seleccionar_en(x, y, tol)
            self._accion(editar)

    def _al_mover(self, ev):
        punto = self._punto(ev)
        if punto is None:
            return
        x, y = punto
        celda = (int(x), int(y))
        if celda != self._ultima_celda:
            self._ultima_celda = celda
            self.lectura.value = lectura_cursor(self.escena, x, y)
        if self._arrastre is not None and self.estado.vertice_activo is not None and self.activo:
            if not self._arrastre["registrado"]:
                self.estado._registrar()
                self._arrastre["registrado"] = True
            self.estado.mover_vertice(self.estado.vertice_activo, x, y)
            self.sin_guardar = True
            if self.provisional:
                self._descartar_provisional("La anotación cambió: la etiqueta provisional se descartó.")
            self._refrescar()

    def _al_soltar(self, ev):
        self._arrastre = None

    def _al_rueda(self, ev):
        if ev.inaxes is not self.ax or ev.xdata is None:
            return
        f = 1 / 1.3 if ev.button == "up" else 1.3
        W, H = self.escena["ancho"], self.escena["alto"]
        (x0, x1), (y1, y0) = self.ax.get_xlim(), self.ax.get_ylim()
        ancho, alto = min((x1 - x0) * f, W), min((y1 - y0) * f, H)
        nx0 = min(max(ev.xdata - (ev.xdata - x0) * ancho / (x1 - x0), 0), W - ancho)
        ny0 = min(max(ev.ydata - (ev.ydata - y0) * alto / (y1 - y0), 0), H - alto)
        self.ax.set_xlim(nx0, nx0 + ancho)
        self.ax.set_ylim(ny0 + alto, ny0)
        self._pintar()

    # --- Dibujo ---------------------------------------------------------------------------
    def _refrescar(self):
        if not self.activo:
            return
        est = self.estado
        for a in self._artistas:
            a.remove()
        self._artistas = []
        anadir = self._artistas.append
        self.capa_calidad.set_visible(self.ver_calidad.value)
        self.capa_soporte.set_visible(self.ver_soporte.value)
        for r in est.regiones:
            anadir(self.ax.add_patch(PoligonoMpl(r["vertices"], closed=True, fill=False, ec=COLORES["region"],
                                                 ls=":", lw=1, zorder=2)))
        for tipo, lista in (("poligono", est.poligonos), ("candidato", est.candidatos)):
            for i, e in enumerate(lista):
                sel = est.seleccion == (tipo, i)
                color = COLORES["seleccion"] if sel else (COLORES["candidato"] if tipo == "candidato"
                                                          else COLORES[e.get("origen", "manual")])
                anadir(self.ax.add_patch(PoligonoMpl(e["vertices"], closed=True, fill=False, ec=color,
                                                     lw=2.4 if sel else 1.4, ls="--" if tipo == "candidato" else "-",
                                                     zorder=4 if sel else 3)))
                for h in e.get("huecos", []):
                    anadir(self.ax.add_patch(PoligonoMpl(h, closed=True, fill=False, ec="#ff9100", ls="--", lw=1,
                                                         zorder=3)))
                p = PoligonoShp(e["vertices"]).representative_point()
                etiqueta = f"{i + 1}" if tipo == "poligono" else f"C{i + 1}" + ("⚠" if e["toca_limite"] else "")
                anadir(self.ax.text(p.x, p.y, etiqueta, color=color, fontsize=8, fontweight="bold", ha="center",
                                    va="center", zorder=5, clip_on=True))
                if sel:
                    v = np.asarray(e["vertices"])
                    anadir(self.ax.plot(v[:, 0], v[:, 1], "o", ms=4, color=color, zorder=5)[0])
                    if est.vertice_activo is not None and est.vertice_activo < len(v):
                        anadir(self.ax.plot(*v[est.vertice_activo], "o", ms=8, mfc="none", mec=COLORES["trazo"],
                                            mew=2, zorder=6)[0])
        if est.trazo:
            v = np.asarray(est.trazo)
            color = COLORES["trazo"] if est.tipo_trazo == "parche" else COLORES["region"]
            anadir(self.ax.plot(v[:, 0], v[:, 1], marker="o", ms=4, lw=1.4, color=color,
                                ls="-" if est.tipo_trazo == "parche" else ":", zorder=6)[0])
        sel = est.seleccion
        texto_sel = "ninguna" if sel is None else (f"polígono {sel[1] + 1}" if sel[0] == "poligono"
                                                   else f"candidato C{sel[1] + 1}")
        self.info.value = (f"Polígonos: <b>{len(est.poligonos)}</b> · candidatos pendientes: <b>{len(est.candidatos)}</b>"
                           f" · selección: {texto_sel} · trazo: {len(est.trazo)} vértice(s)"
                           + (f" ({'región' if est.tipo_trazo == 'region' else 'parche'})" if est.trazo else "")
                           + (f" · base: revisión {est.revision_base}" if est.revision_base else ""))
        self.estado_guardado.value = ("<span style='color:#e65100'>● cambios sin guardar</span>" if self.sin_guardar
                                      else "<span style='color:#2e7d32'>✓ guardado</span>")
        hay_candidato = sel is not None and sel[0] == "candidato"
        self.boton_aceptar.disabled = self.boton_descartar.disabled = not hay_candidato
        self._pintar()

    def _pintar(self):
        if self.interactivo:
            self.canvas.draw_idle()
        elif self.imagen_estatica is not None:
            buf = io.BytesIO()
            self.figura.savefig(buf, format="png")
            self.imagen_estatica.value = buf.getvalue()

    def widget(self):
        if self.interactivo:
            lienzo = self.canvas
        else:
            self.imagen_estatica = widgets.Image(format="png")
            self._pintar()
            lienzo = widgets.VBox([widgets.HTML("<i>Vista estática: instala ipympl y reinicia el kernel para dibujar."
                                                "</i>"), self.imagen_estatica])
        fila = lambda *hijos: widgets.HBox(list(hijos), layout=widgets.Layout(flex_flow="row wrap"))
        return widgets.VBox([
            self.encabezado,
            fila(self.vista, self.ver_calidad, self.ver_soporte), self.modo, fila(*self.botones_trazo),
            fila(self.umbral, self.boton_ndsi, self.boton_aceptar, self.boton_descartar),
            lienzo, self.lectura, self.info, self.mensaje,
            self.notas, fila(self.pendiente, self.completa, self.negativa),
            fila(self.boton_guardar, self.boton_aprobar, self.estado_guardado),
            self.verificacion, self.caja_verificacion])

    def cerrar(self):
        """Desconecta los callbacks y descarta la etiqueta provisional (la anotación guardada no cambia)."""
        if not self.activo:
            return
        for cid in self._cids:
            self.canvas.mpl_disconnect(cid)
        self._cids = []
        self._descartar_provisional()
        self.activo = False
        if self.interactivo:
            self.canvas.close()


class PanelAnotacion:
    """Selector de escenas con estado de anotación; pide confirmación si hay cambios sin guardar."""

    def __init__(self, carpetas, interactivo=True, registro=None):
        self.carpetas = [Path(c) for c in carpetas]
        self.interactivo, self.registro = interactivo, registro
        self.editor, self._guardia, self._solicitada = None, False, None
        self.selector = widgets.Dropdown(options=self._opciones(), description="Escena",
                                         layout=widgets.Layout(width="98%"))
        self.mensaje = widgets.HTML()
        self.botones_confirmacion = []
        for texto, opcion, estilo in (("Guardar y cambiar", "guardar", "info"),
                                      ("Descartar y cambiar", "descartar", "warning"),
                                      ("Cancelar", "cancelar", "")):
            b = widgets.Button(description=texto, button_style=estilo)
            b.on_click(lambda _, opcion=opcion: self.resolver(opcion))
            self.botones_confirmacion.append(b)
        self.confirmacion = widgets.HBox(self.botones_confirmacion, layout=widgets.Layout(display="none"))
        self.contenedor = widgets.VBox()
        self.caja = widgets.VBox([self.selector, self.confirmacion, self.mensaje, self.contenedor])
        self._previa = self.selector.value
        self.selector.observe(self._al_cambiar, names="value")
        if self.selector.value:
            self._abrir(self.selector.value)

    def _opciones(self):
        opciones = [(etiqueta_escena(c), str(c)) for c in self.carpetas]
        return ([("— elige una escena —", "")] + opciones) if len(opciones) > 1 else opciones

    def _fijar_valor(self, valor):
        self._guardia = True
        try:
            self.selector.value = valor
        finally:
            self._guardia = False

    def actualizar_etiquetas(self):
        valor = self.selector.value
        self._guardia = True
        try:
            self.selector.options = self._opciones()
            self.selector.value = valor
        finally:
            self._guardia = False

    def _al_cambiar(self, cambio):
        if self._guardia:
            return
        if self.editor is not None and self.editor.sin_guardar:
            self._solicitada = cambio["new"]
            self.mensaje.value = "<b>La escena actual tiene cambios sin guardar.</b> ¿Qué hago antes de cambiar?"
            self.confirmacion.layout.display = "flex"
            return
        self._abrir(cambio["new"])

    def resolver(self, opcion):
        self.confirmacion.layout.display = "none"
        destino, self._solicitada = self._solicitada, None
        if opcion == "cancelar":
            self._fijar_valor(self._previa)
            self.mensaje.value = "Cambio cancelado: sigues en la misma escena, con tus cambios."
            return
        if opcion == "guardar":
            try:
                self.editor.guardar()
            except OSError as e:
                self._fijar_valor(self._previa)
                self.mensaje.value = f"<span style='color:#c62828'>No se pudo guardar ({e}); sigues en la escena.</span>"
                return
        self._abrir(destino)
        self.actualizar_etiquetas()

    def _abrir(self, valor):
        if self.editor is not None:
            self.editor.cerrar()
        self.editor, self._previa = None, valor
        self.contenedor.children = ()
        self.mensaje.value = ""
        if not valor:
            return
        try:
            self.editor = EditorAnotacion(cargar_escena(valor), interactivo=self.interactivo, registro=self.registro,
                                          al_cambiar_estado=self.actualizar_etiquetas)
        except ErrorEscena as e:
            self.mensaje.value = (f"<span style='color:#c62828'><b>No se puede editar esta escena:</b></span>"
                                  f"<pre>{e}</pre>Elige otra escena en el selector.")
            return
        self.contenedor.children = (self.editor.widget(),)

In [176]:
# 9d. Abrir el editor: ruta_escena, las descargas de esta sesión o todas las escenas de datos_sentinel2.
def escenas_para_anotar():
    if str(ruta_escena).strip():
        ruta = Path(ruta_escena).expanduser()
        return escenas_en(ruta), f"ruta_escena = {ruta}"
    sesion = [Path(r["carpeta"]) for r in (globals().get("resultados_descarga") or [])
              if r.get("carpeta") and (Path(r["carpeta"]) / "metadata.json").is_file()]
    if sesion:
        return list(dict.fromkeys(sesion)), "descargas de esta sesión"
    base = Path(globals().get("directorio_salida", "datos_sentinel2"))
    return escenas_en(base), f"{base.resolve()}"


carpetas_anotacion, origen_anotacion = escenas_para_anotar()
if not carpetas_anotacion:
    print(f"No hay escenas descargadas en {origen_anotacion}.\n"
          "Descarga una en la sección 6 o escribe la carpeta de una descarga en ruta_escena (celda 9a).")
else:
    print(f"{len(carpetas_anotacion)} escena(s) desde {origen_anotacion}.")
    if not lienzo_interactivo_ok:
        print("Sin ipympl el lienzo es estático (ver la celda 9a).")
    panel_anotacion = PanelAnotacion(carpetas_anotacion, interactivo=lienzo_interactivo_ok)
    display(panel_anotacion.caja)

1 escena(s) desde C:\Users\ing_c\OneDrive\Documentos\DESARROLLOS GEE\Selector de imágen satelital\datos_sentinel2.


In [177]:
# 9e. Pruebas de aceptación de la anotación (SDD 02 §6) con datos sintéticos en un directorio temporal.
import pandas as pd

resultados_anotacion = []


def prueba_anot(nombre):
    def decorador(funcion):
        try:
            detalle = funcion()
            resultados_anotacion.append({"prueba": nombre, "resultado": "omitida" if str(detalle).startswith("omitida")
                                         else "aprobada", "detalle": detalle or ""})
        except Exception as e:
            resultados_anotacion.append({"prueba": nombre, "resultado": "FALLÓ", "detalle": f"{type(e).__name__}: {e}"})
        return funcion
    return decorador


def escena_sintetica(raiz, nombre="20240101_TPRUEBA_sintetica", W=120, H=80, ndsi=None, soporte=None, calidad=None,
                     adquisicion="2024-01-01T15:00:00Z"):
    """Escena con el mismo contrato que la sección 6 (EPSG:32618, 10 m, NoData −9999)."""
    c = Path(raiz) / nombre
    (c / "rgb").mkdir(parents=True)
    (c / "espectral").mkdir()
    ndsi = np.full((H, W), -0.2, "float32") if ndsi is None else np.asarray(ndsi, "float32")
    soporte = np.ones((H, W), "uint8") if soporte is None else np.asarray(soporte, "uint8")
    calidad = np.where(soporte == 1, 2, 0).astype("uint8") if calidad is None else np.asarray(calidad, "uint8")
    T, crs = Affine(10, 0, 500000, 0, -10, 540000), "EPSG:32618"
    indices = np.stack([ndsi, np.zeros_like(ndsi)])
    indices[:, soporte == 0] = -9999
    for archivo, datos, tipo, nodata in (("indices.tif", indices, "float32", -9999),
                                         ("calidad.tif", calidad[None], "uint8", 0),
                                         ("soporte_png.tif", soporte[None], "uint8", 0)):
        with rasterio.open(c / archivo, "w", driver="GTiff", width=W, height=H, count=len(datos), dtype=tipo,
                           crs=crs, transform=T, nodata=nodata) as ds:
            ds.write(datos.astype(tipo))
    xx, yy = np.meshgrid(np.arange(W), np.arange(H))
    rgb = np.stack([xx * 255 // W, yy * 255 // H, np.full_like(xx, 128)], axis=-1).astype("uint8")
    esp = np.repeat(np.round(255 * (np.clip(ndsi, -1, 1) + 1) / 2).astype("uint8")[..., None], 3, axis=-1)
    Image.fromarray(rgb).save(c / "rgb" / f"{nombre}.png")
    Image.fromarray(esp).save(c / "espectral" / f"{nombre}.png")
    rutas = {"indices": "indices.tif", "calidad": "calidad.tif", "soporte_png": "soporte_png.tif",
             "rgb": f"rgb/{nombre}.png", "espectral": f"espectral/{nombre}.png"}
    meta = {"estado": "completa", "nombre_escena": nombre, "clave": nombre, "id_asset": f"PRUEBA/{nombre}",
            "grupo_temporal": f"prueba_{adquisicion[:10].replace('-', '')}", "adquisicion_utc": adquisicion,
            "malla": {"crs": crs, "transform": list(T)[:6], "ancho": W, "alto": H, "resolucion_m": 10.0},
            "bandas": {"indices.tif": ["NDSI", "NDVI"], "calidad.tif": ["calidad"], "soporte_png.tif": ["soporte"]},
            "receta": {"version": "prueba", "huella": "prueba"}, "rutas": rutas,
            "archivos": {rel: {"sha256": sha256_archivo(c / rel), "bytes": (c / rel).stat().st_size}
                         for rel in rutas.values()}}
    (c / "metadata.json").write_text(json.dumps(meta), encoding="utf-8")
    return c


def evento(editor, tipo, x, y, boton=1, **extra):
    """Simula un evento del ratón en coordenadas de imagen (pasa por las transformaciones del lienzo)."""
    editor.ax.apply_aspect()
    px, py = editor.ax.transData.transform((x, y))
    editor.canvas.callbacks.process(tipo, MouseEvent(tipo, editor.canvas, px, py, button=boton, **extra))


def clics(editor, puntos, modo="dibujar", cerrar=True):
    editor.modo.value = modo
    for x, y in puntos:
        evento(editor, "button_press_event", x, y)
        evento(editor, "button_release_event", x, y)
    if cerrar:
        editor.botones_trazo[0].click()


def aprobar_y_publicar(editor):
    editor.completa.value = True
    editor.boton_aprobar.click()
    assert editor.provisional is not None, editor.mensaje.value
    editor.boton_coincide.click()
    assert editor.ultimo_txt is not None and editor.ultimo_txt.is_file(), editor.mensaje.value
    return editor.ultimo_txt


def editor_prueba(carpeta, registro=None, interactivo=False, ndsi_al_abrir=False):
    return EditorAnotacion(cargar_escena(carpeta), interactivo=interactivo, ndsi_al_abrir=ndsi_al_abrir,
                           registro=[] if registro is None else registro)


tmp_anot = Path(tempfile.mkdtemp(prefix="anotacion_pruebas_"))
ESPERADA = "0 0.10000000 0.10000000 0.30000000 0.10000000 0.30000000 0.30000000 0.10000000 0.30000000\n"


@prueba_anot("Escala y ejes · rectángulo sintético W=100, H=50")
def _():
    ed = editor_prueba(escena_sintetica(tmp_anot / "escala", W=100, H=50))
    clics(ed, [(10, 5), (30, 5), (30, 15), (10, 15)])
    assert ed.estado.poligonos[0]["vertices"] == [[10, 5], [30, 5], [30, 15], [10, 15]]
    txt = aprobar_y_publicar(ed)
    contenido = txt.read_bytes().decode("utf-8")
    assert contenido == ESPERADA, repr(contenido)
    assert txt.parent.parent.name == "0001" and txt.name == f"{ed.escena['nombre']}.txt"
    return f"TXT = «{contenido.strip()}»"


@prueba_anot("Escala y ejes · vértice en el borde inferior derecho con zoom activo")
def _():
    ed = editor_prueba(escena_sintetica(tmp_anot / "borde", W=100, H=50))
    ed.ax.set_xlim(60, 100)
    ed.ax.set_ylim(50, 30)
    clics(ed, [(70, 35), (100, 35), (100.1, 50.1), (70, 50)])  # la esquina se pulsa ~2 px fuera y se ajusta al borde
    assert ed.ax.get_xlim() == (60, 100) and ed.ax.get_ylim() == (50, 30), "el zoom cambió"
    v = ed.estado.poligonos[0]["vertices"]
    # Los eventos llegan en píxeles de pantalla enteros: tolerancia < 1 px de pantalla fuera de los bordes
    assert np.allclose(v, [[70, 35], [100, 35], [100, 50], [70, 50]], atol=0.1), v
    assert v[1][0] == 100 and v[2] == [100, 50] and v[3][1] == 50, v
    linea = serializar_yolo([v], 100, 50)
    fichas = linea.split()
    assert fichas[3] == fichas[5] == fichas[6] == fichas[8] == "1.00000000", linea
    ideal = serializar_yolo([[[70, 35], [100, 35], [100, 50], [70, 50]]], 100, 50)
    assert ideal == "0 0.70000000 0.70000000 1.00000000 0.70000000 1.00000000 1.00000000 0.70000000 1.00000000\n", ideal
    intercambiado = serializar_yolo([[[y, x] for x, y in v]], 100, 50)
    assert intercambiado != linea, "W≠H no detectaría ejes intercambiados"
    return linea.strip()


@prueba_anot("Escala y ejes · medio píxel y lectura del cursor")
def _():
    ed = editor_prueba(escena_sintetica(tmp_anot / "medio", W=100, H=50))
    ed.ax.set_xlim(30, 45)
    ed.ax.set_ylim(20, 5)
    clics(ed, [(37, 12), (38, 12), (38, 13), (37, 13)])
    m = mascara_poligono(ed.estado.poligonos[0]["vertices"], 100, 50)
    assert m.sum() == 1 and m[12, 37], np.argwhere(m)
    assert lectura_cursor(ed.escena, 37.5, 12.5).startswith("columna 37 · fila 12 · NDSI -0.200")
    assert lectura_cursor(ed.escena, 37.99, 12.01).startswith("columna 37 · fila 12")
    return "píxel (col 37, fila 12) ↔ cuadrado [37,38]×[12,13]; centro 37,5 / 12,5"


def ndsi_asistencia(W=120, H=80):
    n = np.full((H, W), -0.2, "float32")
    n[10:21, 10:26] = 0.6          # A
    n[21:26, 10:26] = 0.35         # banda D junto a A (aparece con umbral 0,30)
    n[10:21, 40:56] = 0.6          # B, con un hueco de 1 px que se cierra
    n[15, 47] = -0.2
    n[40:71, 70:101] = 0.6         # C, con hueco de 8×8
    n[51:59, 81:89] = -0.2
    return n


def candidato_en(ed, x, y):
    for i, c in enumerate(ed.estado.candidatos):
        if PoligonoShp(c["vertices"]).contains(Point(x, y)):
            return i, c
    raise AssertionError(f"no hay candidato en ({x}, {y})")


@prueba_anot("Asistencia y edición · aceptar, corregir, descartar, hueco y cambio de umbral")
def _():
    ed = editor_prueba(escena_sintetica(tmp_anot / "asistencia", ndsi=ndsi_asistencia()))
    clics(ed, [(5, 5), (110, 5), (110, 75), (5, 75)], modo="region")
    assert len(ed.estado.candidatos) == 3, ed.mensaje.value
    assert ed.modo.value == "editar"
    _, b = candidato_en(ed, 45, 12)
    assert b["huecos"] == [] and b["huecos_menores_cerrados"] == 1
    assert not any(c["toca_limite"] for c in ed.estado.candidatos)
    evento(ed, "button_press_event", 18, 15)               # selecciona A y lo acepta
    ed.boton_aceptar.click()
    a = ed.estado.poligonos[0]
    assert a["origen"] == "asistido" and a["umbral"] == 0.4 and a["region"] is not None
    assert sorted(map(tuple, a["vertices"])) == [(10, 10), (10, 21), (26, 10), (26, 21)]
    evento(ed, "button_press_event", 10, 10)               # arrastra el vértice (10, 10) a (9, 9)
    evento(ed, "motion_notify_event", 9.5, 9.5)
    evento(ed, "motion_notify_event", 9, 9)
    evento(ed, "button_release_event", 9, 9)
    assert [9, 9] in a["vertices"] and [10, 10] not in a["vertices"] and a["editado"]
    ed.boton_coincide.click()                               # sin provisional: solo mensaje
    ed.estado.deshacer()
    assert [10, 10] in ed.estado.poligonos[0]["vertices"], "deshacer no revirtió el arrastre"
    evento(ed, "button_press_event", 10, 10)
    evento(ed, "motion_notify_event", 9, 9)
    evento(ed, "button_release_event", 9, 9)
    geometria_a = copy.deepcopy(ed.estado.poligonos[0])
    evento(ed, "button_press_event", 45, 12)               # descarta B
    assert ed.estado.seleccion[0] == "candidato"
    ed.boton_descartar.click()
    evento(ed, "button_press_event", 72, 45)               # C tiene hueco: aceptar se rechaza
    ed.boton_aceptar.click()
    assert "hueco" in ed.mensaje.value and len(ed.estado.candidatos) == 1
    ed.umbral.value = 0.30                                  # regenera solo pendientes
    assert ed.estado.poligonos[0] == geometria_a, "el aceptado cambió al regenerar"
    assert all(not PoligonoShp(c["vertices"]).contains(Point(45, 12)) for c in ed.estado.candidatos), "B reapareció"
    _, c = candidato_en(ed, 72, 45)
    assert c["huecos"] and c["umbral"] == 0.30
    _, d = candidato_en(ed, 18, 23)
    assert d["umbral"] == 0.30
    errores, _ = comprobar_aprobacion(ed.estado, ed.escena)
    assert any("pendientes" in e for e in errores)
    return f"{len(ed.estado.candidatos)} pendientes tras umbral 0,30 (C con hueco y banda D); A intacto; B no vuelve"


@prueba_anot("Asistencia y edición · toca el límite de la región")
def _():
    ed = editor_prueba(escena_sintetica(tmp_anot / "limite", ndsi=ndsi_asistencia()))
    clics(ed, [(0, 0), (18, 0), (18, 15), (0, 15)], modo="region")
    assert len(ed.estado.candidatos) == 1 and ed.estado.candidatos[0]["toca_limite"]
    assert "borde de la región" in ed.mensaje.value
    return "candidato recortado por la región marcado con ⚠"


@prueba_anot("Asistencia y edición · dos parches separados")
def _():
    ed = editor_prueba(escena_sintetica(tmp_anot / "dos", ndsi=ndsi_asistencia()))
    clics(ed, [(5, 5), (60, 5), (60, 30), (5, 30)], modo="region")
    assert len(ed.estado.candidatos) == 2, len(ed.estado.candidatos)
    for x, y in ((18, 15), (45, 12)):
        evento(ed, "button_press_event", x, y)
        ed.boton_aceptar.click()
    txt = aprobar_y_publicar(ed)
    instancias = leer_txt_yolo(txt)
    assert len(instancias) == 2 and all(k == 0 for k, _ in instancias)
    datos = json.loads((txt.parent.parent / "anotacion.json").read_text(encoding="utf-8"))
    assert [p["origen"] for p in datos["poligonos"]] == ["asistido", "asistido"] and datos["instancias"] == 2
    assert datos["anotacion_manual_sin_indices"] is False and datos["ayudas"]["candidatos_aceptados"] == 2
    return "TXT con 2 líneas; procedencia «asistido» y umbral 0,40 en anotacion.json"


def poligono_en(ed, x, y):
    return next((p for p in ed.estado.poligonos if PoligonoShp(p["vertices"]).contains(Point(x, y))), None)


@prueba_anot("Asistencia y edición · polígonos NDSI automáticos al abrir (predeterminado)")
def _():
    carpeta = escena_sintetica(tmp_anot / "automatico", ndsi=ndsi_asistencia())
    ed = editor_prueba(carpeta, ndsi_al_abrir=True)
    a, b = poligono_en(ed, 18, 15), poligono_en(ed, 45, 12)
    assert len(ed.estado.poligonos) == 2 and a and b, ed.mensaje.value
    assert all(p["origen"] == "asistido" and p["automatico"] and p["umbral"] == 0.4 for p in ed.estado.poligonos)
    assert len(ed.estado.candidatos) == 1 and ed.estado.candidatos[0]["huecos"], "C (con hueco) debe quedar pendiente"
    assert poligono_en(ed, 18, 23) is None and not ed.sin_guardar
    assert "Se dibujaron 2 parche(s)" in ed.mensaje.value and "huecos" in ed.mensaje.value
    ed.modo.value = "editar"
    evento(ed, "button_press_event", 18, 15)               # selecciona A
    evento(ed, "button_release_event", 18, 15)
    evento(ed, "button_press_event", 10, 10)               # corrige A: (10, 10) → (9, 9)
    evento(ed, "motion_notify_event", 9, 9)
    evento(ed, "button_release_event", 9, 9)
    assert a["editado"] and ed.sin_guardar
    geometria_a = copy.deepcopy(a)
    evento(ed, "button_press_event", 45, 12)               # elimina B
    ed.botones_trazo[3].click()
    assert poligono_en(ed, 45, 12) is None and len(ed.estado.descartes) == 1
    ed.umbral.value = 0.30                                  # redibuja los automáticos no editados
    assert poligono_en(ed, 18, 15) == geometria_a, "A editado cambió al redibujar"
    assert poligono_en(ed, 45, 12) is None, "B eliminado reapareció"
    d = poligono_en(ed, 18, 23)
    assert d and d["automatico"] and d["umbral"] == 0.30 and not d["editado"], "falta la banda D"
    assert [c["umbral"] for c in ed.estado.candidatos] == [0.30] and "redibujado" in ed.mensaje.value
    errores = comprobar_aprobacion(ed.estado, ed.escena)[0]
    assert any("pendientes" in e for e in errores) and not any("Polígono" in e for e in errores), errores
    ed.botones_trazo[4].click()                             # deshacer el cambio de umbral
    assert poligono_en(ed, 18, 23) is None and ed.estado.umbral == 0.4 and ed.umbral.value == 0.4
    ed.umbral.value = 0.30
    evento(ed, "button_press_event", 72, 45)               # descarta C y publica
    ed.boton_descartar.click()
    txt = aprobar_y_publicar(ed)
    datos = json.loads((txt.parent.parent / "anotacion.json").read_text(encoding="utf-8"))
    assert len(leer_txt_yolo(txt)) == 2 and datos["instancias"] == 2
    assert all(p["origen"] == "asistido" and p["automatico"] for p in datos["poligonos"])
    assert datos["anotacion_manual_sin_indices"] is False
    manual = editor_prueba(escena_sintetica(tmp_anot / "sin_ndsi", ndsi=ndsi_asistencia()), ndsi_al_abrir=False)
    assert not manual.estado.poligonos and not manual.estado.candidatos and not manual.estado.ayudas["sugerencias_generadas"]
    return "abre con A y B dibujados y C pendiente; A editado y B eliminado se respetan con umbral 0,30 (D aparece); TXT 2 líneas"


@prueba_anot("Calidad y negativos")
def _():
    soporte = np.ones((80, 120), "uint8")
    soporte[:, :10] = 0
    calidad = np.where(soporte == 1, 2, 0).astype("uint8")
    calidad[60:70, 20:30] = 1
    carpeta = escena_sintetica(tmp_anot / "calidad", soporte=soporte, calidad=calidad)
    ed = editor_prueba(carpeta)
    ed.completa.value = True
    ed.boton_aprobar.click()
    assert "vacío no es una negativa" in ed.mensaje.value and ed.provisional is None
    ed.completa.value = False
    ed.negativa.value = True
    ed.boton_aprobar.click()
    assert "Revisé toda la superficie" in ed.mensaje.value and ed.provisional is None
    ed.pendiente.value, ed.notas.value = True, "Nube sobre la cumbre norte"
    ed.completa.value = True
    ed.boton_aprobar.click()
    assert "pendiente" in ed.mensaje.value and not revisiones_publicadas(carpeta)
    ed.pendiente.value = False
    txt = aprobar_y_publicar(ed)
    datos = json.loads((txt.parent.parent / "anotacion.json").read_text(encoding="utf-8"))
    assert txt.read_bytes() == b"" and datos["tipo"] == "negativa" and datos["estado"] == "aprobada"
    ed.negativa.value = False
    clics(ed, [(2, 40), (8, 40), (8, 50), (2, 50)])                   # sobre soporte inválido
    errores, _ = comprobar_aprobacion(ed.estado, ed.escena)
    assert any("sin soporte" in e for e in errores)
    ed.completa.value = True
    ed.boton_aprobar.click()
    assert ed.provisional is None and len(revisiones_publicadas(carpeta)) == 1
    ed.estado.poligonos.clear()
    clics(ed, [(20, 60), (30, 60), (30, 70), (20, 70)])               # calidad dudosa: solo aviso
    clics(ed, [(30, 60), (40, 60), (40, 70), (30, 70)])               # contacto de bordes: admitido
    errores, avisos = comprobar_aprobacion(ed.estado, ed.escena)
    assert not errores and any("calidad dudosa" in a for a in avisos), (errores, avisos)
    clics(ed, [(35, 65), (45, 65), (45, 75), (35, 75)])               # solapamiento: rechazado
    errores, _ = comprobar_aprobacion(ed.estado, ed.escena)
    assert any("se solapan" in e for e in errores)
    return "vacío, negativa sin confirmar, pendiente y sin soporte rechazados; negativa confirmada = TXT vacío"


@prueba_anot("Persistencia e interacción · cambiar de escena, cancelar, guardar y recuperar")
def _():
    raiz = tmp_anot / "persistencia"
    s1 = escena_sintetica(raiz, nombre="20240101_TPRUEBA_uno")
    s2 = escena_sintetica(raiz, nombre="20240102_TPRUEBA_dos", adquisicion="2024-01-02T15:00:00Z")
    assert escenas_en(raiz) == [s1, s2]
    panel = PanelAnotacion([s1, s2], interactivo=False, registro=[])
    assert panel.editor is None and panel.selector.value == ""
    panel.selector.value = str(s1)
    ed = panel.editor
    clics(ed, [(10.25, 10.5), (30.75, 10), (30, 20.125), (12.5, 22)])
    clics(ed, [(50, 50), (60.5, 51)], cerrar=False)                   # trazo en curso
    ed.notas.value = "duda: sombra al oeste"
    assert ed.sin_guardar
    panel.selector.value = str(s2)
    assert panel.confirmacion.layout.display == "flex" and panel.editor is ed
    panel.resolver("cancelar")
    assert panel.selector.value == str(s1) and panel.editor is ed and ed.activo
    ed.boton_guardar.click()
    assert not ed.sin_guardar and rutas_anotacion(s1)[1].is_file()
    assert "borrador" in panel.selector.options[1][0]
    esperado = copy.deepcopy((ed.estado.poligonos, ed.estado.trazo, ed.estado.notas))
    panel.selector.value = str(s2)                                     # sin cambios: cambia directamente
    assert panel.editor.escena["nombre"] == s2.name and not ed.activo
    clics(panel.editor, [(1, 1), (5, 1), (5, 5)])
    panel.selector.value = str(s1)
    panel.resolver("guardar")
    assert rutas_anotacion(s2)[1].is_file() and panel.editor.escena["nombre"] == s1.name
    nuevo = editor_prueba(s1)                                          # «reinicio»: lectura desde disco
    assert (nuevo.estado.poligonos, nuevo.estado.trazo, nuevo.estado.notas) == esperado and nuevo.origen == "borrador"
    return "geometrías, trazo en curso y notas recuperados exactamente desde borrador.json"


@prueba_anot("Persistencia e interacción · reejecutar el editor: un clic = un vértice")
def _():
    carpeta = escena_sintetica(tmp_anot / "reejecucion")
    registro = []
    viejo = editor_prueba(carpeta, registro, interactivo=lienzo_interactivo_ok)
    nuevo = editor_prueba(carpeta, registro, interactivo=lienzo_interactivo_ok)
    assert not viejo.activo and registro == [nuevo] and viejo._cids == []
    for ed in (viejo, nuevo):
        ed.modo.value = "dibujar"
    evento(nuevo, "button_press_event", 20, 20)
    evento(viejo, "button_press_event", 20, 20)
    assert len(nuevo.estado.trazo) == 1 and len(viejo.estado.trazo) == 0
    otro = PanelAnotacion([carpeta], interactivo=False, registro=registro)   # reejecutar 9d
    assert not nuevo.activo and registro == [otro.editor]
    lienzo = type(nuevo.canvas).__name__
    nuevo.cerrar(), otro.editor.cerrar()
    return f"lienzo {lienzo}; callbacks antiguos desconectados"


@prueba_anot("Persistencia e interacción · fallo al publicar y revisión existente")
def _():
    carpeta = escena_sintetica(tmp_anot / "fallo")
    ed = editor_prueba(carpeta)
    clics(ed, [(10, 10), (20, 10), (20, 20), (10, 20)])
    txt1 = aprobar_y_publicar(ed)
    huella1 = sha256_archivo(txt1)
    clics(ed, [(40, 40), (50, 40), (50, 50)])
    ed.completa.value = True
    ed.boton_aprobar.click()
    prov = ed.provisional
    assert prov["numero"] == 2 and rutas_anotacion(carpeta)[1].is_file()

    def renombrar_fallido(origen, destino):
        raise OSError("disco lleno (simulado)")
    try:
        publicar_revision(prov, carpeta, renombrar=renombrar_fallido)
        raise AssertionError("el fallo no se propagó")
    except OSError:
        pass
    revs = rutas_anotacion(carpeta)[2]
    assert [p.name for p in revs.iterdir()] == ["0001"] and sha256_archivo(txt1) == huella1
    assert rutas_anotacion(carpeta)[1].is_file(), "se perdió el borrador"
    ed.provisional = None
    ed.boton_aprobar.click()
    (revs / "0002").mkdir()                                              # otra sesión publicó antes
    ed.boton_coincide.click()
    assert "ya existe" in ed.mensaje.value and "se conservan" in ed.mensaje.value.lower()
    assert sorted(p.name for p in revs.iterdir()) == ["0001", "0002"] and sha256_archivo(txt1) == huella1
    assert rutas_anotacion(carpeta)[1].is_file()
    (revs / "0002").rmdir()
    txt2 = aprobar_y_publicar(ed)
    assert txt2.parent.parent.name == "0002" and len(leer_txt_yolo(txt2)) == 2
    assert not rutas_anotacion(carpeta)[1].exists(), "el borrador debía retirarse tras publicar"
    reabierto = editor_prueba(carpeta)
    assert reabierto.origen == "revisión 0002" and reabierto.estado.revision_base == "0002"
    assert not reabierto.estado.revision_completa and len(reabierto.estado.poligonos) == 2
    return "revisión 0001 y borrador intactos tras ambos fallos; la corrección se publicó como 0002"


@prueba_anot("Persistencia · identidad de los insumos al reanudar")
def _():
    carpeta = escena_sintetica(tmp_anot / "identidad")
    ed = editor_prueba(carpeta)
    clics(ed, [(10, 10), (20, 10), (20, 20)])
    ed.guardar()
    borrador = rutas_anotacion(carpeta)[1]
    datos = json.loads(borrador.read_text(encoding="utf-8"))
    datos["escena"]["sha256_rgb"] = "0" * 64
    borrador.write_text(json.dumps(datos), encoding="utf-8")
    try:
        editor_prueba(carpeta)
        raise AssertionError("se reanudó con otra identidad")
    except ErrorEscena as e:
        assert "sha256_rgb" in str(e)
    return "borrador de otros insumos rechazado (sha256_rgb)"


@prueba_anot("Lectura de escena · archivo faltante, clave distinta y capa desplazada")
def _():
    raiz = tmp_anot / "lectura"
    buena = escena_sintetica(raiz, nombre="20240105_TPRUEBA_buena", adquisicion="2024-01-05T15:00:00Z")
    falta = escena_sintetica(raiz, nombre="20240102_TPRUEBA_falta")
    (falta / "calidad.tif").unlink()
    clave = escena_sintetica(raiz, nombre="20240103_TPRUEBA_clave")
    (clave / "rgb" / "20240103_TPRUEBA_clave.png").rename(clave / "rgb" / "20240103_TPRUEBA_otra.png")
    desplazada = escena_sintetica(raiz, nombre="20240104_TPRUEBA_desplazada")
    with rasterio.open(desplazada / "soporte_png.tif", "r+") as ds:
        ds.transform = Affine(10, 0, 500010, 0, -10, 540000)
    meta = leer_metadata(desplazada)
    meta["archivos"]["soporte_png.tif"]["sha256"] = sha256_archivo(desplazada / "soporte_png.tif")
    (desplazada / "metadata.json").write_text(json.dumps(meta), encoding="utf-8")
    esperados = {falta: "Falta calidad.tif", clave: "Falta rgb/20240103_TPRUEBA_clave.png",
                 desplazada: "soporte_png.tif: transformación"}
    for carpeta, texto in esperados.items():
        try:
            cargar_escena(carpeta)
            raise AssertionError(f"{carpeta.name} se abrió")
        except ErrorEscena as e:
            assert texto in str(e), str(e)
    panel = PanelAnotacion(escenas_en(raiz), interactivo=False, registro=[])
    panel.selector.value = str(desplazada)
    assert panel.editor is None and "capa desplazada" in panel.mensaje.value
    panel.selector.value = str(buena)
    assert panel.editor is not None and panel.editor.escena["nombre"] == buena.name
    return "se muestra archivo y discrepancia; el selector sigue abriendo otras escenas"


def buscar_piloto():
    bases = [Path(ruta_escena)] if str(ruta_escena).strip() else []
    bases += [Path(globals().get("directorio_salida", "datos_sentinel2")), Path("datos_sentinel2")]
    for base in bases:
        for c in escenas_en(base):
            meta = leer_metadata(c) or {}
            if str(meta.get("adquisicion_utc", "")).startswith("2024-01-23") and \
                    "santa_isabel" in str(meta.get("grupo_temporal", "")):
                return c
    return None


@prueba_anot("Escena piloto Santa Isabel 23/01/2024 (copia temporal)")
def _():
    original = buscar_piloto()
    if original is None:
        return "omitida: no hay una descarga del 2024-01-23 (descárgala en la sección 6)"
    copia = tmp_anot / "piloto" / original.name
    antes = sorted(p.relative_to(original) for p in original.rglob("*"))
    shutil.copytree(original, copia, ignore=shutil.ignore_patterns("anotaciones", "reflectancia.tif"))
    auto = editor_prueba(copia, ndsi_al_abrir=True)
    errores = [e for e in comprobar_aprobacion(auto.estado, auto.escena)[0] if "Polígono" in e]
    assert auto.estado.poligonos and not errores, (auto.mensaje.value, errores)
    n_auto, n_pend = len(auto.estado.poligonos), len(auto.estado.candidatos)
    ed = editor_prueba(copia)
    W, H = ed.escena["ancho"], ed.escena["alto"]
    # Contorno sobre la cumbre fijado en UTM 18N (esquina NO 454530 E, 536100 N): vale para cualquier encuadre del AOI.
    T = ed.escena["meta"]["malla"]["transform"]
    dc, df = (454530 - T[2]) / 10, (T[5] - 536100) / 10
    assert dc == int(dc) and df == int(df), T
    dc, df = int(dc), int(df)
    contorno = [(480 + dc, 344 + df), (492 + dc, 345 + df), (491 + dc, 356 + df), (479 + dc, 355 + df)]
    assert all(0 < x < W and 0 < y < H for x, y in contorno), "la cumbre quedó fuera del AOI"
    ed.ax.set_xlim(455.5 + dc, 515.5 + dc)
    ed.ax.set_ylim(380.5 + df, 320.5 + df)
    clics(ed, contorno)
    vertices = copy.deepcopy(ed.estado.poligonos[0]["vertices"])
    assert np.allclose(vertices, contorno, atol=1e-9)
    limites, artistas = (ed.ax.get_xlim(), ed.ax.get_ylim()), len(ed._artistas)
    for vista in ("espectral", "ndsi", "rgb", "ndsi"):
        ed.vista.value = vista
        assert (ed.ax.get_xlim(), ed.ax.get_ylim()) == limites and len(ed._artistas) == artistas
        assert ed.capas[vista].get_visible() and ed.estado.poligonos[0]["vertices"] == vertices
    assert ed.estado.ayudas["vista_ndsi"] and ed.estado.ayudas["vista_espectral"]
    lectura = lectura_cursor(ed.escena, 485.5 + dc, 350.5 + df)
    assert lectura.startswith(f"columna {485 + dc} · fila {350 + df}"), lectura
    assert not (mascara_poligono(vertices, W, H) & ~ed.escena["soporte"]).any()
    txt = aprobar_y_publicar(ed)
    assert np.allclose(np.asarray(leer_txt_yolo(txt)[0][1]) * [W, H], contorno, atol=0.01)
    assert sorted(p.relative_to(original) for p in original.rglob("*")) == antes, "se modificó la escena original"
    return (f"{W} × {H}; NDSI al abrir: {n_auto} polígono(s) válidos, {n_pend} pendiente(s); zoom en la cumbre; "
            f"{lectura}; contorno y encuadre estables en las 3 vistas; TXT en la copia")


shutil.rmtree(tmp_anot, ignore_errors=True)
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(pd.DataFrame(resultados_anotacion))
n_ok = sum(r["resultado"] == "aprobada" for r in resultados_anotacion)
n_om = sum(r["resultado"] == "omitida" for r in resultados_anotacion)
print(f"Pruebas de anotación: {n_ok}/{len(resultados_anotacion) - n_om} aprobadas" + (f", {n_om} omitida(s)" if n_om else ""))

,prueba,resultado,detalle
0,"Escala y ejes · rectángulo sintético W=100, H=50",aprobada,TXT = «0 0.10000000 0.10000000 0.30000000 0.10000000 0.30000000 0.30000000 0.10000000 0.30000000»
1,Escala y ejes · vértice en el borde inferior derecho con zoom activo,aprobada,0 0.70000000 0.70000000 1.00000000 0.70024638 1.00000000 1.00000000 0.70000000 1.00000000
2,Escala y ejes · medio píxel y lectura del cursor,aprobada,"píxel (col 37, fila 12) ↔ cuadrado [37,38]×[12,13]; centro 37,5 / 12,5"
3,"Asistencia y edición · aceptar, corregir, descartar, hueco y cambio de umbral",aprobada,"2 pendientes tras umbral 0,30 (C con hueco y banda D); A intacto; B no vuelve"
4,Asistencia y edición · toca el límite de la región,aprobada,candidato recortado por la región marcado con ⚠
5,Asistencia y edición · dos parches separados,aprobada,"TXT con 2 líneas; procedencia «asistido» y umbral 0,40 en anotacion.json"
6,Asistencia y edición · polígonos NDSI automáticos al abrir (predeterminado),aprobada,"abre con A y B dibujados y C pendiente; A editado y B eliminado se respetan con umbral 0,30 (D aparece); TXT 2 líneas"
7,Calidad y negativos,aprobada,"vacío, negativa sin confirmar, pendiente y sin soporte rechazados; negativa confirmada = TXT vacío"
8,"Persistencia e interacción · cambiar de escena, cancelar, guardar y recuperar",aprobada,"geometrías, trazo en curso y notas recuperados exactamente desde borrador.json"
9,Persistencia e interacción · reejecutar el editor: un clic = un vértice,aprobada,lienzo Canvas; callbacks antiguos desconectados


Pruebas de anotación: 14/14 aprobadas
